# 🗺️ PERUPETRO · Mapa de lotes y variables geográficas

Descarga los polígonos de los lotes del **mapa oficial de PERUPETRO en ArcGIS Online** y calcula, para cada lote:

| Variable | Fuente |
|---|---|
| Área (km²), centroide | Polígono del lote (PERUPETRO) |
| Departamento (principal y % por departamento; los lotes en el mar se marcan ZÓCALO) | Límites departamentales (geoBoundaries, basado en INEI) |
| Cuenca sedimentaria | Capa de cuencas de PERUPETRO (o atributo del lote) |
| Distancia al Oleoducto Norperuano | Capa del mapa de PERUPETRO, servicio público elegido u OpenStreetMap |
| Distancia a Iquitos y al río grande más cercano | HydroRIVERS (caudal) + Natural Earth (nombre) |
| % del lote dentro de Áreas Naturales Protegidas | Servicio SERNANP elegido u OpenStreetMap |
| Comunidades nativas dentro y a menos de 5 km | Servicio público elegido (BDPI – Ministerio de Cultura u otro) |

**Uso:** ejecuta los bloques en orden. El **Bloque 2** lista las capas encontradas y propone una selección; revísala antes del Bloque 3.

**Resultados en Drive:**
- `DATA/PP_LOTES_GEO.csv`: una fila por lote con todas las variables (la usa el notebook de producción);
- `DATA/MAPA/`: GeoJSON de cada capa y `PP_mapa_lotes.html` (mapa interactivo);
- `CONTROL/CONTROL PP mapa capas.csv`: qué se descargó, de dónde y cuántas entidades.

Visor original: https://perupetro.maps.arcgis.com/apps/webappviewer/index.html?id=cc883917c55a4264a01debfa02585e2e

In [ ]:
# ═══ BLOQUE 1 · INSTALACIÓN, DRIVE Y FUNCIONES (ejecutar una vez por sesión) ═══
!pip install geopandas pyogrio folium requests cryptography google-api-python-client --quiet
import os, re, io, json, time, logging, unicodedata
from datetime import date
import requests
import pandas as pd
import geopandas as gpd
import ipywidgets as w
from IPython.display import display
try:                                   # selectores visibles en Colab
    from google.colab import output as _out; _out.enable_custom_widget_manager()
except Exception:
    pass
from shapely.geometry import shape, Point, LineString, MultiLineString, Polygon, MultiPolygon

DRIVE_FOLDER_ID = "1dOOcsVyw915yKg6BzSdICVfCpYTiyeaq"            # DATA
DRIVE_FOLDER_CONTROL_ID = "1TKjS2Wjpnf26oladjhqstmN5c9Irq-Aj"    # CONTROL
ruta = "/content/PP_Mapa"; os.makedirs(ruta, exist_ok=True)

# ── Google Drive ─────────────────────────────────────────────────────────────
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
auth.authenticate_user()
drive = build("drive", "v3", cache_discovery=False)
logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)

def archivos_en(folder_id):
    out, token = {}, None
    while True:
        r = drive.files().list(q=f"'{folder_id}' in parents and trashed = false",
                               fields="nextPageToken, files(id,name,mimeType)", pageSize=1000, pageToken=token,
                               supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
        out.update({f["name"]: f["id"] for f in r.get("files", [])})
        token = r.get("nextPageToken")
        if not token:
            return out

def carpeta(nombre, padre, existentes):
    if nombre not in existentes:
        existentes[nombre] = drive.files().create(body={"name": nombre, "parents": [padre],
                                                        "mimeType": "application/vnd.google-apps.folder"},
                                                  fields="id", supportsAllDrives=True).execute()["id"]
    return existentes[nombre]

def subir(ruta_local, folder_id, existentes, mime="text/csv"):
    nombre = os.path.basename(ruta_local)
    for i in range(1, 6):
        try:
            media = MediaFileUpload(ruta_local, mimetype=mime, resumable=True)
            if nombre in existentes:
                drive.files().update(fileId=existentes[nombre], media_body=media, supportsAllDrives=True).execute()
            else:
                f = drive.files().create(body={"name": nombre, "parents": [folder_id]}, media_body=media,
                                         fields="id", supportsAllDrives=True).execute()
                existentes[nombre] = f["id"]
            return True
        except Exception as e:
            print(f"   ↻ Drive reintento {i}/5: {e}"); time.sleep(2 ** i)
    return False

def bajar_de_drive(file_id, destino):
    with open(destino, "wb") as fh:
        dl = MediaIoBaseDownload(fh, drive.files().get_media(fileId=file_id, supportsAllDrives=True))
        hecho = False
        while not hecho:
            _, hecho = dl.next_chunk()

datos_drive = archivos_en(DRIVE_FOLDER_ID)
control_drive = archivos_en(DRIVE_FOLDER_CONTROL_ID)
MAPA_FOLDER_ID = carpeta("MAPA", DRIVE_FOLDER_ID, datos_drive)
mapa_drive = archivos_en(MAPA_FOLDER_ID)
print(f"☁️  Drive conectado · archivos en DATA/MAPA: {len(mapa_drive)}")

# ── Funciones: ArcGIS REST, fuentes externas e indicadores ──────────────────

# ── Conexión ─────────────────────────────────────────────────────────────────
SES = requests.Session()
SES.headers["User-Agent"] = "Mozilla/5.0 (proyecto academico; consulta de datos publicos)"

# Algunos servidores del Estado no envían su certificado intermedio (los navegadores lo completan; Python no).
# Si hay error SSL, se descarga el intermedio desde la dirección AIA del certificado y se reintenta verificando.
_VERIFY = {}
_MUERTOS = set()        # servidores que no responden: no se vuelven a esperar en la misma sesión

def _bundle_con_intermedios(host):
    import ssl, socket, certifi, tempfile
    from cryptography import x509
    from cryptography.hazmat.primitives.serialization import Encoding
    from cryptography.x509.oid import ExtensionOID, AuthorityInformationAccessOID
    ctx = ssl.create_default_context(); ctx.check_hostname = False; ctx.verify_mode = ssl.CERT_NONE
    with socket.create_connection((host, 443), timeout=20) as s, ctx.wrap_socket(s, server_hostname=host) as ss:
        cert = x509.load_der_x509_certificate(ss.getpeercert(binary_form=True))
    pems = []
    for _ in range(3):
        try:
            aia = cert.extensions.get_extension_for_oid(ExtensionOID.AUTHORITY_INFORMATION_ACCESS).value
        except x509.ExtensionNotFound:
            break
        urls = [d.access_location.value for d in aia if d.access_method == AuthorityInformationAccessOID.CA_ISSUERS]
        if not urls or cert.issuer == cert.subject:
            break
        datos = requests.get(urls[0], timeout=30).content
        try:
            cert = x509.load_der_x509_certificate(datos)
        except ValueError:
            cert = x509.load_pem_x509_certificate(datos)
        pems.append(cert.public_bytes(Encoding.PEM).decode())
    ruta = os.path.join(tempfile.gettempdir(), f"ca_{host}.pem")
    with open(ruta, "w") as f:
        f.write(open(certifi.where()).read() + "\n" + "\n".join(pems))
    return ruta

def get_json(url, params=None, post=False, intentos=3):
    """GET/POST a un servicio REST de ArcGIS; reintenta y avisa los errores que ArcGIS manda con código 200."""
    from urllib.parse import urlparse
    params = dict(params or {})
    params.setdefault("f", "json")
    host = urlparse(url).hostname
    if host in _MUERTOS:
        raise RuntimeError(f"el servidor {host} no responde (caído o bloqueado)")
    for i in range(1, intentos + 1):
        try:
            kw = {"timeout": (20, 180), "verify": _VERIFY.get(host, True)}
            r = SES.post(url, data=params, **kw) if post else SES.get(url, params=params, **kw)
            r.raise_for_status()
            js = r.json()
            if isinstance(js, dict) and "error" in js:
                cod = js["error"].get("code")
                if cod in (498, 499, 403):
                    raise PermissionError(f"requiere inicio de sesión ({cod})")
                raise RuntimeError(js["error"].get("message", js["error"]))
            return js
        except PermissionError:
            raise
        except requests.exceptions.SSLError as e:
            if host not in _VERIFY:                       # una sola vez por servidor
                try:
                    _VERIFY[host] = _bundle_con_intermedios(host)
                    continue
                except Exception:
                    _VERIFY[host] = True
            if i == intentos:
                raise RuntimeError(f"certificado HTTPS inválido en {host}: {e}") from None
        except (requests.exceptions.ConnectionError, requests.exceptions.ConnectTimeout) as e:
            if i >= 2:                                    # 2 intentos bastan para saber que está caído
                _MUERTOS.add(host)
                raise RuntimeError(f"el servidor {host} no responde (caído o bloqueado): {type(e).__name__}") from None
        except Exception:
            if i == intentos:
                raise
        time.sleep(2 ** i)

def servicios_del_servidor(url_capa):
    """Lista todos los MapServer/FeatureServer del mismo servidor ArcGIS (raíz /rest/services y una carpeta de profundidad)."""
    m = re.match(r"(https?://[^/]+/[^/]+/rest/services)", url_capa, re.I)
    if not m:
        return []
    raiz, out = m.group(1), []
    js = get_json(raiz)
    carpetas = [""] + js.get("folders", [])
    for c in carpetas:
        j = js if not c else get_json(f"{raiz}/{c}")
        for s in j.get("services", []):
            if s.get("type") in ("MapServer", "FeatureServer"):
                out.append(f"{raiz}/{s['name']}/{s['type']}")
    return sorted(set(out))

def norm_txt(t):
    t = unicodedata.normalize("NFKD", str(t or "")).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", t).strip().upper()

def slug(t):
    return re.sub(r"[^A-Z0-9]+", "_", norm_txt(t)).strip("_")[:60] or "CAPA"

# ── 1. Descubrimiento: aplicación → mapa web → capas ────────────────────────
def _item(portal, item_id):
    return get_json(f"{portal}/sharing/rest/content/items/{item_id}")

def _item_data(portal, item_id):
    return get_json(f"{portal}/sharing/rest/content/items/{item_id}/data")

def mapa_web_de_app(portal, app_id):
    """Devuelve (id del mapa web, título) a partir del id de la aplicación (Web AppBuilder, Instant Apps, etc.)."""
    info = _item(portal, app_id)
    if info.get("type") == "Web Map":
        return app_id, info.get("title", "")
    data = _item_data(portal, app_id)
    candidatos = []
    def buscar(o):
        if isinstance(o, dict):
            for k, v in o.items():
                if k in ("itemId", "webmap", "webMap", "id") and isinstance(v, str) and re.fullmatch(r"[0-9a-f]{32}", v):
                    candidatos.append(v)
                buscar(v)
        elif isinstance(o, list):
            for v in o:
                buscar(v)
    if isinstance(data.get("map"), dict) and data["map"].get("itemId"):
        candidatos.insert(0, data["map"]["itemId"])
    buscar(data)
    for c in dict.fromkeys(candidatos):
        if c == app_id:
            continue
        try:
            inf = _item(portal, c)
        except Exception:
            continue
        if inf.get("type") == "Web Map":
            return c, inf.get("title", "")
    raise RuntimeError("No encontré el mapa web dentro de la aplicación.")

def capas_de_servicio(url, ruta_titulo=""):
    """Expande un servicio MapServer/FeatureServer o una capa suelta en una lista de capas consultables."""
    url = url.rstrip("/")
    out = []
    if re.search(r"/(MapServer|FeatureServer)$", url, re.I):
        js = get_json(url)
        for l in js.get("layers", []) + js.get("tables", []):
            if l.get("subLayerIds"):
                continue                                  # grupo: sus hijas vienen en la misma lista
            out += capas_de_servicio(f"{url}/{l['id']}", ruta_titulo)
        return out
    js = get_json(url)
    if js.get("type") in ("Group Layer",) or js.get("subLayers") and not js.get("geometryType"):
        for s in js.get("subLayers", []):
            out += capas_de_servicio(f"{url.rsplit('/', 1)[0]}/{s['id']}", ruta_titulo)
        return out
    out.append({"CAPA": js.get("name", ""), "GRUPO": ruta_titulo, "URL": url, "TIPO": js.get("type", ""),
                "GEOMETRIA": js.get("geometryType", ""), "CAMPOS": ", ".join(f["name"] for f in js.get("fields") or []),
                "MAX_REG": js.get("maxRecordCount") or 1000,
                "PAGINA": bool((js.get("advancedQueryCapabilities") or {}).get("supportsPagination"))})
    return out

def capas_del_mapa(portal, webmap_id):
    data = _item_data(portal, webmap_id)
    out = []
    def recorrer(capas, ruta=""):
        for c in capas:
            titulo = c.get("title") or c.get("id") or ""
            r = f"{ruta} / {titulo}" if ruta else titulo
            if c.get("layers") and c.get("layerType") == "GroupLayer":
                recorrer(c["layers"], r); continue
            if c.get("featureCollection"):
                for k, l in enumerate(c["featureCollection"].get("layers", [])):
                    ld = l.get("layerDefinition", {})
                    out.append({"CAPA": ld.get("name") or titulo, "GRUPO": r, "URL": f"featureCollection:{c.get('id')}:{k}",
                                "TIPO": "Colección incrustada", "GEOMETRIA": ld.get("geometryType", ""),
                                "CAMPOS": ", ".join(f["name"] for f in ld.get("fields") or []),
                                "MAX_REG": None, "PAGINA": False, "_FC": l})
                continue
            url = c.get("url")
            if not url and c.get("itemId"):
                try:
                    url = _item(portal, c["itemId"]).get("url")
                except Exception:
                    url = None
            if not url:
                out.append({"CAPA": titulo, "GRUPO": r, "URL": "", "TIPO": c.get("layerType", ""), "GEOMETRIA": "",
                            "CAMPOS": "", "MAX_REG": None, "PAGINA": False, "ERROR": "sin URL"}); continue
            try:
                hijas = capas_de_servicio(url, r)
                if c.get("layers") and re.search(r"/MapServer$", url.rstrip("/"), re.I):
                    ids = {str(l.get("id")) for l in c["layers"]}
                    hijas = [h for h in hijas if h["URL"].rsplit("/", 1)[1] in ids] or hijas
                out.extend(hijas)
            except Exception as e:
                out.append({"CAPA": titulo, "GRUPO": r, "URL": url, "TIPO": c.get("layerType", ""), "GEOMETRIA": "",
                            "CAMPOS": "", "MAX_REG": None, "PAGINA": False, "ERROR": str(e)[:200]})
    recorrer(data.get("operationalLayers", []))
    return out

def buscar_arcgis(consulta, n=15, portal="https://www.arcgis.com"):
    """Busca servicios públicos en ArcGIS Online (para ANP, comunidades nativas, oleoducto…)."""
    q = f'({consulta}) AND (type:"Feature Service" OR type:"Map Service")'
    js = get_json(f"{portal}/sharing/rest/search", {"q": q, "num": n, "sortField": "numviews", "sortOrder": "desc"})
    return [{"TITULO": r.get("title", ""), "DUEÑO": r.get("owner", ""), "TIPO": r.get("type", ""),
             "URL": r.get("url") or "", "ID": r.get("id"), "VISTAS": r.get("numViews", 0)}
            for r in js.get("results", []) if r.get("url")]

# ── 2. Descarga de una capa ──────────────────────────────────────────────────
def _anillos_a_poligono(rings):
    """Geometría Esri (anillos) → Polygon/MultiPolygon: horario = exterior, antihorario = hueco."""
    ext, huecos = [], []
    for r in rings:
        if len(r) < 4:
            continue
        p = Polygon(r)
        (huecos if p.exterior.is_ccw else ext).append(p)
    if not ext:                                  # anillos mal orientados: todos como exteriores
        ext, huecos = huecos, []
    polys = []
    for e in ext:
        hs = [h.exterior.coords for h in huecos if e.contains(h.representative_point())]
        polys.append(Polygon(e.exterior.coords, hs))
    return polys[0] if len(polys) == 1 else MultiPolygon(polys)

def esri_a_shapely(g):
    if not g:
        return None
    if "x" in g:
        return None if g["x"] is None else Point(g["x"], g["y"])
    if "points" in g:
        from shapely.geometry import MultiPoint
        return MultiPoint(g["points"])
    if "paths" in g:
        return LineString(g["paths"][0]) if len(g["paths"]) == 1 else MultiLineString(g["paths"])
    if "rings" in g:
        return _anillos_a_poligono(g["rings"])
    return None

def gdf_desde_esri(features, wkid):
    filas = [dict(f.get("attributes") or {}) for f in features]
    geoms = [esri_a_shapely(f.get("geometry")) for f in features]
    crs = "EPSG:3857" if wkid in (102100, 102113, 3857, 900913) else f"EPSG:{wkid or 4326}"
    return gpd.GeoDataFrame(filas, geometry=geoms, crs=crs).to_crs(4326)

# ── WFS (GeoServer del geoportal de PERUPETRO) ──────────────────────────────
GML_A_ESRI = {"Polygon": "esriGeometryPolygon", "Line": "esriGeometryPolyline", "Point": "esriGeometryPoint"}

def wfs_capas(base):
    """Capas (FeatureType) publicadas por un WFS: nombre, título y tipo de geometría."""
    import xml.etree.ElementTree as ET
    from urllib.parse import urlparse
    host = urlparse(base).hostname
    r = None
    for i in range(3):
        try:
            r = SES.get(f"{base}/wfs", params={"service": "WFS", "version": "2.0.0", "request": "GetCapabilities"},
                        timeout=(20, 180), verify=_VERIFY.get(host, True))
            break
        except requests.exceptions.SSLError:
            _VERIFY[host] = _bundle_con_intermedios(host)
    r.raise_for_status()
    raiz = ET.fromstring(r.content)
    out = []
    for ft in raiz.iter():
        if ft.tag.endswith("FeatureType"):
            nombre = next((c.text for c in ft if c.tag.endswith("Name")), None)
            titulo = next((c.text for c in ft if c.tag.endswith("Title")), None) or nombre
            if nombre:
                out.append({"CAPA": titulo, "GRUPO": "Geoportal PERUPETRO (WFS)", "URL": f"{base}/wfs#{nombre}",
                            "TIPO": "WFS", "GEOMETRIA": "", "CAMPOS": "", "MAX_REG": 5000, "PAGINA": True, "WFS": nombre})
    return out

def wfs_describir(base, capas):
    """Completa geometría y campos de varias capas WFS con UNA sola petición DescribeFeatureType."""
    capas = capas if isinstance(capas, list) else [capas]
    js = get_json(f"{base}/wfs", {"service": "WFS", "version": "2.0.0", "request": "DescribeFeatureType",
                                  "typeNames": ",".join(c["WFS"] for c in capas), "outputFormat": "application/json"})
    por_nombre = {}
    for ft in js.get("featureTypes") or []:
        por_nombre[ft.get("typeName")] = ft
        por_nombre[f"{js.get('targetPrefix', '')}:{ft.get('typeName')}"] = ft
    for c in capas:
        ft = por_nombre.get(c["WFS"]) or por_nombre.get(c["WFS"].split(":")[-1]) or (
            (js.get("featureTypes") or [{}])[0] if len(capas) == 1 else {})
        props = ft.get("properties", [])
        geo = next((p.get("localType") or p.get("type", "") for p in props if str(p.get("type", "")).startswith("gml:")), "")
        c["GEOMETRIA"] = next((v for k, v in GML_A_ESRI.items() if k in geo), "")
        c["CAMPOS"] = ", ".join(p["name"] for p in props if not str(p.get("type", "")).startswith("gml:"))
    return capas

def wfs_descargar(base, nombre, paso=5000):
    """Todas las entidades de una capa WFS en GeoJSON (EPSG:4326), paginando; quita la coordenada Z."""
    import shapely
    partes, inicio = [], 0
    while True:
        js = get_json(f"{base}/wfs", {"service": "WFS", "version": "2.0.0", "request": "GetFeature", "typeNames": nombre,
                                      "outputFormat": "application/json", "srsName": "EPSG:4326",
                                      "count": paso, "startIndex": inicio})
        feats = js.get("features", [])
        if feats:
            g = gpd.GeoDataFrame.from_features(feats, crs=4326)
            g["geometry"] = g.geometry.map(lambda x: shapely.force_2d(x) if x is not None else x)
            partes.append(g)
        inicio += len(feats)
        total = js.get("numberMatched") or js.get("totalFeatures")
        if not feats or len(feats) < paso or (isinstance(total, int) and inicio >= total):
            break
    return gpd.GeoDataFrame(pd.concat(partes, ignore_index=True), crs=4326) if partes else gpd.GeoDataFrame(geometry=[], crs=4326)

def descargar_capa(capa, aviso=print):
    """Descarga todas las entidades de una capa en EPSG:4326 (WFS, GeoJSON de ArcGIS o JSON de Esri)."""
    if capa.get("WFS"):
        return wfs_descargar(capa["URL"].split("/wfs#")[0], capa["WFS"])
    if capa.get("_FC"):
        fs = capa["_FC"].get("featureSet", {})
        wkid = ((capa["_FC"].get("layerDefinition", {}).get("extent") or {}).get("spatialReference") or {}).get("wkid") \
               or (fs.get("spatialReference") or {}).get("wkid") or 102100
        return gdf_desde_esri(fs.get("features", []), wkid)
    url = capa["URL"].rstrip("/") + "/query"
    base = {"where": "1=1", "outFields": "*", "outSR": 4326, "returnGeometry": "true"}
    total = get_json(url, {**base, "returnCountOnly": "true"}, post=True).get("count")
    paso = int(capa.get("MAX_REG") or 1000)
    wk = lambda js: (js.get("spatialReference") or {}).get("wkid", 4326)
    partes = []
    def por_ids():
        ids = sorted(get_json(url, {"where": "1=1", "returnIdsOnly": "true"}, post=True).get("objectIds") or [])
        for i in range(0, len(ids), paso):
            js = get_json(url, {**base, "where": "", "objectIds": ",".join(map(str, ids[i:i + paso]))}, post=True)
            partes.append(gdf_desde_esri(js.get("features", []), wk(js)))
    if capa.get("PAGINA"):
        off = 0
        while True:
            js = get_json(url, {**base, "resultOffset": off, "resultRecordCount": paso}, post=True)
            feats = js.get("features", [])
            if feats:
                partes.append(gdf_desde_esri(feats, wk(js)))
            off += len(feats)
            if not feats or (total is not None and off >= total) or (total is None and not js.get("exceededTransferLimit")):
                break
    elif total and total > paso:
        por_ids()
    else:
        js = get_json(url, base, post=True)
        if js.get("exceededTransferLimit"):
            por_ids()
        else:
            partes.append(gdf_desde_esri(js.get("features", []), wk(js)))
    gdf = pd.concat(partes, ignore_index=True) if partes else gpd.GeoDataFrame(geometry=[], crs=4326)
    gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs=4326)
    if total is not None and len(gdf) != total:
        aviso(f"   ⚠️ {capa['CAPA']}: el servidor declara {total} entidades y llegaron {len(gdf)}")
    return gdf

# ── 3. Fuentes externas ──────────────────────────────────────────────────────
GEOBOUNDARIES_API = "https://www.geoboundaries.org/api/current/gbOpen/PER/ADM1/"
NE_RIOS = "https://naciscdn.org/naturalearth/10m/physical/ne_10m_rivers_lake_centerlines.zip"
HYDRORIVERS_SA = "https://data.hydrosheds.org/file/HydroRIVERS/HydroRIVERS_v10_sa_shp.zip"
OVERPASS = "https://overpass-api.de/api/interpreter"
BBOX_PERU = (-81.5, -18.5, -68.5, 0.1)                     # lon_min, lat_min, lon_max, lat_max

def bajar(url, destino, aviso=print):
    if os.path.exists(destino) and os.path.getsize(destino) > 0:
        return destino
    with SES.get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(destino + ".part", "wb") as fh:
            for b in r.iter_content(1 << 20):
                fh.write(b)
    os.replace(destino + ".part", destino)
    return destino

def departamentos(carpeta):
    js = SES.get(GEOBOUNDARIES_API, timeout=60).json()
    ruta = bajar(js["gjDownloadURL"], os.path.join(carpeta, "geoBoundaries-PER-ADM1.geojson"))
    g = gpd.read_file(ruta).to_crs(4326)
    g["DEPARTAMENTO"] = g["shapeName"].map(norm_txt).str.replace(r"^(REGION|DEPARTAMENTO) (DE )?", "", regex=True)
    g["DEPARTAMENTO"] = g["DEPARTAMENTO"].replace({"LIMA PROVINCE": "LIMA", "PROVINCIA DE LIMA": "LIMA",
                                                   "LIMA METROPOLITANA": "LIMA", "CALLAO PROVINCE": "CALLAO",
                                                   "PROVINCIA CONSTITUCIONAL DEL CALLAO": "CALLAO"})
    return g.dissolve("DEPARTAMENTO", as_index=False)[["DEPARTAMENTO", "geometry"]]

GEOBOUNDARIES_RAW = "https://github.com/wmgeolab/geoBoundaries/raw/main/releaseData/gbOpen/PER/{n}/geoBoundaries-PER-{n}.geojson"
GADM_PER = "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_PER_{k}.json.zip"

def _json_api(url, intentos=3):
    """GET que devuelve JSON; si el servidor responde otra cosa (página de error, vacío) dice qué respondió."""
    import time
    for i in range(intentos):
        r = SES.get(url, timeout=60)
        try:
            return r.json()
        except ValueError:
            msg = f"HTTP {r.status_code}, {r.headers.get('Content-Type', '?')}, empieza con {r.text[:80]!r}"
            if i == intentos - 1:
                raise RuntimeError(f"{url} no devolvió JSON ({msg})")
            time.sleep(5 * (i + 1))

def _geojson_valido(ruta):
    try:
        with open(ruta, "rb") as fh:
            return fh.read(200).lstrip().startswith(b"{")
    except OSError:
        return False

def limites_adm(carpeta, nivel, aviso=print):
    """Provincias (ADM2) o distritos (ADM3), con el departamento (y la provincia) por punto interior.
    Fuentes en orden: API de geoBoundaries, el mismo archivo de geoBoundaries en GitHub y GADM 4.1."""
    ruta = os.path.join(carpeta, f"geoBoundaries-PER-{nivel}.geojson")
    if os.path.exists(ruta) and not _geojson_valido(ruta):
        os.remove(ruta)                                   # descarga anterior inválida (p. ej. una página HTML)
    errores = []
    for fuente in ("api", "github"):
        if os.path.exists(ruta):
            break
        try:
            url = (_json_api(GEOBOUNDARIES_API.replace("/ADM1/", f"/{nivel}/"))["gjDownloadURL"] if fuente == "api"
                   else GEOBOUNDARIES_RAW.format(n=nivel))
            bajar(url, ruta)
            if not _geojson_valido(ruta):
                os.remove(ruta); raise RuntimeError(f"{url} no devolvió un GeoJSON")
        except Exception as e:
            errores.append(f"geoBoundaries ({fuente}): {e}")
    if os.path.exists(ruta):
        g = gpd.read_file(ruta).to_crs(4326)
        g["NOMBRE"] = g["shapeName"].map(norm_txt).str.replace(r"^(PROVINCIA|DISTRITO) (DE )?", "", regex=True)
        return g[["NOMBRE", "geometry"]]
    k = {"ADM2": 2, "ADM3": 3}[nivel]                     # respaldo: GADM 4.1 (NAME_2 = provincia, NAME_3 = distrito)
    zipf = os.path.join(carpeta, f"gadm41_PER_{k}.json.zip")
    try:
        import zipfile
        if os.path.exists(zipf) and not zipfile.is_zipfile(zipf):
            os.remove(zipf)
        bajar(GADM_PER.format(k=k), zipf)
        if not zipfile.is_zipfile(zipf):
            os.remove(zipf); raise RuntimeError("GADM no devolvió un ZIP")
        g = gpd.read_file(f"zip://{zipf}!gadm41_PER_{k}.json").to_crs(4326)
    except Exception as e:
        errores.append(f"GADM: {e}")
        raise RuntimeError(" · ".join(errores))
    aviso(f"   ℹ️ {nivel}: geoBoundaries no respondió ({' · '.join(errores)}); se usa GADM 4.1")
    # GADM escribe algunos nombres sin espacios («EmilioSanMartin»): se separan antes de normalizar
    g["NOMBRE"] = g[f"NAME_{k}"].astype(str).str.replace(r"(?<=[a-záéíóúñ])(?=[A-ZÁÉÍÓÚÑ])", " ", regex=True).map(norm_txt)
    return g[["NOMBRE", "geometry"]]

def provincias_distritos(carpeta, deps):
    """(provincias, distritos) con columnas DEPARTAMENTO, PROVINCIA, DISTRITO y CLAVE «DEP|PROV[|DIST]»."""
    prov = limites_adm(carpeta, "ADM2").rename(columns={"NOMBRE": "PROVINCIA"})
    dist = limites_adm(carpeta, "ADM3").rename(columns={"NOMBRE": "DISTRITO"})
    def dentro(g, capa, col):
        pts = gpd.GeoDataFrame(geometry=g.geometry.representative_point(), crs=4326, index=g.index)
        j = gpd.sjoin(pts, capa[[col, "geometry"]], predicate="within", how="left")
        return j[~j.index.duplicated()][col].reindex(g.index)
    prov["DEPARTAMENTO"] = dentro(prov, deps, "DEPARTAMENTO")
    dist["DEPARTAMENTO"] = dentro(dist, deps, "DEPARTAMENTO")
    dist["PROVINCIA"] = dentro(dist, prov, "PROVINCIA")
    prov["CLAVE"] = prov["DEPARTAMENTO"].fillna("") + "|" + prov["PROVINCIA"]
    dist["CLAVE"] = dist["DEPARTAMENTO"].fillna("") + "|" + dist["PROVINCIA"].fillna("") + "|" + dist["DISTRITO"]
    return prov, dist

def rios_hydrorivers(carpeta, caudal_min):
    """Ríos de HydroRIVERS (Sudamérica) con caudal medio ≥ caudal_min m³/s, recortados a Perú."""
    import zipfile
    ruta = os.path.join(carpeta, "HydroRIVERS_v10_sa_shp.zip")
    if os.path.exists(ruta) and not zipfile.is_zipfile(ruta):
        os.remove(ruta)                                   # descarga anterior inválida (p. ej. una página HTML)
    bajar(HYDRORIVERS_SA, ruta)
    if not zipfile.is_zipfile(ruta):
        ini = open(ruta, "rb").read(200)
        os.remove(ruta)
        raise RuntimeError(f"HydroRIVERS no devolvió un ZIP (empieza con {ini[:60]!r}); vuelve a intentar más tarde")
    shp = [n for n in zipfile.ZipFile(ruta).namelist() if n.lower().endswith(".shp")]
    if not shp:
        raise RuntimeError("el ZIP de HydroRIVERS no contiene un .shp")
    g = gpd.read_file(f"zip://{ruta}!{shp[0]}", bbox=BBOX_PERU)   # el .shp está dentro de una carpeta del ZIP
    return g[g["DIS_AV_CMS"] >= caudal_min][["HYRIV_ID", "DIS_AV_CMS", "ORD_FLOW", "geometry"]].to_crs(4326)

def rios_osm_nombres():
    """Ríos principales de OpenStreetMap con nombre (relaciones waterway=river): consulta liviana."""
    q = """[out:json][timeout:300];area["ISO3166-1"="PE"][admin_level=2]->.pe;
    relation["waterway"="river"]["name"](area.pe);out geom;"""
    js = overpass(q)
    filas, geoms = [], []
    for e in js.get("elements", []):
        lineas = [LineString([(p["lon"], p["lat"]) for p in m_["geometry"]]) for m_ in e.get("members", [])
                  if m_.get("type") == "way" and len(m_.get("geometry") or []) >= 2]
        if lineas:
            filas.append({"RIO": e.get("tags", {}).get("name", "")}); geoms.append(MultiLineString(lineas))
    return gpd.GeoDataFrame(filas, geometry=geoms, crs=4326)

def rios_natural_earth(carpeta):
    ruta = bajar(NE_RIOS, os.path.join(carpeta, "ne_10m_rivers_lake_centerlines.zip"))
    g = gpd.read_file(f"zip://{ruta}", bbox=BBOX_PERU).to_crs(4326)
    g["RIO"] = g["name"].fillna(g.get("name_es")).fillna("")
    return g[["RIO", "geometry"]]

OVERPASS_SERVIDORES = ["https://overpass-api.de/api/interpreter", "https://overpass.kumi.systems/api/interpreter",
                       "https://overpass.private.coffee/api/interpreter"]

def overpass(consulta):
    """Consulta Overpass (OpenStreetMap) probando varios servidores; se identifica como pide su política de uso."""
    cab = {"User-Agent": "scraper-mef/1.0 (proyecto academico; github.com/adriansivina-droid/scraper-mef)",
           "Accept": "application/json,*/*"}
    errores = []
    for srv in OVERPASS_SERVIDORES:
        try:
            r = requests.post(srv, data={"data": consulta}, headers=cab, timeout=900)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            errores.append(f"{srv.split('/')[2]}: {str(e)[:80]}")
    raise RuntimeError("Overpass no respondió · " + " | ".join(errores))

def _osm_a_gdf(js, tipo):
    filas, geoms = [], []
    for e in js.get("elements", []):
        t = e.get("tags", {})
        if e["type"] == "way" and e.get("geometry"):
            coords = [(p["lon"], p["lat"]) for p in e["geometry"]]
            if tipo == "linea":
                g = LineString(coords)
            elif len(coords) >= 4 and coords[0] == coords[-1]:
                g = Polygon(coords)
            else:
                continue
        elif e["type"] == "relation" and e.get("members") and tipo == "poligono":
            ext = [[(p["lon"], p["lat"]) for p in m.get("geometry", [])] for m in e["members"]
                   if m.get("role") in ("outer", "") and m.get("geometry")]
            from shapely.ops import linemerge, polygonize, unary_union
            polys = list(polygonize(unary_union([LineString(c) for c in ext if len(c) >= 2])))
            if not polys:
                continue
            g = unary_union(polys)
        else:
            continue
        filas.append({"NOMBRE": t.get("name", ""), "OSM_ID": f"{e['type']}/{e['id']}", **{k: t.get(k, "") for k in
                     ("protect_class", "protection_title", "operator", "substance")}})
        geoms.append(g)
    return gpd.GeoDataFrame(filas, geometry=geoms, crs=4326)

def anp_osm():
    q = """[out:json][timeout:500];area["ISO3166-1"="PE"][admin_level=2]->.pe;
    (way["boundary"="protected_area"](area.pe);relation["boundary"="protected_area"](area.pe);
     way["leisure"="nature_reserve"](area.pe);relation["leisure"="nature_reserve"](area.pe););out geom;"""
    return _osm_a_gdf(overpass(q), "poligono")

def oleoducto_osm():
    q = """[out:json][timeout:300];area["ISO3166-1"="PE"][admin_level=2]->.pe;
    (way["man_made"="pipeline"]["substance"~"oil|crude",i](area.pe););out geom;"""
    return _osm_a_gdf(overpass(q), "linea")

# ── 4. Indicadores por lote ──────────────────────────────────────────────────
CRS_AREA = "ESRI:102033"        # Albers equivalente de Sudamérica (áreas en km²)
CRS_DIST = "EPSG:32718"         # UTM 18S (distancias en km; error < 1 % en la Amazonía peruana)
IQUITOS = Point(-73.2538, -3.7491)

_ROM = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}

def _romano(s):
    if not s or not re.fullmatch(r"M{0,3}(CM|CD|D?C{0,3})(XC|XL|L?X{0,3})(IX|IV|V?I{0,3})", s):
        return None
    total = 0
    for i, ch in enumerate(s):
        v = _ROM[ch]
        total += -v if i + 1 < len(s) and _ROM[s[i + 1]] > v else v
    return total

def lote_clave(t):
    """'LOTE 192' → '192'; 'LOTE_Z62' → 'Z-62'; 'Lote Z - 2B' → 'Z-2B'; '1 AB' → '1-AB'.
    Los lotes de la selva que vienen en números romanos (XCV, CIII, LXVII, VIII) pasan a arábigos; los del noroeste
    (I a XXIX) se mantienen en romanos, como en la producción."""
    s = norm_txt(str(t).replace("_", " "))
    s = re.sub(r"^(LOTE|BLOQUE|AREA|CONTRATO)\s+", "", s)
    s = re.sub(r"\s*-\s*", "-", s).replace(" ", "")
    s = re.sub(r"^([A-Z]{1,2})(\d)", r"\1-\2", s) if re.match(r"^(Z|XX?|CO)\d", s) else s
    s = re.sub(r"^(\d+)([A-Z]+)$", r"\1-\2", s) if re.match(r"^\d+[A-Z]{1,2}$", s) else s
    n = _romano(s)
    if n is not None and (n >= 30 or n == 8):          # 8 (VIII) y ≥ 30 son lotes de la selva numerados en arábigo
        s = str(n)
    return s

def campo_lote(gdf):
    cands = [c for c in gdf.columns if c != "geometry"]
    for pat in (r"^LOTE$", r"LOTE", r"^(NOMBRE|NAME|NOM)$", r"NOMBRE|NAME|CONTRATO|BLOQUE"):
        for c in cands:
            if re.search(pat, norm_txt(c)) and gdf[c].astype(str).str.len().gt(0).mean() > 0.5:
                return c
    return None

def campo(gdf, *patrones):
    for p in patrones:
        for c in gdf.columns:
            if c != "geometry" and re.search(p, norm_txt(c)):
                return c
    return None

def reparto_por_poligonos(lotes, capa, nombre_col, prefijo):
    """% del área de cada lote dentro de cada polígono de `capa` → principal, detalle y suma."""
    if capa is None or capa.empty:
        return pd.DataFrame(index=lotes.index)
    a = lotes[["geometry"]].to_crs(CRS_AREA)
    b = capa[[nombre_col, "geometry"]].to_crs(CRS_AREA)
    b = b[b.geometry.notna() & ~b.geometry.is_empty].copy()
    b["geometry"] = b.geometry.buffer(0)
    a = a.copy(); a["geometry"] = a.geometry.buffer(0); a["_ID"] = a.index
    inter = gpd.overlay(a, b, how="intersection", keep_geom_type=True)
    inter["_KM2"] = inter.geometry.area / 1e6
    area = a.geometry.area / 1e6
    res = {}
    for i, g in inter.groupby("_ID"):
        s = g.groupby(nombre_col)["_KM2"].sum().sort_values(ascending=False)
        pct = (s / area.loc[i] * 100).round(1)
        res[i] = {f"{prefijo}": s.index[0], f"{prefijo}_PCT": min(pct.iloc[0], 100.0),
                  f"{prefijo}_DETALLE": "; ".join(f"{k} {v:.1f}%" for k, v in pct.items() if v >= 0.1),
                  f"{prefijo}_N": int((pct >= 0.1).sum()), f"{prefijo}_PCT_TOTAL": min(round(pct.sum(), 1), 100.0)}
    return pd.DataFrame.from_dict(res, orient="index").reindex(lotes.index)

def distancia_km(lotes, capa, nombre_col=None):
    """Distancia mínima (km) de cada lote a la capa (0 si la toca) y, si hay nombre, el elemento más cercano."""
    if capa is None or capa.empty:
        return pd.Series(index=lotes.index, dtype=float), pd.Series(index=lotes.index, dtype=object)
    a = lotes.to_crs(CRS_DIST)
    b = capa[capa.geometry.notna()].to_crs(CRS_DIST)
    j = gpd.sjoin_nearest(a[["geometry"]], b[[c for c in [nombre_col] if c] + ["geometry"]], how="left",
                          distance_col="_D")
    j = j[~j.index.duplicated()]
    return (j["_D"] / 1000).round(1), (j[nombre_col] if nombre_col else pd.Series(index=lotes.index, dtype=object))

def comunidades(lotes, ccnn, nombre_col, radio_km=5):
    """Comunidades nativas dentro del lote y a menos de `radio_km` km (cuenta y nombres)."""
    if ccnn is None or ccnn.empty:
        return pd.DataFrame(index=lotes.index)
    a = lotes[["geometry"]].to_crs(CRS_DIST)
    b = ccnn[[nombre_col, "geometry"]].to_crs(CRS_DIST)
    b = b[b.geometry.notna()].copy()
    pol = b.geom_type.isin(["Polygon", "MultiPolygon"])
    b.loc[pol, "geometry"] = b.loc[pol, "geometry"].buffer(0)
    out = {}
    dentro = gpd.sjoin(a, b, predicate="intersects")
    cerca = gpd.sjoin(gpd.GeoDataFrame(geometry=a.geometry.buffer(radio_km * 1000), crs=a.crs), b, predicate="intersects")
    for i in lotes.index:
        d = dentro[dentro.index == i][nombre_col].dropna().astype(str).unique()
        c = cerca[cerca.index == i][nombre_col].dropna().astype(str).unique()
        out[i] = {"CCNN_DENTRO_N": len(d), f"CCNN_{radio_km}KM_N": len(c),
                  "CCNN_DENTRO": "; ".join(sorted(d)[:40]) + (" …" if len(d) > 40 else "")}
    return pd.DataFrame.from_dict(out, orient="index")


# ── 5. Producción por lote y por campo ───────────────────────────────────────
import calendar

def campos(geo):
    """Agrupa los lotes que comparten polígono (1-AB/192, 31-B/31-D/31-B/D…) en un mismo campo: componentes conexas."""
    padre = {}
    def raiz(x):
        while padre.setdefault(x, x) != x:
            padre[x] = padre[padre[x]]; x = padre[x]
        return x
    for lote, gd in zip(geo["LOTE"], geo["GEOMETRIA_DE"].astype(str)):
        for p in gd.split("+"):
            padre[raiz(lote)] = raiz(p)
    grupos = {}
    for lote in geo["LOTE"]:
        grupos.setdefault(raiz(lote), set()).add(lote)
    return {lote: grupo for grupo in grupos.values() for lote in grupo}

def _dias(t):                      # t = año*12 + mes-1
    return calendar.monthrange(t // 12, t % 12 + 1)[1]

def _mm_aaaa(t):
    return f"{t % 12 + 1:02d}/{t // 12}"

def indicadores_produccion(prod, clave, grupo=None):
    """Indicadores por lote o campo a partir de la producción mensual (todos los líquidos fiscalizados).
    prod: AÑO, MES, PRODUCTO, OPERADOR, BARRILES y la columna `clave`.
    grupo: columna del campo; los meses en que el campo produjo con otro contrato no cuentan como paralización."""
    p = prod.copy()
    p["T"] = p["AÑO"].astype(int) * 12 + p["MES"].astype(int) - 1
    fin, ini_datos = int(p["T"].max()), int(p["T"].min())
    anio_fin_completo = fin // 12 if fin % 12 == 11 else fin // 12 - 1
    activo_grupo = ({k: set(g.loc[g["BARRILES"] > 0, "T"]) for k, g in p.groupby(grupo)} if grupo else {})
    out = {}
    for k, g in p.groupby(clave):
        m = g.groupby("T")["BARRILES"].sum()
        pos = m[m > 0]
        prodp = g.groupby("PRODUCTO")["BARRILES"].sum()
        base = {"PETROLEO_ACUM_MMBBL": round(float(prodp.get("PETROLEO", 0)) / 1e6, 3),
                "LIQUIDOS_ACUM_MMBBL": round(float(prodp.sum()) / 1e6, 3),
                "PRODUCTO_PRINCIPAL": prodp.idxmax() if prodp.sum() > 0 else ""}
        if pos.empty:
            out[k] = {**base, "ESTADO_PRODUCCION": "SIN PRODUCCIÓN REGISTRADA"}; continue
        t0, t1 = int(pos.index.min()), int(pos.index.max())
        serie = m.reindex(range(t0, fin + 1), fill_value=0.0)
        bpd_m = pd.Series({t: v / _dias(t) for t, v in serie.items()})
        anual, completos = {}, []
        for a in range(t0 // 12, fin // 12 + 1):
            ts = [t for t in serie.index if t // 12 == a]
            anual[a] = sum(serie[t] for t in ts) / sum(_dias(t) for t in ts)
            if len(ts) == 12:
                completos.append(a)
        anual = pd.Series(anual)
        completos_pos = [a for a in completos if anual[a] > 0]
        pico = anual[completos_pos] if completos_pos else anual   # el pico se busca en años completos con producción
        ceros_anio = {a: sum(1 for t in serie.index if t // 12 == a and serie[t] == 0) for a in anual.index}
        ult12 = [t for t in serie.index if t > fin - 12]
        bpd_act = sum(serie[t] for t in ult12) / sum(_dias(t) for t in ult12)
        # declinación: variación anual compuesta de la producción diaria en 5 años, entre dos años "normales"
        # (≤ 2 meses sin producción) para que una paralización (2016, 2020…) no la distorsione
        y_fin = min(anio_fin_completo, t1 // 12 if t1 % 12 == 11 else t1 // 12 - 1)
        decl, periodo = None, ""
        for y1 in range(y_fin, y_fin - 4, -1):
            y0 = y1 - 5
            if all(y in anual and y in completos and anual[y] > 0 and ceros_anio[y] <= 2 for y in (y0, y1)):
                decl, periodo = round(((anual[y1] / anual[y0]) ** (1 / 5) - 1) * 100, 1), f"{y0}–{y1}"
                break
        cubiertos = activo_grupo.get(g[grupo].iloc[0], set()) if grupo else set()
        ceros = ((serie == 0) & ~serie.index.isin(list(cubiertos))).astype(int)
        racha, mejor, inicio, mejor_ini = 0, 0, None, None
        for t, c in ceros.items():
            if c:
                racha += 1; inicio = t if racha == 1 else inicio
                if racha > mejor:
                    mejor, mejor_ini = racha, inicio
            else:
                racha = 0
        ops = (g[(g["BARRILES"] > 0) & (g["OPERADOR"].fillna("").astype(str).str.strip() != "")].groupby("OPERADOR")["AÑO"].agg(["min", "max"]).sort_values(["min", "max"]))
        out[k] = {**base,
                  "ESTADO_PRODUCCION": "EN PRODUCCIÓN" if t1 > fin - 12 else "PARALIZADO",
                  "PRIMERA_PRODUCCION": ("≤" if t0 == ini_datos else "") + _mm_aaaa(t0),
                  "ULTIMA_PRODUCCION": _mm_aaaa(t1),
                  "BPD_ACTUAL": int(round(float(bpd_act))),
                  "PICO_AÑO": int(pico.idxmax()), "PICO_BPD": int(round(float(pico.max()))),
                  "DECLINACION_ANUAL_5A_PCT": None if decl is None else float(decl),
                  "DECLINACION_PERIODO": periodo,
                  "MESES_SIN_PRODUCCION": int(ceros.sum()),
                  "PARALIZACION_MAX_MESES": int(mejor),
                  "PARALIZACION_MAX_DESDE": _mm_aaaa(mejor_ini) if mejor_ini is not None else "",
                  "OPERADORES": " → ".join(f"{o} ({int(r['min'])}" + (f"–{int(r['max'])})" if r['max'] != r['min'] else ")")
                                          for o, r in ops.iterrows()),
                  "_SERIE_ANUAL_MMBBL": {int(a): round(float(v), 3) for a, v in (g.groupby("AÑO")["BARRILES"].sum() / 1e6).items()}}
    return out

def sparkline_svg(serie, anio_ini=None, anio_fin=None, ancho=220, alto=46, color="#b30000"):
    """Mini gráfico SVG de barras con la producción anual (millones de barriles), en el mismo eje de años para todos."""
    if not serie:
        return ""
    anios = list(range(anio_ini or min(serie), (anio_fin or max(serie)) + 1)); vmax = max(serie.values()) or 1
    w = ancho / len(anios)
    barras = "".join(f'<rect x="{i * w:.1f}" y="{alto - 12 - (serie.get(a, 0) / vmax) * (alto - 16):.1f}" width="{max(w - 1, 1):.1f}" '
                     f'height="{(serie.get(a, 0) / vmax) * (alto - 16):.1f}" fill="{color}"><title>{a}: {serie.get(a, 0):.2f} MM bbl</title></rect>'
                     for i, a in enumerate(anios))
    return (f'<svg width="{ancho}" height="{alto}" xmlns="http://www.w3.org/2000/svg">{barras}'
            f'<text x="0" y="{alto - 1}" font-size="9" fill="#555">{anios[0]}</text>'
            f'<text x="{ancho}" y="{alto - 1}" font-size="9" fill="#555" text-anchor="end">{anios[-1]}</text></svg>')


# ── 6. Operadores (misma homologación que el notebook de producción) ─────────
# Nombres de operadores: el PDF corta los nombres largos en dos renglones y tiene erratas; se homologan.
# Se conserva el nombre leído en OPERADOR_ORIGINAL.
OPERADOR_NORM = [
    (r"^PLUSPETRO$|^PLUSPETROL PERU( CORP\.?)?$", "PLUSPETROL"),
    (r"^PPN( EN( LIQUIDACI[OÓ]N)?)?$", "PLUSPETROL NORTE (EN LIQUIDACIÓN)"),
    (r"^UPLAND( OIL( AND( GAS)?)?)?$", "UPLAND OIL AND GAS"),
    (r"^CORPORACI[OÓ]N DEL( SUR)?$", "CORPORACIÓN DEL SUR"),
    (r"^PETROLERA MONTERR+ICO$|^PETROMONT$", "PETROLERA MONTERRICO"),   # PETROMONT = su nombre corto
    (r"^UNIENE?R?G[IÍ]A( ABC)?$", "UNIENERGÍA ABC"),
    (r"^UNIPETRO( ABC)?$", "UNIPETRO ABC"),
    (r"^PETRO-?TECH$", "PETROTECH"),
    (r"^G\.? ?M\.? ?P\.?( S\.? ?A\.?)?$", "GMP"),
    (r"^PETROPER[UÚ]$", "PETROPERÚ"),
    (r"^VIGO( ENERGY)?$", "VIGO ENERGY"),
    (r"^PACIFIC( STRATUS)?$", "PACIFIC STRATUS"),
    (r"^(DPB|BPD|M?M?BBL|LBB)$", ""),               # rótulos de gráficos leídos como operador
]

def fecha_dmy(v):
    """Fecha de un atributo (epoch en milisegundos de ArcGIS, ISO del WFS o texto) → dd/mm/aaaa."""
    if v is None or (isinstance(v, float) and v != v) or str(v).strip() in ("", "None", "nan", "NaT"):
        return ""
    x = str(v).strip()
    try:
        if re.fullmatch(r"-?\d{9,14}(\.0+)?", x):                     # epoch ms (o s)
            n = float(x)
            t = pd.to_datetime(n, unit="ms" if abs(n) > 1e11 else "s", utc=True)
        elif re.match(r"\d{4}-\d{2}-\d{2}", x):                       # ISO
            t = pd.to_datetime(x[:10])
        elif re.fullmatch(r"\d{1,2}[/.-]\d{1,2}[/.-]\d{4}", x):       # ya viene día/mes/año
            t = pd.to_datetime(x, dayfirst=True)
        else:
            return x
        return t.strftime("%d/%m/%Y")
    except Exception:
        return x

def norm_operador(nombre):
    """Homologa un nombre de operador; los consorcios 'A / B' se homologan por partes."""
    partes = []
    if nombre is None or (isinstance(nombre, float) and nombre != nombre):    # None / NaN
        return ""
    for parte in str(nombre).split("/"):
        x = re.sub(r"\s+", " ", parte).strip().upper()
        for patron, bueno in OPERADOR_NORM:
            if re.match(patron, x):
                x = bueno
                break
        if x and x not in ("NAN", "NONE"):
            partes.append(x)
    return " / ".join(partes)



print("✅ Funciones listas")

In [ ]:
# ═══ BLOQUE 2 · DESCUBRIR CAPAS Y ELEGIR FUENTES ═══
PORTAL = "https://perupetro.maps.arcgis.com"
APP_ID = "cc883917c55a4264a01debfa02585e2e"          # visor "mapa de lotes" de PERUPETRO
URLS_EXTRA = []        # opcional: URLs de servicios MapServer/FeatureServer adicionales (una por elemento)
BUSQUEDAS = {          # búsquedas en ArcGIS Online (respaldo y capas que no son de PERUPETRO)
    "LOTES": "lotes hidrocarburos Peru",
    "LOTES2": "lotes de contrato PERUPETRO",
    "LOTES3": "lotes petroleros Peru",
    "LOTES4": "contratos hidrocarburos Peru lotes",
    "ANP": "areas naturales protegidas SERNANP",
    "CCNN": "comunidades nativas Peru",
    "OLEODUCTO": "oleoducto norperuano",
}
GEOM_TEMA = {"LOTES": ("Polygon",), "LOTES2": ("Polygon",), "LOTES3": ("Polygon",), "LOTES4": ("Polygon",), "ANP": ("Polygon",), "CCNN": ("Polygon", "Point"), "OLEODUCTO": ("Polyline",)}

print("🔎 Leyendo el visor de PERUPETRO…")
try:
    WEBMAP_ID, WEBMAP_TIT = mapa_web_de_app(PORTAL, APP_ID)
    print(f"   Mapa web: «{WEBMAP_TIT}» ({WEBMAP_ID})")
    capas_pp = capas_del_mapa(PORTAL, WEBMAP_ID)
except Exception as e:                       # p. ej. desde GitHub Actions ArcGIS responde «Invalid URL»: sigue con el geoportal
    print(f"   ⚠️ El visor ArcGIS de PERUPETRO no respondió ({e}); se usan el geoportal (WFS) y los polígonos guardados en Drive.")
    WEBMAP_ID, WEBMAP_TIT, capas_pp = None, "", []
for u in URLS_EXTRA:
    try:
        capas_pp += capas_de_servicio(u, "EXTRA")
    except Exception as e:
        print(f"   ⚠️ {u}: {e}")
for c in capas_pp:
    c["FUENTE"], c["TEMA"] = "PERUPETRO", ""

# Geoportal oficial de PERUPETRO (GeoServer): WFS con lotes de contrato vigentes, cuencas, ANP, comunidades…
GEOSERVER = "https://geoportal.perupetro.com.pe/geoserver"
UTILES_WFS = r"LOTES?_CONTRATO$|CUENCAS?.SEDIMENT|AREAS?.NATURAL|COMUNIDAD|PUEBLOS?.INDIGENA|RESERVAS?.TERRITORIAL|OLEODUCTO|^DUCTOS$|YACIMIENTOS$|AREA.PROMOCION|CONVENIO|ARCOS?$"
EXCLUIR_WFS = r"_(INICIO|FIN|F|NW|PASH|PUNTOS)$|ACUEDUCTO|GASODUCTO|POLIDUCTO"
TEMAS_GEOPORTAL = set()
try:
    wfs = wfs_capas(GEOSERVER)
    utiles = [c for c in wfs if re.search(UTILES_WFS, norm_txt(c["WFS"].split(":")[-1]).replace(" ", "_"))
              and not re.search(EXCLUIR_WFS, norm_txt(c["WFS"].split(":")[-1]).replace(" ", "_"))]
    try:
        wfs_describir(GEOSERVER, utiles)                 # una sola petición para todas
    except Exception:
        for c in utiles:                                  # respaldo: una por una
            try:
                wfs_describir(GEOSERVER, c)
            except Exception as e:
                c["ERROR"] = f"DescribeFeatureType: {e}"[:200]
    for c in utiles:
        c["FUENTE"], c["TEMA"] = "PERUPETRO", ""
        t = norm_txt(c["WFS"])
        TEMAS_GEOPORTAL |= {k for k, p in [("ANP", r"NATURAL"), ("CCNN", r"COMUNIDAD|INDIGENA"), ("OLEODUCTO", r"OLEODUCTO")]
                            if re.search(p, t)}
    capas_pp = [c for c in utiles if c["GEOMETRIA"]] + capas_pp      # el geoportal va primero (prioridad)
    print(f"🛰️ Geoportal PERUPETRO (WFS): {len(wfs)} capas publicadas · {len(utiles)} útiles: "
          f"{', '.join(c['WFS'].split(':')[-1] for c in utiles)}")
except Exception as e:
    print(f"⚠️ Geoportal PERUPETRO (WFS) no disponible: {e}")

# Si la capa del visor no responde, se revisan los demás servicios del mismo servidor de PERUPETRO
fallidas = [c for c in capas_pp if c.get("ERROR")]
if fallidas:
    print("\n⚠️ Capas del visor que no responden (también fallan en el navegador):")
    for c in fallidas:
        print(f"   · {c['CAPA']}: {c['ERROR']}\n     {c['URL']}")
    vistos = set()
    for c in fallidas:
        try:
            servs = servicios_del_servidor(c["URL"])
        except Exception as e:
            print(f"   ↳ No se pudo listar el servidor: {e}"); continue
        print(f"   ↳ El servidor responde y tiene {len(servs)} servicios; reviso los de lotes/hidrocarburos…")
        for u in servs:
            if u in vistos or not re.search(r"LOTE|CONTRAT|MAPA.?BASE|HIDROCARB|BLOQUE", norm_txt(u)):
                continue
            vistos.add(u)
            try:
                for h in capas_de_servicio(u, u.split("/services/")[1]):
                    h["FUENTE"], h["TEMA"] = "PERUPETRO", ""; capas_pp.append(h)
            except Exception as e:
                print(f"     · {u}: {e}")

PATRON_TEMA = {"LOTES": r"\bLOTES?\b|CONTRAT|BLOQUES? PETROL|HIDROCARB",
               "ANP": r"AREAS? NATURAL|PROTEGID|\bANP\b|SERNANP|RESERVA|PARQUE NACIONAL",
               "CCNN": r"COMUNIDAD(ES)? NATIVA|CC\.?NN|PUEBLOS? INDIGENA|INDIGEN",
               "OLEODUCTO": r"OLEODUCTO|\bONP\b|NORPERUANO|DUCTO"}
def texto(c):
    return norm_txt(f"{c['CAPA']} {c['GRUPO']} {c['URL']} {c['FUENTE']}")

capas_ext = []
for tema, q in BUSQUEDAS.items():
    if tema.rstrip("234") in TEMAS_GEOPORTAL:
        continue                                          # el geoportal oficial ya trae este tema
    try:
        res = buscar_arcgis(q)
    except Exception as e:
        print(f"   ⚠️ búsqueda {tema}: {e}"); res = []
    base = tema.rstrip("234")
    for r in res[:5]:
        try:
            for c in capas_de_servicio(r["URL"], r["TITULO"]):
                c.update({"FUENTE": f"ArcGIS Online · {r['DUEÑO']}", "TEMA": base})
                # solo capas cuyo nombre corresponde al tema (evita cientos de capas ajenas del mismo servicio)
                if (any(g in c["GEOMETRIA"] for g in GEOM_TEMA[tema]) and re.search(PATRON_TEMA[base], texto(c))
                        and c["URL"] not in {x["URL"] for x in capas_ext}):
                    capas_ext.append(c)
        except Exception as e:
            pass


def etiqueta(c):
    g = c["GEOMETRIA"].replace("esriGeometry", "")
    return f"{c['CAPA']}  [{g}]  ·  {c['GRUPO'][:50]}  ·  {c['FUENTE'][:40]}"

capas = capas_pp + capas_ext
for i, c in enumerate(capas):
    c["N"] = i
POR_N_ = {c["N"]: c for c in capas}
cat = pd.DataFrame([{k: v for k, v in c.items() if not k.startswith("_")} for c in capas])
for _k in ("N", "CAPA", "GRUPO", "GEOMETRIA", "TIPO", "FUENTE", "TEMA", "URL"):    # sin capas (servidores caídos): tabla vacía
    if _k not in cat:
        cat[_k] = pd.Series(dtype=object)
pd.set_option("display.max_colwidth", 70)
print(f"\n📋 Capas de PERUPETRO: {len(capas_pp)}")
display(cat[cat["FUENTE"] == "PERUPETRO"][["N", "CAPA", "GRUPO", "GEOMETRIA", "TIPO"] + (["ERROR"] if "ERROR" in cat else [])])
print(f"\n📋 Candidatas externas (filtradas por tema): {len(capas_ext)}")
if capas_ext:
    with pd.option_context("display.max_rows", 200):
        display(cat[cat["FUENTE"] != "PERUPETRO"][["N", "TEMA", "CAPA", "GRUPO", "GEOMETRIA", "FUENTE"]])

# ── Propuesta automática (revísala) ─────────────────────────────────────────
con_geo = [c for c in capas_pp if c["GEOMETRIA"] and not c.get("ERROR")]
poligonos = [c for c in con_geo if "Polygon" in c["GEOMETRIA"]]
es_lote = lambda c: re.search(r"LOTE|CONTRAT|BLOQUE|ASIGNAC|EXPLOTAC|EXPLORAC", texto(c)) and not re.search(r"CUENCA|SISMIC", texto(c))
def mejor(cands, patron, extra=r"SERNANP|CULTURA|MINCU|GOB.PE|MINEM|OSINERG"):
    puntaje = lambda c: (bool(re.search(patron, texto(c))) * 2 + bool(re.search(extra, texto(c))))
    cands = sorted(cands, key=puntaje, reverse=True)
    return cands[0]["N"] if cands and puntaje(cands[0]) >= 2 else None

OSM = "OpenStreetMap (respaldo)"; NINGUNA = "(ninguna)"
_l = {"layout": w.Layout(width="95%")}
sel_capas = w.SelectMultiple(options=[(etiqueta(c), c["N"]) for c in con_geo],
                             value=[c["N"] for c in con_geo if not c.get("WFS")],      # las del geoportal: solo las que se usan
                             rows=min(12, max(3, len(con_geo))), description="Guardar extra", **_l)
lotes_ext = [c for c in capas_ext if c["TEMA"] == "LOTES"]
_prio = {"LOTE_CONTRATO": 0, "LOTES_CONTRATO": 1}            # contratos vigentes primero
_wl = sorted([c for c in poligonos if c.get("WFS") and norm_txt(c["WFS"].split(":")[-1]) in _prio],
             key=lambda c: _prio[norm_txt(c["WFS"].split(":")[-1])])
_lv = [c["N"] for c in _wl] or [c["N"] for c in poligonos if not c.get("WFS") and es_lote(c)]
_m = mejor(lotes_ext, r"LOTE", r"PERUPETRO|MINEM|OSINERG|GOB.PE|INGEMMET|GEOCATMIN")
if not _lv:                                   # PERUPETRO no da lotes: mejor candidata externa
    _lv = [_m] if _m is not None else []
    print("\n⚠️ PERUPETRO no entregó polígonos de lotes; propongo:",
          POR_N_[_lv[0]]["CAPA"] + " · " + POR_N_[_lv[0]]["FUENTE"] if _lv else "ninguna (agrega una URL en URLS_EXTRA)")
elif _m is not None:                          # complemento: lotes antiguos que ya no tienen contrato vigente
    _lv.append(_m)
    print(f"\nℹ️ Lotes: geoportal de PERUPETRO (vigentes) + {POR_N_[_m]['CAPA']} ({POR_N_[_m]['FUENTE']}) para lotes antiguos; "
          "si un lote está en ambas, manda el de PERUPETRO.")
sel_lotes = w.SelectMultiple(options=[(etiqueta(c), c["N"]) for c in poligonos + lotes_ext], value=_lv,
                             rows=min(10, max(3, len(poligonos) + len(lotes_ext))), description="Son lotes", **_l)
_cu = mejor(poligonos, r"CUENCA")
dd_cuenca = w.Dropdown(options=[(NINGUNA, None)] + [(etiqueta(c), c["N"]) for c in poligonos], value=_cu,
                       description="Cuencas", **_l)
def dd_tema(tema, patron, respaldo):
    pp = [c for c in con_geo if any(g in c["GEOMETRIA"] for g in GEOM_TEMA[tema])]
    ex = [c for c in capas_ext if c["TEMA"] == tema]
    ops = [(etiqueta(c), c["N"]) for c in pp + ex] + [(respaldo, respaldo)]
    val = mejor(pp, patron) if mejor(pp, patron) is not None else mejor(ex, patron)
    return w.Dropdown(options=ops, value=val if val is not None else respaldo, description=tema, **_l)
dd_anp = dd_tema("ANP", r"AREA(S)? NATURAL|PROTEGID|\bANP\b", OSM)
dd_ccnn = dd_tema("CCNN", r"COMUNIDAD(ES)?.(NATIVA|AMAZONICA)|CC\.?NN", NINGUNA)
if dd_ccnn.value == NINGUNA:
    dd_ccnn.value = (lambda v: v if v is not None else NINGUNA)(
        mejor([c for c in con_geo + capas_ext if any(g in c["GEOMETRIA"] for g in GEOM_TEMA["CCNN"])], r"COMUNIDAD|PUEBLOS? INDIGENA"))
dd_onp = dd_tema("OLEODUCTO", r"NORPERUANO|\bONP\b", OSM)
if dd_onp.value == OSM:                       # ninguna se llama "Norperuano": cualquier oleoducto
    dd_onp.value = (lambda v: v if v is not None else OSM)(
        mejor([c for c in con_geo + capas_ext if "Polyline" in c["GEOMETRIA"]], r"OLEODUCTO|DUCTO"))
txt_caudal = w.IntText(value=5000, description="Río grande: caudal medio ≥ (m³/s)", style={"description_width": "260px"})
display(w.VBox([w.HTML("<b>Capas de PERUPETRO a descargar</b> (Ctrl/Cmd + clic para cambiar la selección)"), sel_capas,
                w.HTML("<b>¿Cuáles son polígonos de lotes?</b>"), sel_lotes,
                w.HTML("<b>Fuentes para los indicadores</b>"), dd_cuenca, dd_onp, dd_anp, dd_ccnn, txt_caudal]))

def _nom(v):
    return v if isinstance(v, str) or v is None else f"[{v}] {POR_N_[v]['CAPA']} · {POR_N_[v]['FUENTE']}"
print("\n✅ Propuesta que usará el Bloque 3 (cámbiala en los selectores o con los números N al inicio del Bloque 3):")
print("   Lotes:     ", "; ".join(_nom(v) for v in sel_lotes.value) or "NINGUNA ⚠️")
print("   Cuencas:   ", _nom(dd_cuenca.value) or "(ninguna)")
print("   Oleoducto: ", _nom(dd_onp.value))
print("   ANP:       ", _nom(dd_anp.value))
print("   Com. nat.: ", _nom(dd_ccnn.value))

cat_ruta = os.path.join(ruta, "CONTROL PP mapa capas.csv")
cat.assign(FECHA_CONSULTA=date.today().strftime("%d/%m/%Y")).to_csv(cat_ruta, index=False, encoding="utf-8-sig")
subir(cat_ruta, DRIVE_FOLDER_CONTROL_ID, control_drive)
print("☁️  Catálogo de capas guardado en CONTROL. Revisa la selección y ejecuta el Bloque 3.")

In [ ]:
# ═══ BLOQUE 3 · DESCARGA Y VARIABLES POR LOTE ═══
# Lote de producción → polígono(s) del mapa del mismo campo (se suman si son varios)
LOTES_ALIAS_GEOMETRIA = {"1-AB": ["192"], "Z-69": ["Z-2B"], "Z-2B": ["Z-69"], "31-B/D": ["31-B", "31-D"]}
# Lotes que se integraron a otro sin compartir polígono (nota del anual 1996 de PERUPETRO)
SUCESION_MANUAL = {"LOB/EA/LO": "X (Folche y Cruz al VI), desde 24/07/1996"}
# Si los selectores no se ven o quieres otra fuente: números N de las tablas del Bloque 2 (vacío/None = propuesta)
N_LOTES = []          # p. ej. [14, 15]
N_CUENCAS = None
N_OLEODUCTO = None    # o "OpenStreetMap (respaldo)"
N_ANP = None
N_CCNN = None
SEL_LOTES = list(N_LOTES) or list(sel_lotes.value)
V_CUENCA = N_CUENCAS if N_CUENCAS is not None else dd_cuenca.value
V_ONP = N_OLEODUCTO if N_OLEODUCTO is not None else dd_onp.value
V_ANP = N_ANP if N_ANP is not None else dd_anp.value
V_CCNN = N_CCNN if N_CCNN is not None else dd_ccnn.value
POR_N = {c["N"]: c for c in capas}
log = []

def guardar_geojson(gdf, nombre):
    p = os.path.join(ruta, nombre)
    gdf.to_file(p, driver="GeoJSON")
    subir(p, MAPA_FOLDER_ID, mapa_drive, mime="application/geo+json")

def capa_gdf(n, cache={}):
    if n in cache:
        return cache[n]
    c = POR_N[n]
    print(f"⬇️  {c['CAPA']} …", end=" ")
    try:
        g = descargar_capa(c)
        print(f"{len(g)} entidades")
        guardar_geojson(g, f"{'PP' if c['FUENTE'] == 'PERUPETRO' else 'EXT'}_{slug(c['CAPA'])}.geojson")
        log.append({"CAPA": c["CAPA"], "URL": c["URL"], "FUENTE": c["FUENTE"], "ENTIDADES": len(g), "ESTADO": "OK"})
    except Exception as e:
        print(f"❌ {e}"); g = None
        log.append({"CAPA": c["CAPA"], "URL": c["URL"], "FUENTE": c["FUENTE"], "ENTIDADES": 0, "ESTADO": f"ERROR: {e}"[:200]})
    cache[n] = g
    return g

for n in sel_capas.value:
    capa_gdf(n)

# ── 1. Lotes ─────────────────────────────────────────────────────────────────
partes = []
for n in SEL_LOTES:
    g = capa_gdf(n)
    if g is None or g.empty:
        continue
    f_lote = campo_lote(g)
    if not f_lote:
        print(f"⚠️ {POR_N[n]['CAPA']}: no encontré el campo con el nombre del lote. Campos: {list(g.columns)}"); continue
    f_op = campo(g, r"OPERADOR|CONTRATISTA|EMPRESA|COMPANIA|TITULAR|CIA_OPER|^CIA")
    f_est = campo(g, r"ESTADO|SITUACION|FASE|ETAPA|^CONTRATO$")
    f_cu = campo(g, r"CUENCA")
    f_tipo = campo(g, r"TIPO_CONTR|MODALIDAD")
    f_fec = campo(g, r"F_SUSCRIP|FECHA_SUSC|SUSCRIP")
    txt = lambda f: g[f].astype(str).replace({"None": "", "nan": ""}) if f else ""
    p_ = gpd.GeoDataFrame({
        "LOTE": g[f_lote].map(lote_clave), "LOTE_MAPA": g[f_lote].astype(str),
        "OPERADOR_MAPA": txt(f_op), "ESTADO_MAPA": txt(f_est), "TIPO_CONTRATO": txt(f_tipo), "F_SUSCRIPCION": g[f_fec].map(fecha_dmy) if f_fec else "",
        "CUENCA_ATRIBUTO": txt(f_cu), "CAPA": POR_N[n]["CAPA"], "FUENTE_POLIGONO": POR_N[n]["FUENTE"],
        "_PRIORIDAD": len(partes)}, geometry=g.geometry, crs=4326)
    p_ = p_[p_.geometry.notna() & ~p_.geometry.is_empty & p_["LOTE"].ne("")]
    p_["geometry"] = p_.geometry.buffer(0)
    partes.append(p_.dissolve("LOTE", aggfunc="first").reset_index())     # una fila por lote y fuente
if not partes and "PP_lotes_indicadores.geojson" in mapa_drive:
    # respaldo: los polígonos de la corrida anterior (DATA/MAPA/PP_lotes_indicadores.geojson), si hoy no responde ninguna fuente
    _pg = os.path.join(ruta, "PP_lotes_indicadores_previo.geojson")
    bajar_de_drive(mapa_drive["PP_lotes_indicadores.geojson"], _pg)
    g = gpd.read_file(_pg).to_crs(4326)
    if "GEOMETRIA_DE" in g:
        g = g[g["GEOMETRIA_DE"].astype(str) == g["LOTE"].astype(str)]          # sin los ubicados por equivalencia (se rehacen)
    _c = lambda k: g[k].astype(str).replace({"None": "", "nan": ""}) if k in g else ""
    p_ = gpd.GeoDataFrame({"LOTE": g["LOTE"].astype(str), "LOTE_MAPA": _c("LOTE_MAPA"), "OPERADOR_MAPA": _c("OPERADOR_MAPA"),
                           "ESTADO_MAPA": _c("ESTADO_MAPA"), "TIPO_CONTRATO": _c("TIPO_CONTRATO"), "F_SUSCRIPCION": _c("F_SUSCRIPCION"),
                           "CUENCA_ATRIBUTO": _c("CUENCA_ATRIBUTO"), "CAPA": "Corrida anterior (Drive)", "FUENTE_POLIGONO": "PERUPETRO (copia en Drive)",
                           "_PRIORIDAD": 0}, geometry=g.geometry, crs=4326)
    partes.append(p_[p_.geometry.notna() & ~p_.geometry.is_empty & p_["LOTE"].ne("")])
    print(f"   ⚠️ Ninguna fuente de lotes respondió hoy: se usan los {len(partes[0])} polígonos de la corrida anterior (DATA/MAPA).")
if not partes:
    raise RuntimeError("No hay polígonos de lotes: el servidor de PERUPETRO no responde y no se eligió otra fuente. "
                       "Elige una capa de lotes en el Bloque 2 (o N_LOTES arriba), agrega una URL en URLS_EXTRA, "
                       "o usa el shapefile de PERUPETRO (Banco de Datos / promocion@perupetro.com.pe).")
lotes = pd.concat(partes, ignore_index=True).sort_values("_PRIORIDAD")
lotes = lotes.drop_duplicates("LOTE", keep="first").reset_index(drop=True)                            # manda la 1.ª fuente

print("   Polígonos por fuente: " + " · ".join(f"{k}: {v}" for k, v in lotes["CAPA"].value_counts().items()))
lotes["GEOMETRIA_DE"] = lotes["LOTE"]

# Producción (último CSV del notebook de producción): para ubicar los lotes con otro nombre en el mapa
def _cobertura(nombre):
    """(-año inicio, año fin) de «…_<ini>-<fin>.csv»: se prefiere el archivo que cubre MÁS años
    (una corrida rápida «2026-2026» no debe reemplazar a la serie «1994-2026»)."""
    a = re.findall(r"(\d{4})-(\d{4})", nombre)
    return (-int(a[-1][0]), int(a[-1][1])) if a else (0, 0)

prod_csv = sorted([k for k in datos_drive if k.startswith("PP_produccion_lote_mes_") and k.endswith(".csv")], key=_cobertura)
prod = None
if prod_csv:
    p = os.path.join(ruta, prod_csv[-1]); bajar_de_drive(datos_drive[prod_csv[-1]], p)
    prod = pd.read_csv(p, encoding="utf-8-sig")
    prod["OPERADOR"] = prod["OPERADOR"].map(norm_operador)    # por si el CSV es anterior a la homologación
# Gas natural fiscalizado (MPC), del mismo notebook de producción (1994–hoy)
gas_csv = sorted([k for k in datos_drive if k.startswith("PP_produccion_gas_lote_mes_") and k.endswith(".csv")], key=_cobertura)
gas = None
if gas_csv:
    p = os.path.join(ruta, gas_csv[-1]); bajar_de_drive(datos_drive[gas_csv[-1]], p)
    gas = pd.read_csv(p, encoding="utf-8-sig", dtype={"LOTE": str})
    gas["OPERADOR"] = gas["OPERADOR"].map(norm_operador)
# Producción diaria (notebook de producción, sección 8): para la vista «Diaria» del panel
diaria = None
if "PP_produccion_diaria.csv" in datos_drive:
    p = os.path.join(ruta, "PP_produccion_diaria.csv"); bajar_de_drive(datos_drive["PP_produccion_diaria.csv"], p)
    diaria = pd.read_csv(p, encoding="utf-8-sig", dtype={"LOTE": str, "LOTE_ORIGINAL": str})
# Un polígono de una fuente secundaria que ya está cubierto (> 50 % de su área) por la fuente principal es el mismo
# lote con otro nombre o dividido en sectores (XIII-A/B/C ⊂ XIII; Z-2B A..D ⊂ Z-69): se descarta, salvo que tenga
# producción propia (entonces es un lote antiguo distinto y se conserva).
_pri = lotes[lotes["_PRIORIDAD"] == lotes["_PRIORIDAD"].min()]
if len(_pri) and (lotes["_PRIORIDAD"] > lotes["_PRIORIDAD"].min()).any():
    _u = _pri.to_crs(CRS_AREA).geometry.union_all() if hasattr(_pri.geometry, "union_all") else _pri.to_crs(CRS_AREA).geometry.unary_union
    _sec = lotes["_PRIORIDAD"] > lotes["_PRIORIDAD"].min()
    _g = lotes.loc[_sec].to_crs(CRS_AREA).geometry
    _cubierto = (_g.intersection(_u).area / _g.area.where(_g.area > 0)).fillna(0) > 0.5
    _con_prod = set(prod["LOTE"].astype(str)) if prod is not None else set()
    _fuera = [i for i in _cubierto[_cubierto].index if lotes.loc[i, "LOTE"] not in _con_prod]
    if len(_fuera):
        print(f"   Polígonos secundarios ya cubiertos por la fuente principal (se descartan): {', '.join(lotes.loc[_fuera, 'LOTE'])}")
        lotes = lotes.drop(index=_fuera).reset_index(drop=True)
lotes = lotes.drop(columns="_PRIORIDAD")
en_mapa = set(lotes["LOTE"])
def poligonos_de(lote):
    """Lotes del mapa que corresponden a un lote de producción: alias, sin sufijo (T)/(TT) o partes de 'VII/VI'."""
    base = re.sub(r"\(T+\)$", "", lote)
    for cand in (LOTES_ALIAS_GEOMETRIA.get(lote), [base] if base != lote else None,
                 LOTES_ALIAS_GEOMETRIA.get(base), base.split("/") if "/" in base else None):
        if cand and all(c in en_mapa for c in cand):
            return cand
    if cand := [c for c in (LOTES_ALIAS_GEOMETRIA.get(lote) or LOTES_ALIAS_GEOMETRIA.get(base) or []) if c in en_mapa]:
        return cand                                       # al menos una parte está en el mapa
    for nombre in [base] + (LOTES_ALIAS_GEOMETRIA.get(base) or []):
        # el mapa divide el lote en sectores: Z-2B → Z-2B A/B/C/D, XIII → XIII-A/B/C
        if cand := sorted(c for c in en_mapa if re.fullmatch(re.escape(nombre) + r"-?[A-Z]", c)):
            return cand
    return None
extra = []
for lote in sorted(set(prod["LOTE"].astype(str)) if prod is not None else set(LOTES_ALIAS_GEOMETRIA)):
    if lote in en_mapa:
        continue
    cand = poligonos_de(lote)
    if cand:
        f = lotes[lotes["LOTE"].isin(cand)]
        extra.append({**f.iloc[0].drop("geometry").to_dict(), "LOTE": lote, "GEOMETRIA_DE": "+".join(cand),
                      "geometry": f.geometry.union_all() if hasattr(f.geometry, "union_all") else f.geometry.unary_union})
if extra:
    lotes = pd.concat([lotes, gpd.GeoDataFrame(extra, geometry="geometry", crs=4326)], ignore_index=True)
lotes = gpd.GeoDataFrame(lotes, geometry="geometry", crs=4326)
print(f"\n🧩 Lotes con polígono: {len(en_mapa)} en el mapa + {len(extra)} ubicados por equivalencia "
      f"({', '.join(e['LOTE'] + '→' + e['GEOMETRIA_DE'] for e in extra) or 'ninguno'})")

# ── 2. Capas externas ────────────────────────────────────────────────────────
def externa(nombre, fn):
    print(f"⬇️  {nombre} …", end=" ")
    try:
        g = fn(); print(f"{len(g)} entidades")
        guardar_geojson(g, f"EXT_{slug(nombre)}.geojson")
        log.append({"CAPA": nombre, "URL": "", "FUENTE": "externa", "ENTIDADES": len(g), "ESTADO": "OK"})
        return g
    except Exception as e:
        print(f"❌ {e}")
        log.append({"CAPA": nombre, "URL": "", "FUENTE": "externa", "ENTIDADES": 0, "ESTADO": f"ERROR: {e}"[:200]})
        return None

def elegida(v, respaldo_fn, nombre):
    if v in (None, NINGUNA):
        return None, None
    if v == OSM:
        return externa(f"{nombre} OSM", respaldo_fn), "NOMBRE"
    g = capa_gdf(v)
    return g, (campo(g, r"^(NOMBRE|NOM_ANP|ANP_NOMB|NOMB_CCNN|NOM_COMUNI|NAME)$", r"NOMB|NOMBRE|NAME|DENOMIN") if g is not None else None)

deps = externa("Departamentos geoBoundaries", lambda: departamentos(ruta))
# provincias y distritos (geoBoundaries ADM2 / ADM3): para cruzar el lote con el canon de sus municipalidades
provs = dists = None
if deps is not None:
    try:
        print("⬇️  Provincias y distritos (geoBoundaries; respaldo GADM) …", end=" ")
        provs, dists = provincias_distritos(ruta, deps)
        print(f"{len(provs)} provincias · {len(dists)} distritos")
        log.append({"CAPA": "Provincias y distritos geoBoundaries", "URL": "", "FUENTE": "externa",
                    "ENTIDADES": len(dists), "ESTADO": "OK"})
    except Exception as e:
        print(f"❌ {e}")
        log.append({"CAPA": "Provincias y distritos geoBoundaries", "URL": "", "FUENTE": "externa", "ENTIDADES": 0,
                    "ESTADO": f"ERROR: {e}"[:200]})
# Umbrales amazónicos (m³/s): 1.000 afluentes navegables, 5.000 ríos troncales, 20.000 Amazonas / Marañón y Ucayali bajos
CAUDAL = int(txt_caudal.value or 5000)
UMBRALES = sorted({1000, 5000, 20000, CAUDAL})
rios_q = externa("Rios HydroRIVERS", lambda: rios_hydrorivers(ruta, min(UMBRALES)))
if rios_q is not None and deps is not None:          # solo ríos dentro de Perú
    rios_q = gpd.clip(rios_q, deps.dissolve()[["geometry"]]).explode(index_parts=False)
    rios_q = rios_q[rios_q.geom_type.isin(["LineString", "MultiLineString"])]
rios_n = externa("Rios OSM con nombre", rios_osm_nombres)
if rios_n is None:
    rios_n = externa("Rios Natural Earth", lambda: rios_natural_earth(ruta))
onp, onp_n = elegida(V_ONP, oleoducto_osm, "Oleoducto")
anp, anp_n = elegida(V_ANP, anp_osm, "ANP")
ccnn, ccnn_n = elegida(V_CCNN, None, "Comunidades nativas")
cuencas = capa_gdf(V_CUENCA) if V_CUENCA is not None else None
cuenca_n = campo(cuencas, r"CUENCA", r"NOMB|NOMBRE|NAME") if cuencas is not None else None

# ── 3. Indicadores ───────────────────────────────────────────────────────────
geo = lotes.copy()
geo["AREA_KM2"] = (geo.to_crs(CRS_AREA).geometry.area / 1e6).round(1)
rp = geo.geometry.representative_point()
geo["LAT"], geo["LON"] = rp.y.round(5), rp.x.round(5)
if deps is not None:
    geo = geo.join(reparto_por_poligonos(geo, deps, "DEPARTAMENTO", "DEPARTAMENTO"))
    sin = geo["DEPARTAMENTO"].isna()
    # zócalo: lote mayormente en el mar (la línea de costa generalizada puede tocar una franja de tierra)
    geo["ZOCALO"] = (sin | (geo["DEPARTAMENTO_PCT_TOTAL"].fillna(0) < 50)).map({True: "SI", False: "NO"})
    geo.loc[(geo["ZOCALO"] == "SI") & ~sin, "DEPARTAMENTO_DETALLE"] = (
        "ZÓCALO; en tierra: " + geo.loc[(geo["ZOCALO"] == "SI") & ~sin, "DEPARTAMENTO_DETALLE"].astype(str))
    if sin.any():                                    # lotes en el mar: región costera más cercana
        d, n = distancia_km(geo[sin], deps, "DEPARTAMENTO")
        geo.loc[sin, "DEPARTAMENTO"] = n.values
        geo.loc[sin, "DEPARTAMENTO_DETALLE"] = "ZÓCALO frente a " + n.astype(str).values
if dists is not None:                                 # distritos y provincias con % del área del lote («DEP|PROV|DIST 85.0%; …»)
    geo = geo.join(reparto_por_poligonos(geo, dists, "CLAVE", "DISTRITO")[["DISTRITO_DETALLE"]])
    geo = geo.join(reparto_por_poligonos(geo, provs, "CLAVE", "PROVINCIA")[["PROVINCIA_DETALLE"]])
if cuencas is not None and cuenca_n:
    geo = geo.join(reparto_por_poligonos(geo, cuencas, cuenca_n, "CUENCA")[["CUENCA", "CUENCA_PCT"]])
else:
    geo["CUENCA"] = geo["CUENCA_ATRIBUTO"]
if onp is not None:
    geo["DIST_OLEODUCTO_KM"], _ = distancia_km(geo, onp)
geo["DIST_IQUITOS_KM"] = (geo.to_crs(CRS_DIST).distance(gpd.GeoSeries([IQUITOS], crs=4326).to_crs(CRS_DIST).iloc[0]) / 1000).round(1)
# distancias a ríos desde el punto interior del lote (desde el borde casi todos los lotes amazónicos darían 0)
pts = gpd.GeoDataFrame(geometry=geo.geometry.representative_point(), crs=4326, index=geo.index)
if rios_q is not None:
    for u in UMBRALES:
        geo[f"DIST_RIO_{u}_KM"], _ = distancia_km(pts, rios_q[rios_q["DIS_AV_CMS"] >= u])
    geo["DIST_RIO_GRANDE_KM"] = geo[f"DIST_RIO_{CAUDAL}_KM"]
if rios_n is not None:
    cand = rios_n
    if rios_q is not None:                               # nombre del río grande más cercano (no de una quebrada)
        grandes = rios_q[rios_q["DIS_AV_CMS"] >= CAUDAL].to_crs(CRS_DIST).buffer(1500)
        zona = gpd.GeoDataFrame(geometry=[grandes.union_all() if hasattr(grandes, "union_all") else grandes.unary_union], crs=CRS_DIST)
        cand = gpd.sjoin(rios_n.to_crs(CRS_DIST), zona, predicate="intersects").drop(columns="index_right").to_crs(4326)
        cand = cand if len(cand) else rios_n
    _, geo["RIO_CERCANO"] = distancia_km(pts, cand, "RIO")
if anp is not None and anp_n:
    a = reparto_por_poligonos(geo, anp, anp_n, "ANP")
    geo["ANP_PCT"] = a.get("ANP_PCT_TOTAL"); geo["ANP_DETALLE"] = a.get("ANP_DETALLE")
    geo["ANP_PCT"] = geo["ANP_PCT"].fillna(0); geo["ANP_DETALLE"] = geo["ANP_DETALLE"].fillna("")
if ccnn is not None and ccnn_n:
    geo = geo.join(comunidades(geo, ccnn, ccnn_n, 5))

# ── 4. Producción: indicadores por lote y por campo ─────────────────────────
def enteros(df):
    """Columnas enteras sin decimales en el CSV (1994 y no 1994.0), aunque tengan vacíos."""
    for c in ["BPD_ACTUAL", "PICO_AÑO", "PICO_BPD", "MESES_SIN_PRODUCCION", "PARALIZACION_MAX_MESES",
              "CCNN_DENTRO_N", "CCNN_5KM_N", "DEPARTAMENTO_N"]:
        if c in df:
            df[c] = pd.to_numeric(df[c], errors="coerce").round().astype("Int64")
    return df

COLS_IND = ["ESTADO_PRODUCCION", "PRODUCTO_PRINCIPAL", "PRIMERA_PRODUCCION", "ULTIMA_PRODUCCION", "BPD_ACTUAL",
            "PICO_AÑO", "PICO_BPD", "DECLINACION_ANUAL_5A_PCT", "DECLINACION_PERIODO", "MESES_SIN_PRODUCCION", "PARALIZACION_MAX_MESES",
            "PARALIZACION_MAX_DESDE", "OPERADORES"]
ind_lote, ind_campo, CAMPO_DE = {}, {}, {}
if prod is not None:
    # campo = lotes que comparten polígono (1-AB/192; 31-B, 31-D y 31-B/D; Z-2B y Z-69…)
    hist = (prod.drop_duplicates("LOTE").set_index("LOTE")["LOTE_HISTORICO"].astype(str).to_dict()
            if "LOTE_HISTORICO" in prod else {})
    for lote, grupo in campos(geo).items():
        CAMPO_DE[lote] = " + ".join(sorted({hist.get(l, l) for l in grupo if l in hist} or set(grupo)))
    for lote in prod["LOTE"].astype(str).unique():
        CAMPO_DE.setdefault(lote, hist.get(lote, lote))
    prod["LOTE"] = prod["LOTE"].astype(str)
    prod["CAMPO"] = prod["LOTE"].map(CAMPO_DE)
    ind_lote = indicadores_produccion(prod, "LOTE", grupo="CAMPO")
    ind_campo = indicadores_produccion(prod, "CAMPO")
    # un contrato que terminó y siguió con otro en el mismo campo (1-AB → 192) no está "paralizado": fue sucedido
    def _t(mmaaaa):
        mm, aa = str(mmaaaa).lstrip("≤").split("/"); return int(aa) * 12 + int(mm)
    for l, d in ind_lote.items():
        if d.get("ESTADO_PRODUCCION") != "PARALIZADO":
            continue
        despues = [o for o, e in ind_lote.items() if o != l and CAMPO_DE.get(o) == CAMPO_DE.get(l)
                   and e.get("ULTIMA_PRODUCCION") and _t(e["ULTIMA_PRODUCCION"]) > _t(d["ULTIMA_PRODUCCION"])]
        if not despues and l in SUCESION_MANUAL:
            despues = [SUCESION_MANUAL[l]]
        if despues:
            d["ESTADO_PRODUCCION"] = f"SUCEDIDO POR {', '.join(sorted(despues))}"
            d.update({"PARALIZACION_MAX_MESES": None, "PARALIZACION_MAX_DESDE": None, "MESES_SIN_PRODUCCION": None})
    geo["CAMPO"] = geo["LOTE"].map(CAMPO_DE)
    geo["PETROLEO_ACUM_MMBBL"] = geo["LOTE"].map(lambda l: ind_lote.get(l, {}).get("PETROLEO_ACUM_MMBBL", 0.0))
    geo["PETROLEO_ACUM_CAMPO_MMBBL"] = geo["CAMPO"].map(lambda k: ind_campo.get(k, {}).get("PETROLEO_ACUM_MMBBL", 0.0))
    if gas is not None and len(gas):
        _h = gas["LOTE_HISTORICO"].astype(str) if "LOTE_HISTORICO" in gas else gas["LOTE"]
        gas["CAMPO"] = gas["LOTE"].map(CAMPO_DE).fillna(_h)
        _acum = gas.groupby("CAMPO")["MPC"].sum() / 1e6               # miles de millones de pies cúbicos (BCF)
        geo["GAS_ACUM_CAMPO_BCF"] = geo["CAMPO"].map(_acum).round(2)
    for c in COLS_IND:                                  # indicadores del campo al que pertenece el polígono
        geo[c] = geo["CAMPO"].map(lambda k, c=c: ind_campo.get(k, {}).get(c))
    geo["ESTADO_PRODUCCION"] = geo["ESTADO_PRODUCCION"].fillna("SIN PRODUCCIÓN REGISTRADA")

    tind = pd.DataFrame([{"LOTE": l, "CAMPO": CAMPO_DE.get(l, l),
                          "DEPARTAMENTO": dict(zip(geo["LOTE"], geo.get("DEPARTAMENTO", pd.Series(dtype=str)))).get(l, ""),
                          "TIENE_POLIGONO": "SI" if l in set(geo["LOTE"]) else "NO",
                          **{k: v for k, v in d.items() if not k.startswith("_")}} for l, d in ind_lote.items()])
    if "DEPARTAMENTO" in prod:                          # lotes sin polígono: departamento de la base de producción
        dep_prod = prod[prod["DEPARTAMENTO"].fillna("") != ""].groupby("LOTE")["DEPARTAMENTO"].agg(lambda s: s.mode().iloc[0])
        tind["DEPARTAMENTO"] = tind["DEPARTAMENTO"].replace("", None).fillna(tind["LOTE"].map(dep_prod)).fillna("")
    tind = enteros(tind).sort_values(["DEPARTAMENTO", "PETROLEO_ACUM_MMBBL"], ascending=[True, False])
    tind.insert(0, "FECHA_EXTRACCION", date.today().strftime("%d/%m/%Y"))
    p = os.path.join(ruta, "PP_LOTES_INDICADORES.csv"); tind.to_csv(p, index=False, encoding="utf-8-sig")
    subir(p, DRIVE_FOLDER_ID, datos_drive)

    falta = sorted(set(prod["LOTE"]) - set(geo["LOTE"]))
    print(f"\n🔗 Producción leída de {prod_csv[-1]} · indicadores de {len(ind_lote)} lotes y {len(ind_campo)} campos "
          f"→ DATA/PP_LOTES_INDICADORES.csv")
    print(f"   Lotes con producción y sin polígono ({len(falta)}): {', '.join(falta) if falta else 'ninguno'}")
    usados = {p for e in extra for p in e["GEOMETRIA_DE"].split("+")}
    sin_prod = sorted(en_mapa - set(prod["LOTE"]) - usados)
    print(f"   Lotes del mapa sin producción ({len(sin_prod)}): {', '.join(sin_prod)}")
    log.append({"CAPA": "LOTES SIN POLIGONO", "URL": "", "FUENTE": prod_csv[-1], "ENTIDADES": len(falta),
                "ESTADO": ", ".join(falta)[:500]})
else:
    geo["ESTADO_PRODUCCION"] = "SIN DATOS DE PRODUCCIÓN"
    print("\nℹ️ No encontré PP_produccion_lote_mes_*.csv en DATA: corre antes el notebook de producción para el cruce.")

# ── 5. Guardar ───────────────────────────────────────────────────────────────
cols = [c for c in ["LOTE", "GEOMETRIA_DE", "LOTE_MAPA", "OPERADOR_MAPA", "ESTADO_MAPA", "TIPO_CONTRATO", "F_SUSCRIPCION",
                    "CAPA", "FUENTE_POLIGONO", "DEPARTAMENTO",
                    "DEPARTAMENTO_PCT", "DEPARTAMENTO_DETALLE", "DEPARTAMENTO_N", "PROVINCIA_DETALLE", "DISTRITO_DETALLE",
                    "ZOCALO", "CUENCA", "CUENCA_PCT",
                    "AREA_KM2", "LAT", "LON", "DIST_OLEODUCTO_KM", "DIST_IQUITOS_KM", "DIST_RIO_GRANDE_KM"] + [f"DIST_RIO_{u}_KM" for u in UMBRALES] + ["RIO_CERCANO",
                    "ANP_PCT", "ANP_DETALLE", "CCNN_DENTRO_N", "CCNN_5KM_N", "CCNN_DENTRO", "CAMPO", "PETROLEO_ACUM_MMBBL",
                    "PETROLEO_ACUM_CAMPO_MMBBL", "GAS_ACUM_CAMPO_BCF"] + COLS_IND
        if c in geo.columns]
tabla = enteros(pd.DataFrame(geo[cols])).sort_values("LOTE")
tabla.insert(0, "FECHA_EXTRACCION", date.today().strftime("%d/%m/%Y"))
p = os.path.join(ruta, "PP_LOTES_GEO.csv"); tabla.to_csv(p, index=False, encoding="utf-8-sig")
subir(p, DRIVE_FOLDER_ID, datos_drive)
guardar_geojson(geo[cols + ["geometry"]], "PP_lotes_indicadores.geojson")
pc = os.path.join(ruta, "CONTROL PP mapa descarga.csv")
pd.DataFrame(log).assign(FECHA=date.today().strftime("%d/%m/%Y")).to_csv(pc, index=False, encoding="utf-8-sig")
subir(pc, DRIVE_FOLDER_CONTROL_ID, control_drive)
print(f"\n☁️  DATA/PP_LOTES_GEO.csv ({len(tabla)} lotes) · DATA/MAPA/PP_lotes_indicadores.geojson · CONTROL actualizado")

if "DEPARTAMENTO" in tabla:
    print("\n📍 Lotes por departamento:")
    display(tabla.groupby("DEPARTAMENTO").agg(LOTES=("LOTE", "count"), AREA_KM2=("AREA_KM2", "sum")).sort_values("LOTES", ascending=False))
    lor = tabla[tabla["DEPARTAMENTO"] == "LORETO"]
    if len(lor):
        print("\n🛢️ Loreto (indicadores del campo):")
        ver = [c for c in ["LOTE", "CAMPO", "ESTADO_PRODUCCION", "BPD_ACTUAL", "PICO_AÑO", "PICO_BPD", "DECLINACION_ANUAL_5A_PCT",
                           "PARALIZACION_MAX_MESES", "PETROLEO_ACUM_CAMPO_MMBBL", "DIST_OLEODUCTO_KM", "DIST_RIO_GRANDE_KM",
                           "ANP_PCT", "CCNN_DENTRO_N"] if c in lor]
        display(lor[ver].sort_values("PETROLEO_ACUM_CAMPO_MMBBL" if "PETROLEO_ACUM_CAMPO_MMBBL" in lor else "LOTE", ascending=False))

In [ ]:
# ═══ BLOQUE 4 · MAPA INTERACTIVO ═══
import folium, math, json, re, html as _h, shapely, unicodedata
from folium.plugins import MarkerCluster
m = folium.Map(location=[-4.8, -75.5], zoom_start=6, tiles=None, control_scale=True)
# Fondos de Esri: los mosaicos de OpenStreetMap bloquean las vistas embebidas de Colab ("403 Access blocked")
_ESRI = "https://server.arcgisonline.com/ArcGIS/rest/services/{}/MapServer/tile/{{z}}/{{y}}/{{x}}"
folium.TileLayer(_ESRI.format("World_Topo_Map"), attr="Esri, HERE, Garmin, USGS, NGA", name="Topográfico (Esri)").add_to(m)
folium.TileLayer(_ESRI.format("World_Imagery"), attr="Esri, Maxar, Earthstar Geographics", name="Satélite (Esri)", show=False).add_to(m)

def simpl(g, tol=0.002):
    # Simplifica y redondea a 5 decimales (~1 m): el mapa queda liviano y abre en cualquier computadora
    g = g.copy()
    g["geometry"] = g.geometry.simplify(tol, preserve_topology=True).map(lambda x: shapely.set_precision(x, 1e-5) if x is not None else x)
    return g[g.geometry.notna() & ~g.geometry.is_empty]

ANIO_INI, ANIO_FIN = (int(prod["AÑO"].min()), int(prod["AÑO"].max())) if prod is not None else (None, None)
COLOR_ESTADO = {"EN PRODUCCIÓN": "#2e7d32", "PARALIZADO": "#ef6c00", "SIN PRODUCCIÓN REGISTRADA": "#9e9e9e",
                "SIN DATOS DE PRODUCCIÓN": "#9e9e9e"}
ETIQUETAS = [("CAMPO", "Campo", ""), ("OPERADOR_MAPA", "Operador (contrato vigente)", ""), ("ESTADO_MAPA", "Fase del contrato", ""),
             ("TIPO_CONTRATO", "Tipo de contrato", ""), ("F_SUSCRIPCION", "Suscripción", ""),
             ("DEPARTAMENTO_DETALLE", "Departamento", ""), ("PROVINCIA_DETALLE", "Provincia", ""),
             ("DISTRITO_DETALLE", "Distrito", ""), ("AREA_KM2", "Área", "km²"),
             ("PRIMERA_PRODUCCION", "Primera producción", ""),
             ("ULTIMA_PRODUCCION", "Última producción", ""), ("PICO_AÑO", "Año pico", ""),
             ("PICO_BPD", "Producción en el pico", "bpd"), ("DECLINACION_ANUAL_5A_PCT", "Variación anual (5 años)", "%"),
             ("DECLINACION_PERIODO", "… periodo", ""),
             ("PARALIZACION_MAX_MESES", "Paralización más larga", "meses"), ("PARALIZACION_MAX_DESDE", "… desde", ""),
             ("PETROLEO_ACUM_CAMPO_MMBBL", "Petróleo acumulado del campo", "millones de bbl"),
             ("GAS_ACUM_CAMPO_BCF", "Gas natural acumulado del campo", "miles de millones de pies³"), ("OPERADORES", "Operadores", ""),
             ("DIST_OLEODUCTO_KM", "Distancia al oleoducto", "km"),
             ("DIST_RIO_1000_KM", "Distancia a un afluente (≥ 1.000 m³/s)", "km"),
             ("DIST_RIO_GRANDE_KM", "Distancia a un río troncal (≥ 5.000 m³/s)", "km"),
             ("RIO_CERCANO", "Río grande cercano", ""), ("DIST_IQUITOS_KM", "Distancia a Iquitos", "km"),
             ("ANP_DETALLE", "Áreas protegidas", ""), ("CCNN_DENTRO_N", "Comunidades nativas dentro", ""),
             ("CCNN_5KM_N", "Comunidades a menos de 5 km", ""),
             ("_SOCIOS", "Empresas del contrato", ""), ("_VIGENCIA", "Vigencia del contrato", "")]

ENTEROS = {"BPD_ACTUAL", "PICO_BPD", "PARALIZACION_MAX_MESES", "CCNN_DENTRO_N", "CCNN_5KM_N", "AREA_KM2", "DIST_OLEODUCTO_KM",
           "DIST_RIO_GRANDE_KM", "DIST_RIO_1000_KM", "DIST_IQUITOS_KM"}

def _num(v, dec):                                   # formato peruano: 12.299,5
    return f"{v:,.{dec}f}".replace(",", "§").replace(".", ",").replace("§", ".")

def fmt(c, v, u):
    if v is None or (isinstance(v, float) and math.isnan(v)) or str(v).strip() in ("", "nan", "None", "<NA>"):
        return None
    if c == "PICO_AÑO":
        v = str(int(float(v)))
    elif isinstance(v, (int, float)) and not isinstance(v, bool):
        v = _num(float(v), 0 if c in ENTEROS else (2 if c in ("PETROLEO_ACUM_CAMPO_MMBBL", "GAS_ACUM_CAMPO_BCF") else 1))
    return f"{_h.escape(str(v))} {u}".strip()

# ── Series mensuales para el panel de producción (van dentro del HTML: el mapa funciona sin conexión a Drive) ──
PANEL_HTML = r'''<script>
// «Guardar como imagen»: arma un PNG con título, leyenda, el gráfico (SVG) y la nota de fuente
window.ppImagen=window.ppImagen||function(svg,o){
  if(!svg)return;o=o||{};
  const vb=svg.viewBox&&svg.viewBox.baseVal,W=(vb&&vb.width)||svg.clientWidth||700,H=(vb&&vb.height)||svg.clientHeight||260;
  const c=svg.cloneNode(true),org=svg.querySelectorAll("*"),cl=c.querySelectorAll("*");
  org.forEach((e,i)=>{const cs=getComputedStyle(e),d=cl[i];                       // estilos de la hoja → atributos
    ["fill","stroke","stroke-width","stroke-dasharray","opacity","fill-opacity","font-size","font-weight"].forEach(p=>{
      const v=cs.getPropertyValue(p);if(v&&!d.getAttribute(p))d.setAttribute(p,v);});});
  c.setAttribute("xmlns","http://www.w3.org/2000/svg");c.setAttribute("width",W);c.setAttribute("height",H);
  c.setAttribute("font-family","Barlow, Arial, sans-serif");
  const img=new Image();
  img.onload=()=>{
    const k=2,ancho=Math.max(W,640),pad=24,cv=document.createElement("canvas"),ctx=cv.getContext("2d");
    const F=(w,s)=>`${w} ${s}px Barlow, Arial, sans-serif`;
    const lineas=(t,f,max)=>{ctx.font=f;const out=[];let l="";(t||"").split(/\s+/).forEach(p=>{const q=l?l+" "+p:p;
      if(ctx.measureText(q).width>max&&l){out.push(l);l=p;}else l=q;});if(l)out.push(l);return out;};
    const tit=lineas(o.titulo,F(700,17),ancho),sub=lineas(o.sub,F(400,12.5),ancho),nota=lineas(o.nota,F(400,11),ancho);
    const ley=o.ley||[];ctx.font=F(400,12.5);const filasLey=[];let fl=[],x=0;
    ley.forEach(([col,nom])=>{const w=16+ctx.measureText(nom).width+14;if(x+w>ancho&&fl.length){filasLey.push(fl);fl=[];x=0;}fl.push([col,nom]);x+=w;});
    if(fl.length)filasLey.push(fl);
    const h=pad+tit.length*22+sub.length*17+(sub.length?6:0)+filasLey.length*19+(filasLey.length?6:0)+H+10+nota.length*15+pad;
    cv.width=(ancho+2*pad)*k;cv.height=h*k;ctx.scale(k,k);ctx.fillStyle="#fefefe";ctx.fillRect(0,0,ancho+2*pad,h);
    let y=pad;ctx.textBaseline="top";
    ctx.fillStyle="#181818";ctx.font=F(700,17);tit.forEach(t=>{ctx.fillText(t,pad,y);y+=22;});
    ctx.fillStyle="#555";ctx.font=F(400,12.5);sub.forEach(t=>{ctx.fillText(t,pad,y);y+=17;});if(sub.length)y+=6;
    filasLey.forEach(f=>{let xx=pad;f.forEach(([col,nom])=>{ctx.fillStyle=col;ctx.fillRect(xx,y+3,10,10);ctx.fillStyle="#444";
      ctx.font=F(400,12.5);ctx.fillText(nom,xx+15,y+1);xx+=16+ctx.measureText(nom).width+14;});y+=19;});if(filasLey.length)y+=6;
    ctx.drawImage(img,pad,y,W,H);y+=H+10;
    ctx.fillStyle="#666";ctx.font=F(400,11);nota.forEach(t=>{ctx.fillText(t,pad,y);y+=15;});
    cv.toBlob(b=>{const a=document.createElement("a");a.href=URL.createObjectURL(b);
      a.download=(o.archivo||"grafico").replace(/[^\wÁÉÍÓÚÑáéíóúñ-]+/g,"_").slice(0,90)+".png";
      document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(a.href),1500);},"image/png");};
  img.src="data:image/svg+xml;charset=utf-8,"+encodeURIComponent(new XMLSerializer().serializeToString(c));
};
// leyenda de un contenedor: [[color, nombre]] de sus <span><i style="background"></i>nombre</span>
window.ppLeyendaDe=window.ppLeyendaDe||function(el){return el?[...el.querySelectorAll("span")].filter(s=>s.querySelector("i"))
  .map(s=>[getComputedStyle(s.querySelector("i")).backgroundColor,s.textContent.trim()]):[];};
</script>
<style>
#pp-panel,#pp-abrir,#pp-ley{font:12px/1.4 system-ui,-apple-system,"Segoe UI",Roboto,sans-serif;color:#181818}
.pp-caja{background:#fcfcfb;border-radius:8px;box-shadow:0 2px 10px rgba(0,0,0,.28)}
.pp-btn{border:1px solid #c9c8c2;background:#f3f2ee;border-radius:5px;min-width:26px;height:24px;padding:0 7px;cursor:pointer;
  font:600 13px/20px system-ui,sans-serif;color:#181818}
.pp-btn:hover{background:#e7e6e0}.pp-btn:focus-visible,.pp-sel:focus-visible{outline:2px solid #2a78d6;outline-offset:1px}
#pp-panel{position:fixed;left:50%;top:50%;transform:translate(-50%,-50%);z-index:10001;width:min(900px,calc(100vw - 24px));
  display:none;flex-direction:column;max-height:calc(100vh - 32px);box-shadow:0 8px 32px rgba(0,0,0,.35)}
#pp-fondo{position:fixed;inset:0;z-index:10000;background:rgba(20,20,20,.38);display:none;cursor:pointer}
#pp-cab{display:flex;align-items:center;gap:8px;padding:8px 10px;border-bottom:1px solid #e6e5e0;cursor:default}
#pp-cab b{font-size:14px;flex:1;min-width:0;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
#pp-cuerpo{padding:8px 10px 10px;overflow:auto}
#pp-filtros{display:flex;flex-wrap:wrap;gap:6px 10px;align-items:flex-end;margin-bottom:8px}
#pp-filtros label{display:flex;flex-direction:column;font-size:11px;color:#52514e;gap:2px}
.pp-sel{font:12px system-ui,sans-serif;padding:3px 4px;border:1px solid #c9c8c2;border-radius:5px;background:#fff;color:#181818;max-width:210px}
#pp-resumen{border-collapse:collapse;margin:2px 0 8px;font-size:12px;width:100%}
#pp-resumen th{font-weight:normal;color:#52514e;text-align:right;padding:2px 8px;border-bottom:1px solid #e6e5e0}
#pp-resumen td{text-align:right;padding:3px 8px;font-variant-numeric:tabular-nums}
#pp-resumen td:first-child,#pp-resumen th:first-child{text-align:left}
#pp-resumen td b{font-weight:600}
#pp-titulo{font-weight:600;margin:2px 0 0}#pp-nota{color:#52514e;font-size:11px;min-height:15px}
#pp-graf{position:relative;width:100%;height:270px;touch-action:none}
#pp-graf svg{display:block;width:100%;height:100%}
#pp-tip{position:absolute;pointer-events:none;background:#fff;border:1px solid #d8d7d1;border-radius:6px;padding:6px 8px;
  box-shadow:0 2px 6px rgba(0,0,0,.18);font-size:12px;display:none;white-space:nowrap;z-index:2}
#pp-tip .v{font-weight:700;font-variant-numeric:tabular-nums}#pp-tip .s{color:#52514e}
#pp-tip i{display:inline-block;width:12px;height:0;border-top:2px solid;vertical-align:middle;margin-right:5px}
#pp-leyenda{display:flex;gap:14px;font-size:12px;color:#52514e;margin-top:2px}
#pp-leyenda span i{display:inline-block;width:10px;height:10px;border-radius:2px;margin-right:5px;vertical-align:-1px}
#pp-tabla-c{max-height:220px;overflow:auto;margin-top:8px;border-top:1px solid #e6e5e0;display:none}
#pp-tabla{border-collapse:collapse;width:100%;font-size:12px}
#pp-tabla th{position:sticky;top:0;background:#f3f2ee;font-weight:600;text-align:right;padding:3px 8px}
#pp-tabla td{text-align:right;padding:2px 8px;border-bottom:1px solid #f0efec;font-variant-numeric:tabular-nums}
#pp-tabla td:first-child,#pp-tabla th:first-child,#pp-tabla td:last-child,#pp-tabla th:last-child{text-align:left}
#pp-tabla td:last-child{white-space:normal;color:#52514e}
#pp-abrir{position:fixed;left:10px;bottom:24px;z-index:9999;padding:6px 10px;height:auto;font-size:12px}
/* el botón va dentro del selector de capas, debajo de «Productos» */
#pp-abrir.en-capas{position:static;display:block;width:100%;margin:2px 0 0;box-shadow:none;text-align:left;white-space:normal}
.pp-abrir-caja{border-top:1px solid #ddd;margin-top:6px;padding-top:6px}
#pp-ley{position:fixed;bottom:28px;right:10px;z-index:9999;padding:6px 9px;max-width:240px;font-size:11px;line-height:1.5}
#pp-ley-cab{display:flex;align-items:center;justify-content:space-between;gap:12px}
#pp-tabs{display:none;gap:4px;padding:8px 10px 0;flex-wrap:wrap;border-bottom:1px solid #d6d4cc}
#pp-tabs button{border:1px solid #d6d4cc;background:#f7f6f2;border-radius:6px 6px 0 0;padding:5px 10px;cursor:pointer;
  font:600 12px sans-serif;color:#333;margin-bottom:-1px}
#pp-tabs button[aria-selected=true]{background:#fcfcfb;border-bottom-color:#fcfcfb;color:#000}
#pp-tabs button:disabled{opacity:.4;cursor:default}
.pp-mini #pp-cuerpo,.pp-mini #pp-tabs{display:none!important}
#pp-panel.pp-mini{top:auto;bottom:24px;transform:translateX(-50%);width:min(520px,calc(100vw - 24px))}
@media (max-width:600px){#pp-graf{height:220px}.pp-sel{max-width:150px}}
</style>
<button id="pp-abrir" class="pp-btn pp-caja" type="button" onclick="ppAbrir()">📊 Producción fiscalizada anual y mensual</button>
<div id="pp-fondo" onclick="ppCerrar()" title="Clic fuera del panel para cerrarlo"></div>
<div id="pp-panel" class="pp-caja" aria-modal="true" role="dialog" aria-label="Producción fiscalizada anual y mensual">
  <div id="pp-cab"><b id="pp-cab-t">Producción</b>
    <button class="pp-btn" type="button" id="pp-min" title="Minimizar" aria-label="Minimizar panel" onclick="ppMin()">–</button>
    <button class="pp-btn" type="button" title="Cerrar" aria-label="Cerrar panel" onclick="ppCerrar()">×</button></div>
  <div id="pp-tabs" role="tablist"></div>
  <div id="pp-cuerpo">
    <div id="pp-filtros">
      <label>Serie<select class="pp-sel" id="pp-a"></select></label>
      <label>Comparar con<select class="pp-sel" id="pp-b"></select></label>
      <label>Producto<select class="pp-sel" id="pp-prod"><option value="T">Líquidos totales</option>
        <option value="P">Petróleo</option><option value="L">LGN y condensado</option>
        <option value="G">Gas natural</option></select></label>
      <label>Vista<select class="pp-sel" id="pp-vista"><option value="A">Anual</option><option value="M">Mensual</option></select></label>
      <label>Desde<select class="pp-sel" id="pp-desde"></select></label>
      <label>Hasta<select class="pp-sel" id="pp-hasta"></select></label>
      <label>Unidad<select class="pp-sel" id="pp-u"><option value="bbl">Barriles</option>
        <option value="bpd">Barriles por día</option></select></label>
      <button class="pp-btn" type="button" id="pp-volver" style="display:none" onclick="ppVolver()">↩ Serie anual</button>
      <button class="pp-btn" type="button" id="pp-btab" onclick="ppTabla()">Tabla</button>
      <button class="pp-btn" type="button" onclick="ppCSV()">CSV</button>
      <button class="pp-btn" type="button" onclick="ppImagenProd()" title="Guardar el gráfico como imagen (PNG)">🖼 Imagen</button>
    </div>
    <table id="pp-resumen"></table>
    <div id="pp-titulo"></div><div id="pp-nota"></div>
    <div id="pp-graf"><div id="pp-tip"></div></div>
    <div id="pp-leyenda"></div>
    <div id="pp-tabla-c"><table id="pp-tabla"></table></div>
  </div>
</div>
<script>
(function(){
const D=__DATA__;
const MES=["ene","feb","mar","abr","may","jun","jul","ago","set","oct","nov","dic"];
const COL=["#111149","#72d2d5"], GRID="#e6e5e0", INK2="#52514e", NS="http://www.w3.org/2000/svg";
const PAL=["#111149","#72d2d5","#ccd3df","#ffc41d","#2a9da1","#5a5a9e","#8c96ad","#181818","#b9e9eb","#dd6140"];
const S={}; D.s.forEach(x=>S[x.id]=x);
// color de cada empresa operadora: mismo orden (primer año con producción) en el campo y en sus lotes, y en la pestaña «Empresas»
function ordenOps(x){const b=(x.cp&&S["C:"+x.cp]&&S["C:"+x.cp].ob)?S["C:"+x.cp]:x;if(!b.ob)return [];
  const ini=o=>{const r=b.ob[o];let m=1e9;for(const k in r){const i=r[k].findIndex(v=>v>0);if(i>=0&&i<m)m=i;}return m;};
  return Object.keys(b.ob).sort((a,c)=>ini(a)-ini(c)||a.localeCompare(c));}
window.ppTieneSerie=id=>!!S[id];
window.ppProd={D:D,S:S};                    // «Comparar lotes» lee las mismas series
window.ppColOp=function(id,op){const x=S[id];if(!x)return null;const o=ordenOps(x),i=o.indexOf(op);return i<0?null:PAL[i%PAL.length];};
const A0=D.ini, A1s=D.ini+Math.floor((D.n-1)/12);          // último año de las estadísticas mensuales
const A1=Math.max(A1s,D.dia?new Date(Date.UTC(D.dia.ini[0],D.dia.ini[1]-1,D.dia.ini[2])+(D.dia.n-1)*864e5).getUTCFullYear():0);
const st={a:D.def,b:"",prod:"T",vista:"A",desde:A0,hasta:A1,u:"bbl",tabla:false,prev:null,pila:[],dmes:null};
const DIA=D.dia||null, DIA0=DIA?Date.UTC(DIA.ini[0],DIA.ini[1]-1,DIA.ini[2]):0;
const DA0=DIA?DIA.ini[0]:0, DA1=DIA?new Date(DIA0+(DIA.n-1)*864e5).getUTCFullYear():0;
const pad=v=>(v<10?"0":"")+v;
function mesDiario(x,a,m){const dd=x.dd||{};let b=0,n=0,ult=0;
  for(let i=Math.max(0,Math.round((Date.UTC(a,m,1)-DIA0)/864e5));i<DIA.n;i++){const f=new Date(DIA0+i*864e5);
    if(f.getUTCFullYear()!==a||f.getUTCMonth()!==m)break;
    const p=(dd.p||[])[i]||0,l=(dd.l||[])[i]||0,g=(dd.g||[])[i]||0;
    b+=esGas()?g:st.prod==="P"?p:st.prod==="L"?l:p+l;n++;ult=f.getUTCDate();}
  return n?{b:b,n:n,ult:ult}:null;}
function hayDias(a,m){if(!DIA)return false;const i0=Math.round((Date.UTC(a,m,1)-DIA0)/864e5),i1=Math.round((Date.UTC(a,m+1,1)-DIA0)/864e5);return i1>0&&i0<DIA.n;}
const $=id=>document.getElementById(id);
const PROD={T:"líquidos totales",P:"petróleo",L:"LGN y condensado",G:"gas natural"};
const esGas=()=>st.prod==="G";

function num(v,d){d=d||0;if(v==null||!isFinite(v))return "–";const s=Math.abs(v).toFixed(d).split(".");
  return (v<0?"−":"")+s[0].replace(/\B(?=(\d{3})+(?!\d))/g,".")+(s[1]?","+s[1]:"");}
function corto(v){const a=Math.abs(v);if(a>=1e6)return num(v/1e6,a>=1e7?0:1)+" M";if(a>=1e4)return num(v/1e3,0)+" mil";return num(v,0);}
function vol(b){if(esGas())return b>=1e6?num(b/1e6,1)+" miles de millones de pies³":num(b/1e3,0)+" millones de pies³";   // gas: b en MPC
  return Math.abs(b)>=1e6?num(b/1e6,2)+" millones de bbl":num(b,0)+" bbl";}
function dias(t){return new Date(A0+Math.floor(t/12),t%12+1,0).getDate();}
function mensual(x){if(esGas())return x.gs||x.p.map(()=>0);
  return x.p.map((v,i)=>st.prod==="P"?v:st.prod==="L"?x.l[i]:v+x.l[i]);}
function datos(x){
  if(st.vista==="D"){
    const dd=x.dd||{},out=[],g=esGas()?1/1000:1;
    for(let i=0;DIA&&i<DIA.n;i++){const f=new Date(DIA0+i*864e5),a=f.getUTCFullYear(),mm=f.getUTCMonth(),di=f.getUTCDate();
      if(a<st.desde||a>st.hasta||(st.dmes&&(st.dmes.a!==a||st.dmes.m!==mm)))continue;
      const p=(dd.p||[])[i]||0,l=(dd.l||[])[i]||0,gg=(dd.g||[])[i]||0;
      const v=esGas()?gg:st.prod==="P"?p:st.prod==="L"?l:p+l;
      out.push({k:i,lab:pad(di)+"/"+pad(mm+1)+"/"+a,eje:String(di),dia:di,mes:mm,anio:a,bbl:v,d:1,ops:(x.o||{})[a]||"",v:v*g});}
    return out;
  }
  const m=mensual(x),o=x.o||{},out=[];
  for(let a=st.desde;a<=st.hasta;a++){
    if(st.vista==="A"){let b=0,d=0,n=0;for(let k=0;k<12;k++){const t=(a-A0)*12+k;if(t>=D.n)break;b+=m[t];d+=dias(t);n++;}
      if(n){const r={k:a,lab:a+(n<12?"* (solo ene–"+MES[n-1]+", estadística mensual)":""),eje:a+(n<12?"*":""),bbl:b,d:d,ops:o[a]||"",anio:a,parcial:n<12};
        if(x.ob){const ks=esGas()?["g"]:st.prod==="P"?["p"]:st.prod==="L"?["l"]:["p","l"];
          r.por=Object.keys(x.ob).map(op=>[op,ks.reduce((s_,k)=>s_+((x.ob[op][k]||[])[a-A0]||0),0)]).filter(z=>z[1]>0);}
        out.push(r);}}
    else for(let k=0;k<12;k++){const t=(a-A0)*12+k;
      if(t<D.n)out.push({k:t,lab:MES[k]+" "+a,eje:MES[k],mes:k,bbl:m[t],d:dias(t),ops:o[a]||"",anio:a});
      else if(hayDias(a,k)){const r=mesDiario(x,a,k);   // meses después de la última estadística: suma de los reportes diarios
        if(r){const parcial=r.n<new Date(Date.UTC(a,k+1,0)).getUTCDate();
          out.push({k:t,lab:MES[k]+" "+a+(parcial?"* (reporte diario en curso, días 1–"+r.ult+": mes incompleto)":" (suma de los reportes diarios)"),
                    eje:MES[k]+(parcial?"*":""),mes:k,bbl:r.b,d:r.n,ops:o[a]||"",anio:a,diario:true,parcial:parcial,ult:r.ult});}}}
  }
  const f=esGas()?1/1000:1;                         // gas: MPC → MMPC
  out.forEach(r=>{r.v=(st.u==="bpd"?r.bbl/r.d:r.bbl)*f;if(r.por)r.porv=r.por.map(([o,b])=>[o,(st.u==="bpd"?b/r.d:b)*f]);});return out;
}
function uni(){if(st.vista==="D")return esGas()?"millones de pies³/día":"bpd";                      // un día: volumen = promedio diario
  return esGas()?(st.u==="bpd"?"millones de pies³/día":"millones de pies³"):(st.u==="bpd"?"bpd":"bbl");}
function el(tag,at,par){const e=document.createElementNS(NS,tag);for(const k in at)e.setAttribute(k,at[k]);if(par)par.appendChild(e);return e;}
function paso(max){if(max<=0)return 1;const r=max/4,e=Math.pow(10,Math.floor(Math.log10(r))),f=r/e;return (f<=1?1:f<=2?2:f<=2.5?2.5:f<=5?5:10)*e;}
function barra(x,y,w,h,r){r=Math.min(r,w/2,h);if(h<=0)return "";
  return `M${x},${y+h}V${y+r}Q${x},${y} ${x+r},${y}H${x+w-r}Q${x+w},${y} ${x+w},${y+r}V${y+h}Z`;}

function opciones(sel,vacia){
  sel.textContent="";if(vacia){const o=document.createElement("option");o.value="";o.textContent="— ninguna —";sel.appendChild(o);}
  const grupos={};D.s.forEach(x=>{(grupos[x.g]=grupos[x.g]||[]).push(x);});
  D.grupos.forEach(g=>{if(!grupos[g])return;const og=document.createElement("optgroup");og.label=g;
    grupos[g].forEach(x=>{const o=document.createElement("option");o.value=x.id;o.textContent=x.n;og.appendChild(o);});sel.appendChild(og);});
}
function anios(){["pp-desde","pp-hasta"].forEach(id=>{const s=$(id);s.textContent="";
  for(let a=A0;a<=A1;a++){const o=document.createElement("option");o.value=a;o.textContent=a;s.appendChild(o);}});}

function series(){const r=[S[st.a]];if(st.b&&S[st.b]&&st.b!==st.a)r.push(S[st.b]);return r;}

function resumen(ss,ds){
  const t=$("pp-resumen");t.textContent="";
  const h=t.insertRow();["Serie","Total del periodo","Promedio diario","Máximo ("+(st.vista==="A"?"año":st.vista==="M"?"mes":"día")+")",
    st.vista==="D"?"Días sin producción":"Meses sin producción"]
    .forEach(c=>{const th=document.createElement("th");th.textContent=c;h.appendChild(th);});
  ss.forEach((x,i)=>{const d=ds[i],tb=d.reduce((s,r)=>s+r.bbl,0),td=d.reduce((s,r)=>s+r.d,0);
    let mx=null;d.forEach(r=>{if(!mx||r.v>mx.v)mx=r;});
    const m=mensual(x);let ceros=0,ini=m.findIndex(v=>v>0);
    if(st.vista==="D")ceros=d.filter(r=>!(r.bbl>0)).length;
    else for(let t=Math.max((st.desde-A0)*12,ini<0?D.n:ini);t<Math.min((st.hasta-A0+1)*12,D.n);t++)if(!(m[t]>0))ceros++;
    const tr=t.insertRow(),c0=tr.insertCell(),sw=document.createElement("i");
    sw.style.cssText=`display:inline-block;width:10px;height:10px;border-radius:2px;margin-right:6px;vertical-align:-1px;background:${COL[i]}`;
    c0.appendChild(sw);c0.appendChild(document.createTextNode(x.n));
    const b=document.createElement("b");b.textContent=vol(tb);tr.insertCell().appendChild(b);
    tr.insertCell().textContent=esGas()?num(td?tb/td/1000:0,1)+" millones de pies³/día":num(td?tb/td:0,0)+" bpd";
    tr.insertCell().textContent=mx&&mx.v>0?num(mx.v,esGas()&&st.u==="bpd"?1:0)+" "+uni()+" · "+mx.lab:"–";
    tr.insertCell().textContent=num(ceros,0);
  });
}

function grafico(ss,ds){
  const box=$("pp-graf"),tip=$("pp-tip");box.querySelectorAll("svg").forEach(s=>s.remove());tip.style.display="none";
  const W=box.clientWidth||700,H=box.clientHeight||270,M={l:52,r:10,t:10,b:26},iw=W-M.l-M.r,ih=H-M.t-M.b;
  const svg=el("svg",{viewBox:`0 0 ${W} ${H}`,role:"img","aria-label":$("pp-titulo").textContent});box.insertBefore(svg,tip);
  const n=ds[0].length;if(!n)return;
  let max=0;ds.forEach(d=>d.forEach(r=>{if(r.v>max)max=r.v;}));
  const p=paso(max),top=Math.max(p,Math.ceil(max/p)*p),y=v=>M.t+ih-(v/top)*ih;
  for(let v=0;v<=top+1e-9;v+=p){el("line",{x1:M.l,x2:W-M.r,y1:y(v),y2:y(v),stroke:v===0?"#b9b8b2":GRID,"stroke-width":1},svg);
    const tx=el("text",{x:M.l-6,y:y(v)+4,"text-anchor":"end","font-size":11,fill:INK2},svg);tx.textContent=corto(v);}
  const bw=iw/n,cx=i=>M.l+bw*(i+.5);
  // etiquetas del eje X sin chocar
  const cada=Math.max(1,Math.ceil(n/Math.max(2,Math.floor(iw/(st.vista==="A"?38:30)))));
  if(st.vista==="D")ds[0].forEach((r,i)=>{
    if(n>45?r.dia===1:i%Math.max(1,Math.ceil(n/Math.floor(iw/26)))===0){
      const tx=el("text",{x:cx(i),y:H-8,"text-anchor":n>45?"start":"middle","font-size":11,fill:INK2},svg);tx.textContent=n>45?MES[r.mes]+(r.mes===0?" "+r.anio:""):r.eje;}});
  else ds[0].forEach((r,i)=>{const enero=st.vista==="M"&&n>24;
    if(enero?(r.mes===0&&((r.anio-st.desde)%Math.max(1,Math.ceil((n/12)/Math.floor(iw/40))))===0):i%cada===0){
      const tx=el("text",{x:cx(i),y:H-8,"text-anchor":"middle","font-size":11,fill:INK2},svg);tx.textContent=enero?String(r.anio):r.eje;}});
  {const i=n-1,r=ds[0][i];                       // el último mes o año incompleto siempre lleva su etiqueta («oct*»)
    if(st.vista!=="D"&&r&&r.parcial&&!(st.vista==="M"&&n>24)&&i%cada!==0){
      const tx=el("text",{x:cx(i),y:H-8,"text-anchor":"middle","font-size":11,fill:INK2},svg);tx.textContent=r.eje;}}
  const marcas=el("g",{},svg),cruz=el("line",{y1:M.t,y2:M.t+ih,stroke:"#8a8984","stroke-width":1,visibility:"hidden"},svg);
  const puntos=[];
  if(st.vista==="A"){
    const k=ds.length,g=2,w=Math.max(2,Math.min(46,(bw*.72-g*(k-1))/k));
    const porEmp=k===1&&ss[0].ob;                  // una sola serie de campo o lote: barras apiladas por empresa operadora
    ds.forEach((d,j)=>d.forEach((r,i)=>{const x0=cx(i)-(k*w+g*(k-1))/2+j*(w+g),hh=M.t+ih-y(r.v);
      if(porEmp&&r.porv&&r.porv.length){let base=0;const ult=r.porv.length-1;
        r.porv.slice().sort((a,b)=>ordenOps(ss[0]).indexOf(a[0])-ordenOps(ss[0]).indexOf(b[0])).forEach(([op,v],q)=>{
          const y1=y(base+v),h1=y(base)-y1;base+=v;
          el("path",{d:barra(x0,y1,w,h1,q===ult?Math.min(4,w/2):0),fill:ppColOp(ss[0].id,op)||COL[0],"fill-opacity":r.parcial?.45:1,"data-i":i},marcas);});}
      else el("path",{d:barra(x0,y(r.v),w,hh,Math.min(4,w/2)),fill:COL[j],"fill-opacity":r.parcial?.45:1,"data-i":i},marcas);}));
  } else {
    ds.forEach((d,j)=>{let s="",sd="";
      d.forEach((r,i)=>{const pt=cx(i).toFixed(1)+","+y(r.v).toFixed(1);
        if(r.diario&&st.vista==="M"&&i>0)sd+="M"+cx(i-1).toFixed(1)+","+y(d[i-1].v).toFixed(1)+"L"+pt;   // tramo desde reportes diarios
        else s+=(s?"L":"M")+pt;});
      el("path",{d:s,fill:"none",stroke:COL[j],"stroke-width":2,"stroke-linejoin":"round"},marcas);
      if(sd)el("path",{d:sd,fill:"none",stroke:COL[j],"stroke-width":2,"stroke-dasharray":"5 4"},marcas);
      d.forEach((r,i)=>{if(r.parcial&&st.vista==="M")el("circle",{cx:cx(i),cy:y(r.v),r:4.5,fill:"#fcfcfb",stroke:COL[j],"stroke-width":2},marcas);});
      puntos.push(el("circle",{r:4,fill:COL[j],stroke:"#fcfcfb","stroke-width":2,visibility:"hidden"},svg));});
  }
  const hit=el("rect",{x:M.l,y:M.t,width:iw,height:ih,fill:"transparent",style:"cursor:"+(st.vista==="A"||(st.vista==="M"&&DIA)?"pointer":"crosshair"),tabindex:0},svg);
  let actual=-1;
  function mostrar(i){
    if(i<0||i>=n){ocultar();return;}actual=i;const r0=ds[0][i];
    cruz.setAttribute("x1",cx(i));cruz.setAttribute("x2",cx(i));cruz.setAttribute("visibility","visible");
    marcas.querySelectorAll("path[data-i]").forEach(b=>b.setAttribute("opacity",+b.dataset.i===i?1:.55));
    puntos.forEach((c,j)=>{c.setAttribute("cx",cx(i));c.setAttribute("cy",y(ds[j][i].v));c.setAttribute("visibility","visible");});
    tip.textContent="";const t=document.createElement("div");t.className="s";t.textContent=r0.lab;tip.appendChild(t);
    ds.forEach((d,j)=>{const f=document.createElement("div"),k=document.createElement("i"),v=document.createElement("span"),s=document.createElement("span");
      k.style.borderColor=COL[j];v.className="v";v.textContent=num(d[i].v,esGas()&&st.u==="bpd"?1:0)+" "+uni();s.className="s";s.textContent="  "+ss[j].n;
      f.appendChild(k);f.appendChild(v);f.appendChild(s);tip.appendChild(f);
      if(d[i].porv&&d[i].porv.length>1&&ds.length===1)d[i].porv.forEach(([op,v])=>{const o=document.createElement("div"),c=document.createElement("i");
        c.style.cssText="width:9px;height:9px;border:0;border-radius:2px;background:"+(ppColOp(ss[0].id,op)||COL[0]);
        o.className="s";o.style.paddingLeft="17px";o.appendChild(c);o.appendChild(document.createTextNode(op+": "+num(v,esGas()&&st.u==="bpd"?1:0)+" "+uni()));tip.appendChild(o);});
      else if(d[i].ops){const o=document.createElement("div");o.className="s";o.style.paddingLeft="17px";o.textContent=d[i].ops;tip.appendChild(o);}});
    const pista=st.vista==="A"?"Clic: ver los meses de "+r0.anio:(st.vista==="M"&&hayDias(r0.anio,r0.mes)?"Clic: ver los días de "+MES[r0.mes]+" "+r0.anio:"");
    if(pista){const a=document.createElement("div");a.className="s";a.style.marginTop="3px";a.textContent=pista;tip.appendChild(a);}
    tip.style.display="block";const tw=tip.offsetWidth,x=cx(i)+12;
    tip.style.left=(x+tw>W?cx(i)-tw-12:x)+"px";tip.style.top=M.t+"px";
  }
  function ocultar(){actual=-1;tip.style.display="none";cruz.setAttribute("visibility","hidden");
    puntos.forEach(c=>c.setAttribute("visibility","hidden"));marcas.querySelectorAll("path[data-i]").forEach(b=>b.setAttribute("opacity",1));}
  function indice(ev){const r=svg.getBoundingClientRect(),x=(ev.clientX-r.left)*(W/r.width);return Math.min(n-1,Math.max(0,Math.floor((x-M.l)/bw)));}
  hit.addEventListener("pointermove",ev=>mostrar(indice(ev)));
  hit.addEventListener("pointerleave",ocultar);
  function entrar(r){if(!r)return;if(st.vista==="A")ppMeses(r.anio);else if(st.vista==="M"&&hayDias(r.anio,r.mes))ppDias(r.anio,r.mes);}
  hit.addEventListener("click",ev=>entrar(ds[0][indice(ev)]));
  hit.addEventListener("keydown",ev=>{if(ev.key==="ArrowRight")mostrar(Math.min(n-1,actual+1));else if(ev.key==="ArrowLeft")mostrar(Math.max(0,actual<0?n-1:actual-1));
    else if(ev.key==="Enter"&&actual>=0)entrar(ds[0][actual]);else return;ev.preventDefault();});
  hit.addEventListener("blur",ocultar);
}

function tabla(ss,ds){
  const t=$("pp-tabla");t.textContent="";const h=t.insertRow();
  [st.vista==="A"?"Año":st.vista==="M"?"Mes":"Día"].concat(ss.map(x=>x.n+" ("+uni()+")"),["Operador(es)"]).forEach(c=>{const th=document.createElement("th");th.textContent=c;h.appendChild(th);});
  ds[0].forEach((r,i)=>{const tr=t.insertRow();tr.insertCell().textContent=r.lab;ds.forEach(d=>tr.insertCell().textContent=num(d[i].v,esGas()&&st.u==="bpd"?1:0));
    tr.insertCell().textContent=ss.map(x=>x.o&&x.o[r.anio]||"").filter(Boolean).join(" | ");});
}

function dibujar(){
  if(!S[st.a])st.a=D.def;if(st.desde>st.hasta){const x=st.desde;st.desde=st.hasta;st.hasta=x;}
  ["a","b","prod","vista","desde","hasta","u"].forEach(k=>{$("pp-"+k).value=st[k];});
  const ss=series(),ds=ss.map(datos);
  $("pp-u").options[0].textContent=esGas()?"Millones de pies cúbicos":"Barriles";
  $("pp-u").options[1].textContent=esGas()?"Millones de pies cúbicos por día":"Barriles por día";
  $("pp-cab-t").textContent="Producción fiscalizada · "+ss.map(x=>x.n).join(" vs. ");
  $("pp-titulo").textContent=ss.map(x=>x.n).join(" vs. ")+" · producción fiscalizada de "+PROD[st.prod]+" · "+
    (esGas()?(st.u==="bpd"?"millones de pies cúbicos por día":"millones de pies cúbicos"):(st.u==="bpd"?"barriles por día":"barriles"))+
    (st.vista==="A"?" por año":st.vista==="M"?" por mes":"")+" · "+(st.dmes?MES[st.dmes.m]+" "+st.dmes.a:(st.desde===st.hasta?st.desde:st.desde+"–"+st.hasta));
  if(st.vista==="D")$("pp-titulo").textContent=$("pp-titulo").textContent.replace(/ · (barriles|millones de pies cúbicos)( por día)?/,(m,u)=>` · ${u} por día (diario)`);
  const finEst=MES[(D.n-1)%12]+" "+A1s;
  let ult=" Estadística mensual de PERUPETRO hasta "+finEst+".";
  if(st.vista==="A"&&D.n%12)ult+=" "+A1s+"*: solo ene–"+MES[(D.n-1)%12]+" (barra clara); los meses siguientes, de los reportes diarios, se ven en la vista mensual.";
  if(st.vista==="M"){const dd=ds[0].filter(r=>r.diario),cierre=dd.filter(r=>!r.parcial),cur=dd.find(r=>r.parcial);
    if(cierre.length)ult+=" "+MES[cierre[0].mes]+(cierre.length>1?"–"+MES[cierre[cierre.length-1].mes]:"")+" "+cierre[0].anio+
      ": suma de los reportes diarios de cierre de mes (línea punteada).";
    if(cur)ult+=" "+MES[cur.mes]+"* "+cur.anio+": reporte diario en curso, días 1–"+cur.ult+" (mes incompleto: en barriles se ve más bajo; usa «barriles por día» para comparar).";}
  let nota;
  $("pp-u").disabled=st.vista==="D";                     // en la vista diaria cada valor ya es «por día»
  if(st.vista==="D")nota=DIA?"Reportes diarios de PERUPETRO (producción fiscalizada del día)."+(esGas()?"":
      " En líquidos varios lotes se fiscalizan por entregas: hay días en 0 y días altos; para comparar usa el promedio diario.")
                          :"No hay datos diarios: corre el notebook de producción con la casilla «Producción diaria».";
  else nota=(st.vista==="A"?"Pasa el cursor por una barra para ver el detalle; haz clic para ver sus meses."
             :"Pasa el cursor para ver el detalle de cada mes"+(DIA?"; los meses con reporte diario se abren con un clic.":"."))+ult+
            " Fuente: PERUPETRO, producción fiscalizada (medida en el punto de fiscalización); un mes en 0 puede ser paralización.";
  $("pp-nota").textContent=nota;
  $("pp-volver").style.display=st.pila.length?"":"none";
  if(st.pila.length)$("pp-volver").textContent="↩ "+(st.pila[st.pila.length-1].vista==="A"?"Serie anual":"Serie mensual");
  const ley=$("pp-leyenda");ley.textContent="";
  if(ss.length>1)ss.forEach((x,i)=>{const s=document.createElement("span"),k=document.createElement("i");k.style.background=COL[i];s.appendChild(k);s.appendChild(document.createTextNode(x.n));ley.appendChild(s);});
  else if(st.vista==="A"&&ss[0].ob){const vis=new Set();ds[0].forEach(r=>(r.por||[]).forEach(z=>vis.add(z[0])));
    ordenOps(ss[0]).filter(o=>vis.has(o)).forEach(o=>{const s=document.createElement("span"),k=document.createElement("i");
      k.style.background=ppColOp(ss[0].id,o);s.appendChild(k);s.appendChild(document.createTextNode(o));ley.appendChild(s);});
    if(ley.childNodes.length)ley.insertBefore(document.createTextNode("Operador: "),ley.firstChild);}
  resumen(ss,ds);grafico(ss,ds);tabla(ss,ds);
  $("pp-tabla-c").style.display=st.tabla?"block":"none";$("pp-btab").textContent=st.tabla?"Ocultar tabla":"Tabla";
}
function guardar(){st.pila.push({vista:st.vista,desde:st.desde,hasta:st.hasta,dmes:st.dmes});}
window.ppMeses=function(a){guardar();st.vista="M";st.dmes=null;st.desde=st.hasta=a;dibujar();};
window.ppDias=function(a,m){guardar();st.vista="D";st.dmes={a:a,m:m};st.desde=st.hasta=a;dibujar();};
window.ppVolver=function(){const p=st.pila.pop();if(p){st.vista=p.vista;st.desde=p.desde;st.hasta=p.hasta;st.dmes=p.dmes;}dibujar();};
window.ppTabla=function(){st.tabla=!st.tabla;dibujar();};
window.ppMin=function(){const p=$("pp-panel"),m=p.classList.toggle("pp-mini");$("pp-min").textContent=m?"+":"–";
  $("pp-fondo").style.display=m?"none":"block";                 // minimizado: el mapa vuelve a quedar libre
  $("pp-min").title=m?"Expandir":"Minimizar";$("pp-min").setAttribute("aria-label",m?"Expandir panel":"Minimizar panel");if(!m)dibujar();};
window.ppCerrar=function(){$("pp-panel").style.display="none";$("pp-fondo").style.display="none";$("pp-abrir").style.display="";};
function ctxDe(id){const x=S[id];if(!x||!(x.cp||x.dp))return null;    // Perú (total): sin pestañas
  const l=id.startsWith("L:")?id.slice(2):x.cp||"";
  return {c:x.cp||"",l:l,d:x.dp||"",pp:id,n:x.n};}
function pestanas(ctx){               // abierto desde la ficha de un lote: las mismas pestañas del panel de estadísticas
  const t=$("pp-tabs");st.ctx=ctx&&window.ppActTabs?ctx:null;
  if(!st.ctx){t.style.display="none";t.textContent="";return;}
  t.innerHTML='<button type="button" role="tab" aria-selected="true">Producción</button>'+
    window.ppActTabs(ctx).map(([k,n,ok])=>`<button type="button" role="tab" aria-selected="false" data-t="${k}" ${ok?"":"disabled"}>${n}</button>`).join("");
  t.style.display="flex";
  t.querySelectorAll("button[data-t]").forEach(b=>b.onclick=()=>{const o=Object.assign({},st.ctx,{t:b.dataset.t});ppCerrar();window.ppActAbrir(o);});}
window.ppAbrir=function(id,anio,ctx,rango){
  if(id&&S[id])st.a=id;
  st.conPest=true;pestanas(ctx||ctxDe(st.a));     // con o sin ficha de lote: siempre las mismas pestañas (las de la serie elegida)
  for(const k in window){try{const o=window[k];if(o&&window.L&&o instanceof L.Map)o.closePopup();}catch(e){}}  // la ficha no tapa el panel
  $("pp-abrir").style.display="none";$("pp-fondo").style.display="block";const p=$("pp-panel");p.style.display="flex";
  p.classList.remove("pp-mini");$("pp-min").textContent="–";st.pila=[];st.dmes=null;st.vista="A";st.desde=A0;st.hasta=A1;
  if(rango){st.desde=Math.max(A0,rango[0]);st.hasta=Math.min(A1,rango[1]);}
  dibujar();if(anio)ppMeses(anio);
};
window.ppImagenProd=function(){const ss=series();
  window.ppImagen($("pp-graf").querySelector("svg"),{titulo:$("pp-titulo").textContent,ley:window.ppLeyendaDe($("pp-leyenda")),
    nota:$("pp-nota").textContent.replace(/^Pasa el cursor[^.]*\.\s*/,""),archivo:"produccion_"+ss.map(x=>x.n).join("_vs_")});};
window.ppCSV=function(){const ss=series(),ds=ss.map(datos);
  let s="﻿"+[st.vista==="A"?"AÑO":st.vista==="M"?"MES":"FECHA"].concat(ss.map(x=>x.n+" ("+uni()+")")).join(";")+"\n";
  ds[0].forEach((r,i)=>{s+=[r.lab].concat(ds.map(d=>(Math.round(d[i].v*10)/10).toString().replace(".",","))).join(";")+"\n";});
  const a=document.createElement("a");a.href=URL.createObjectURL(new Blob([s],{type:"text/csv"}));
  a.download="produccion_"+ss.map(x=>x.n).join("_vs_").replace(/[^\w-]+/g,"_")+".csv";a.click();setTimeout(()=>URL.revokeObjectURL(a.href),1000);};
window.ppLey=function(){const c=$("pp-ley-c"),b=$("pp-ley-btn"),ab=c.style.display==="none";c.style.display=ab?"block":"none";
  b.textContent=ab?"–":"+";b.title=ab?"Minimizar leyenda":"Mostrar leyenda";b.setAttribute("aria-expanded",ab);};
function init(){opciones($("pp-a"),false);opciones($("pp-b"),true);anios();
  if(DIA){const o=document.createElement("option");o.value="D";o.textContent="Diaria";$("pp-vista").appendChild(o);}
  ["a","b","prod","vista","desde","hasta","u"].forEach(k=>$("pp-"+k).addEventListener("change",e=>{
    st[k]=(k==="desde"||k==="hasta")?+e.target.value:e.target.value;
    if(k==="a")pestanas(ctxDe(st.a));   // otra serie: las pestañas pasan a ese campo o departamento (Perú: sin pestañas)
    if(k==="vista"){st.pila=[];st.dmes=null;
      if(st.vista==="D"&&DIA&&(st.hasta<DA0||st.desde>DA1)){st.desde=DA0;st.hasta=DA1;}}   // la serie diaria es corta
    if(k==="desde"||k==="hasta")st.dmes=null;
    dibujar();}));
  let tm;window.addEventListener("resize",()=>{clearTimeout(tm);tm=setTimeout(()=>{if($("pp-panel").style.display==="flex")dibujar();},150);});
  document.addEventListener("keydown",ev=>{if(ev.key==="Escape"&&$("pp-panel").style.display==="flex")ppCerrar();});
  // la ficha (popup) de Leaflet: evitar que el clic en el botón se propague al mapa
  document.addEventListener("click",ev=>{const b=ev.target.closest&&ev.target.closest("[data-pp]");if(b){ev.preventDefault();ppAbrir(b.dataset.pp);}});
}
if(document.readyState==="loading")document.addEventListener("DOMContentLoaded",init);else init();
})();
</script>
<script>
// mueve «Producción fiscalizada anual y mensual» dentro del selector de capas, debajo de «Productos»
(function mover(n){const l=document.querySelector(".leaflet-control-layers-list"),b=document.getElementById("pp-abrir");
  if(!b)return;
  if(l&&(document.querySelector(".pp-prod-sel")||n>40)){const d=document.createElement("div");d.className="pp-abrir-caja";
    d.appendChild(b);l.appendChild(d);b.classList.add("en-capas");if(window.L&&L.DomEvent)L.DomEvent.disableClickPropagation(d);return;}
  if(n<100)setTimeout(()=>mover(n+1),100);})(0);
</script>
'''
ACT_HTML = r'''<script>
// «Guardar como imagen»: arma un PNG con título, leyenda, el gráfico (SVG) y la nota de fuente
window.ppImagen=window.ppImagen||function(svg,o){
  if(!svg)return;o=o||{};
  const vb=svg.viewBox&&svg.viewBox.baseVal,W=(vb&&vb.width)||svg.clientWidth||700,H=(vb&&vb.height)||svg.clientHeight||260;
  const c=svg.cloneNode(true),org=svg.querySelectorAll("*"),cl=c.querySelectorAll("*");
  org.forEach((e,i)=>{const cs=getComputedStyle(e),d=cl[i];                       // estilos de la hoja → atributos
    ["fill","stroke","stroke-width","stroke-dasharray","opacity","fill-opacity","font-size","font-weight"].forEach(p=>{
      const v=cs.getPropertyValue(p);if(v&&!d.getAttribute(p))d.setAttribute(p,v);});});
  c.setAttribute("xmlns","http://www.w3.org/2000/svg");c.setAttribute("width",W);c.setAttribute("height",H);
  c.setAttribute("font-family","Barlow, Arial, sans-serif");
  const img=new Image();
  img.onload=()=>{
    const k=2,ancho=Math.max(W,640),pad=24,cv=document.createElement("canvas"),ctx=cv.getContext("2d");
    const F=(w,s)=>`${w} ${s}px Barlow, Arial, sans-serif`;
    const lineas=(t,f,max)=>{ctx.font=f;const out=[];let l="";(t||"").split(/\s+/).forEach(p=>{const q=l?l+" "+p:p;
      if(ctx.measureText(q).width>max&&l){out.push(l);l=p;}else l=q;});if(l)out.push(l);return out;};
    const tit=lineas(o.titulo,F(700,17),ancho),sub=lineas(o.sub,F(400,12.5),ancho),nota=lineas(o.nota,F(400,11),ancho);
    const ley=o.ley||[];ctx.font=F(400,12.5);const filasLey=[];let fl=[],x=0;
    ley.forEach(([col,nom])=>{const w=16+ctx.measureText(nom).width+14;if(x+w>ancho&&fl.length){filasLey.push(fl);fl=[];x=0;}fl.push([col,nom]);x+=w;});
    if(fl.length)filasLey.push(fl);
    const h=pad+tit.length*22+sub.length*17+(sub.length?6:0)+filasLey.length*19+(filasLey.length?6:0)+H+10+nota.length*15+pad;
    cv.width=(ancho+2*pad)*k;cv.height=h*k;ctx.scale(k,k);ctx.fillStyle="#fefefe";ctx.fillRect(0,0,ancho+2*pad,h);
    let y=pad;ctx.textBaseline="top";
    ctx.fillStyle="#181818";ctx.font=F(700,17);tit.forEach(t=>{ctx.fillText(t,pad,y);y+=22;});
    ctx.fillStyle="#555";ctx.font=F(400,12.5);sub.forEach(t=>{ctx.fillText(t,pad,y);y+=17;});if(sub.length)y+=6;
    filasLey.forEach(f=>{let xx=pad;f.forEach(([col,nom])=>{ctx.fillStyle=col;ctx.fillRect(xx,y+3,10,10);ctx.fillStyle="#444";
      ctx.font=F(400,12.5);ctx.fillText(nom,xx+15,y+1);xx+=16+ctx.measureText(nom).width+14;});y+=19;});if(filasLey.length)y+=6;
    ctx.drawImage(img,pad,y,W,H);y+=H+10;
    ctx.fillStyle="#666";ctx.font=F(400,11);nota.forEach(t=>{ctx.fillText(t,pad,y);y+=15;});
    cv.toBlob(b=>{const a=document.createElement("a");a.href=URL.createObjectURL(b);
      a.download=(o.archivo||"grafico").replace(/[^\wÁÉÍÓÚÑáéíóúñ-]+/g,"_").slice(0,90)+".png";
      document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(a.href),1500);},"image/png");};
  img.src="data:image/svg+xml;charset=utf-8,"+encodeURIComponent(new XMLSerializer().serializeToString(c));
};
// leyenda de un contenedor: [[color, nombre]] de sus <span><i style="background"></i>nombre</span>
window.ppLeyendaDe=window.ppLeyendaDe||function(el){return el?[...el.querySelectorAll("span")].filter(s=>s.querySelector("i"))
  .map(s=>[getComputedStyle(s.querySelector("i")).backgroundColor,s.textContent.trim()]):[];};
</script>
<link rel="preconnect" href="https://fonts.googleapis.com"><link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Barlow:ital,wght@0,400;0,500;0,600;0,700;1,400&display=swap" rel="stylesheet">
<style>
/* tipo de letra del proyecto: Barlow (Google Fonts); sin conexión cae en la letra del sistema. Los íconos (<i>) no se tocan */
.leaflet-container, .leaflet-container *:not(i), #pp-act, #pp-act *:not(i), #pp-panel, #pp-panel *:not(i), #pp-abrir, #pp-ley, #pp-ley *:not(i)
  {font-family:"Barlow",system-ui,-apple-system,"Segoe UI",Roboto,sans-serif !important}
#pp-act-fondo{position:fixed;inset:0;background:rgba(0,0,0,.35);z-index:10000;display:none}
#pp-act{position:fixed;left:50%;top:50%;transform:translate(-50%,-50%);z-index:10001;display:none;background:#fff;border-radius:10px;
  box-shadow:0 4px 24px rgba(0,0,0,.35);width:min(900px,calc(100vw - 24px));max-height:calc(100vh - 24px);overflow:auto;
  font:12px/1.4 system-ui,-apple-system,"Segoe UI",Roboto,sans-serif;color:#1a1a1a}
#pp-act .cab{display:flex;align-items:flex-start;justify-content:space-between;gap:8px;padding:12px 14px 6px;border-bottom:1px solid #eee}
#pp-act .cab b{font-size:15px}
#pp-act .cab .sub{color:#666;font-size:12px}
#pp-act .x{border:0;background:#f3f2ee;border-radius:6px;width:28px;height:28px;cursor:pointer;font:600 16px sans-serif}
#pp-act .tabs{display:flex;gap:4px;padding:8px 14px 0;flex-wrap:wrap}
#pp-act .tabs button{border:1px solid #d6d4cc;background:#f7f6f2;border-radius:6px 6px 0 0;padding:5px 10px;cursor:pointer;font:600 12px sans-serif;color:#333}
#pp-act .tabs button[aria-selected=true]{background:#fff;border-bottom-color:#fff;color:#000}
#pp-act .tabs button:disabled{opacity:.4;cursor:default}
#pp-act .cuerpo{padding:8px 14px 14px;border-top:1px solid #d6d4cc;margin-top:-1px}
#pp-act .res{color:#333;margin:4px 0 6px}
#pp-act .tit-v{font:700 13.5px system-ui,sans-serif;color:#1a1a1a;margin:4px 0 0}
#pp-act .ley{display:flex;gap:12px;flex-wrap:wrap;color:#444;margin:2px 0 4px}
#pp-act .ley i{display:inline-block;width:10px;height:10px;border-radius:2px;margin-right:4px;vertical-align:middle}
#pp-act svg{display:block;width:100%;height:auto;overflow:visible}
#pp-act svg .b{cursor:pointer}
#pp-act svg .sel{stroke:#000;stroke-width:1.5}
#pp-act .tip{position:fixed;pointer-events:none;background:#222;color:#fff;border-radius:5px;padding:5px 8px;font-size:11px;
  display:none;z-index:10002;white-space:nowrap}
#pp-act table{border-collapse:collapse;width:100%;margin-top:8px}
#pp-act th,#pp-act td{text-align:right;padding:3px 6px;border-bottom:1px solid #eee}
#pp-act th:first-child,#pp-act td:first-child{text-align:left}
#pp-act th{color:#555;font-weight:600;background:#faf9f6}
#pp-act tfoot td{font-weight:700}
#pp-act .nota{color:#777;margin-top:6px}
#pp-act .herr{display:flex;gap:6px;justify-content:flex-end;align-items:center;flex-wrap:wrap;margin:2px 0 4px}
#pp-act .herr button{border:1px solid #c9c8c2;background:#f3f2ee;border-radius:5px;padding:3px 9px;cursor:pointer;font:600 12px sans-serif}
#pp-act .tabla-wrap{max-height:340px;overflow:auto;border:1px solid #eee;border-radius:6px}
#pp-act .tabla-wrap table{margin-top:0}
#pp-act .tabla-wrap thead th{position:sticky;top:0}
#pp-act .vistas{display:flex;gap:4px;flex-wrap:wrap;margin:2px 0 6px}
#pp-act .vistas button{border:1px solid #c9c8c2;background:#f7f6f2;border-radius:14px;padding:3px 10px;cursor:pointer;font:600 11.5px sans-serif;color:#333}
#pp-act .vistas button[aria-pressed=true]{background:#111149;border-color:#111149;color:#fefefe}
#pp-act .vistas .grupo{display:flex;gap:4px;align-items:center;flex-wrap:wrap;padding:2px 8px 2px 0;margin-right:4px;border-right:1px solid #e3e1db}
#pp-act .vistas .grupo:last-child{border-right:0}
#pp-act .vistas .grupo>span{font:600 10.5px sans-serif;color:#6b6b6b;text-transform:uppercase;letter-spacing:.03em}
#pp-act .herr .rg{font:12px sans-serif;color:#555;margin-right:6px}
#pp-act .herr .rg select{font:12px sans-serif;padding:2px 3px;border:1px solid #c9c8c2;border-radius:5px;background:#fff;margin-left:3px}
#pp-act .nota.fuentes ul{margin:3px 0 0 16px;padding:0}
#pp-act .nota.fuentes li{margin:1px 0}
#pp-act .alcance{font:12px sans-serif;color:#4a4a4a;background:#f4f6f9;border-left:3px solid #72d2d5;padding:4px 8px;margin:6px 0 2px;border-radius:0 4px 4px 0}
#pp-act .sel-anio{display:flex;align-items:center;gap:6px;margin:8px 0 0;color:#333}
#pp-act .sel-anio select{font:600 12px sans-serif;padding:2px 4px;border:1px solid #c9c8c2;border-radius:5px;background:#fff}
#pp-act .dep-b{border:1px solid #c9c8c2;background:#f7f6f2;border-radius:12px;padding:1px 8px;cursor:pointer;font:600 12px sans-serif;color:#111149}
#pp-act .dep-b[aria-expanded=true]{background:#111149;border-color:#111149;color:#fefefe}
#pp-act .benef{margin-top:10px;padding:8px 10px;border:1px solid #ccd3df;border-radius:6px;background:#f5f7fb}
#pp-act .benef b{font-size:12.5px}
</style>
<div id="pp-act-fondo"></div>
<div id="pp-act" role="dialog" aria-modal="true" aria-labelledby="pp-act-tit">
  <div class="cab"><div><b id="pp-act-tit"></b><div class="sub" id="pp-act-sub"></div></div>
    <button type="button" class="x" title="Cerrar (Esc)" aria-label="Cerrar">×</button></div>
  <div class="tabs" role="tablist"></div>
  <div class="cuerpo"></div>
  <div class="tip"></div>
</div>
<script>
(function(){
const A=__ACT__;
window.ppActDatos=A;                           // «Comparar lotes» lee el canon generado por lote
const COL_P={d:"#111149",e:"#72d2d5"}, NOM_P={d:"Desarrollo",e:"Exploratorios y confirmatorios"};
const COL_R={"PETRÓLEO":"#111149","LGN":"#ffc41d","GAS NATURAL":"#72d2d5","CONDENSADO":"#ccd3df"};
const COL_C="#111149";
const TABS=[["pp","Producción"],["p","Pozos perforados"],["r","Regalías"],["c","Canon y sobrecanon"],["e","Empresas"]];
// paleta del proyecto: azul noche #111149, turquesa #72D2D5, gris azulado #CCD3DF, amarillo #FFC41D, negro #181818 (el naranja #DD6140 queda al final, solo si hay más de 9 series)
const PAL=["#111149","#72d2d5","#ccd3df","#ffc41d","#2a9da1","#5a5a9e","#8c96ad","#181818","#b9e9eb","#dd6140"];
const caja=document.getElementById("pp-act"),fondo=document.getElementById("pp-act-fondo"),tip=caja.querySelector(".tip");
let st=null;
function num(v,d){const s=Math.abs(v).toFixed(d||0).split(".");return (v<0?"-":"")+s[0].replace(/\B(?=(\d{3})+(?!\d))/g,".")+(s[1]?","+s[1]:"");}
// canon: cuatro vistas (st.cv) — l: generado por el lote · b: departamento por beneficiario · n: nivel de gobierno · m: municipalidades del lote
const VISTAS=[["l","Generado por el lote"],["b","Por beneficiario"],["n","Por nivel de gobierno"],["m","Municipalidades del lote"]];
// grupos de vistas según su alcance: el lote · todo el departamento · las municipalidades donde está el lote
function gruposVistas(){const lote=st.l?"Lote "+st.l:"Campo "+st.c,dep=st.d?titulo(st.d):"Departamento";
  return [[lote,["l"]],[dep+" (todos sus lotes)",["b","n"]],["Donde está el lote",["m"]]];}
function alcance(){const lote=st.l?"lote "+st.l:"campo "+st.c,dep=st.d?titulo(st.d):"el departamento";
  if(st.cv==="l")return `📍 <b>Alcance: el ${lote}.</b> Lo que generó su producción y cómo se reparte entre departamentos.`;
  if(st.cv==="m")return `📍 <b>Alcance: municipalidades de los distritos y la provincia donde está el ${lote}.</b> Cada una recibe canon por todos los lotes del departamento, no solo por este.`;
  return `📍 <b>Alcance: todo ${dep}.</b> Es el canon que recibió el departamento por todos sus lotes${st.c||st.l?", no solo por el "+lote:""}.`;}
// vista «Nivel de gobierno» (st.nf): mef = recaudado por nivel en el departamento (MEF) · pp = beneficiarios de PERUPETRO agrupados
const NIV={R:["Gobierno regional","#111149"],M:["Gobiernos locales","#72d2d5"],E:["Gobierno nacional","#ccd3df"],
  O:["Universidades, institutos e IIAP","#ffc41d"],D:["Descuento: pago de deuda (Ley 29289)","#a9b1c2"]};
const GRUPO_PP={"GOBIERNO REGIONAL":"R","MUNICIPALIDADES":"M","UNIVERSIDADES":"O","INSTITUTOS":"O","IIAP":"O","LEY 29289":"D","OTRO":"O"};
function nf(o){return "pp";}   // solo PERUPETRO (el recaudado del MEF por nivel queda en el CSV, no en el mapa)
const COL_B={"GOBIERNO REGIONAL":"#111149","MUNICIPALIDADES":"#72d2d5","UNIVERSIDADES":"#ccd3df","INSTITUTOS":"#ffc41d","IIAP":"#2a9da1","LEY 29289":"#a9b1c2","OTRO":"#5a5a9e"};
const NOM_B={"GOBIERNO REGIONAL":"Gobierno regional","MUNICIPALIDADES":"Municipalidades","UNIVERSIDADES":"Universidades","INSTITUTOS":"Institutos","IIAP":"IIAP","LEY 29289":"Descuento: pago de deuda (Ley 29289)","OTRO":"Otros"};
const COL_D=["#111149","#72d2d5","#ccd3df","#ffc41d","#2a9da1","#5a5a9e"];
function hayVista(v,o){o=o||st;return v==="l"?!!((A.cl||{})[o.c]):v==="m"?!!((A.cm||{})[o.c]):v==="n"?!!((A.cb||{})[o.d]):!!(A.c[o.d]||(A.cb||{})[o.d]);}
function rk(k){if(k!=="c")return k;return st.cv==="l"?"cl":st.cv==="m"?"cm":st.cv==="n"?(nf()==="mef"?"cng":"cb"):((A.cb||{})[st.d]?"cb":"c");}
function rangoTotal(k){const r=A.rango[rk(k)];const o=[];for(let a=r[0];a<=r[1];a++)o.push(a);return o;}
function rango(k){            // años de la serie dentro de «Desde» / «Hasta» (si el corte deja la serie vacía, toda la serie)
  const t=rangoTotal(k),o=t.filter(a=>(!st.desde||a>=st.desde)&&(!st.hasta||a<=st.hasta));return o.length?o:t;}
function titulo(s){return s.charAt(0)+s.slice(1).toLowerCase();}
function esc(s){return String(s).replace(/[&<>"]/g,c=>({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;"}[c]));}
// ── Empresas: operadores con producción (tramos de años), socios del contrato vigente y contratos terminados ──
function E(){return (A.e||{})[st.c]||{};}
function filasE(){            // una fila por lote y operador, con sus tramos de años
  const o=(E().o||[]).slice().sort((a,b)=>a[2]-b[2]||String(a[0]).localeCompare(String(b[0])));const fs=[];
  o.forEach(x=>{const k=x[0]+"|"+x[1];let f=fs.find(f=>f.k===k);if(!f){f={k,l:x[0],op:x[1],tr:[]};fs.push(f);}f.tr.push(x);});
  return fs;}
function resumenE(){const o=E().o||[];
  if(!o.length)return E().s?"Sin producción fiscalizada registrada en este campo.":"Sin datos.";
  const fin=Math.max(...o.map(x=>x[3])),act=o.filter(x=>x[3]===fin),ops=[...new Set(o.map(x=>x[1]))];
  return `Con producción en ${fin}: <b>${esc([...new Set(act.map(x=>x[1]))].join(" / "))}</b> (desde ${Math.min(...act.map(x=>x[2]))}) · `+
    `${ops.length} operador${ops.length>1?"es":""} con producción desde ${Math.min(...o.map(x=>x[2]))}`;}
function gantt(){const fs=filasE();if(!fs.length)return "";const an=rango("e");
  const ops=[...new Set(fs.map(f=>f.op))],col=op=>(window.ppColOp&&st.pp&&window.ppColOp(st.pp,op))||PAL[ops.indexOf(op)%PAL.length];
  const W=720,ml=200,mr=8,fh=22,mt=6,H=mt+fs.length*fh+22,w=(W-ml-mr)/an.length;let g="";
  const marca=(a,i)=>a%5===0||(i===0&&a%5<3)||(i===an.length-1&&a%5>2);   // sin años encimados (1994/1995)
  an.forEach((a,i)=>{if(marca(a,i)){const x=ml+i*w+w/2;
    g+=`<line x1="${x}" x2="${x}" y1="${mt}" y2="${H-20}" stroke="#eee"/><text x="${x}" y="${H-6}" font-size="10" fill="#666" text-anchor="middle">${a}</text>`;}});
  fs.forEach((f,j)=>{const y=mt+j*fh,et=`Lote ${f.l} · ${f.op}`;
    g+=`<text x="${ml-6}" y="${y+fh/2+4}" font-size="11" text-anchor="end" fill="#333">${esc(et.length>32?et.slice(0,31)+"…":et)}<title>${esc(et)}</title></text>`;
    f.tr.forEach((x,n)=>{const i0=x[2]-an[0],i1=x[3]-an[0];
      g+=`<rect class="b" style="cursor:pointer" data-e="${j}|${n}" x="${(ml+i0*w+0.5).toFixed(1)}" y="${y+3}" width="${Math.max((i1-i0+1)*w-1,1).toFixed(1)}" height="${fh-6}" rx="3" fill="${col(f.op)}"/>`;});});
  return `<svg viewBox="0 0 ${W} ${H}" role="img" aria-label="Operadores con producción por año">${g}</svg>`;}
function tablaSocios(){const s=E().s||[],t=E().t||[];let h="";
  if(s.length)h+=`<div class="res" style="margin-top:10px"><b>Contrato vigente</b> <span style="color:#777">(PERUPETRO, contratos al 31/12/${A.ed})</span></div>
    <table><thead><tr><th>Lote</th><th style="text-align:left">Empresa</th><th>Participación</th><th>Suscripción</th><th>Término</th><th>Fase</th></tr></thead><tbody>${
    s.map(x=>`<tr><td>${esc(x[0])}</td><td style="text-align:left">${esc(x[1])}</td><td>${x[2]!=null?num(x[2],x[2]%1?2:0)+" %":"–"}</td><td>${x[3]||"–"}</td><td>${x[4]||"–"}</td><td>${esc(x[5])}</td></tr>`).join("")}</tbody></table>`;
  else if(A.ed)h+=`<div class="nota">Sin contrato vigente al 31/12/${A.ed} en este campo.</div>`;
  if(t.length)h+=`<div class="res" style="margin-top:10px"><b>Contratos terminados</b></div>
    <table><thead><tr><th>Lote</th><th style="text-align:left">Operador</th><th>Suscripción</th><th>Último día de vigencia</th><th>Edición</th></tr></thead><tbody>${
    t.slice().sort((a,b)=>b[4]-a[4]).map(x=>`<tr><td>${esc(x[0])}</td><td style="text-align:left">${esc(x[1])}</td><td>${x[2]||"–"}</td><td>${x[3]||"–"}</td><td>${x[4]}</td></tr>`).join("")}</tbody></table>`;
  return h;}
// series apiladas por año: [{k, nombre, color, v:{año:valor}}]
function series(t){
  if(t==="p"){const d=A.p[st.c]||{};return ["d","e"].map(k=>{const v={};for(const l in d)for(const a in (d[l][k]||{}))v[a]=(v[a]||0)+d[l][k][a];
    return {k,nombre:NOM_P[k],color:COL_P[k],v};});}
  if(t==="r"){const d=A.r[st.c]||{};const hs=Object.keys(COL_R);return hs.map(h=>{const v={};for(const l in d)for(const a in (d[l][h]||{}))v[a]=(v[a]||0)+d[l][h][a];
    return {k:h,nombre:titulo(h),color:COL_R[h],v};}).filter(s=>Object.keys(s.v).length);}
  if(st.cv==="l"){const d=(A.cl||{})[st.c]||{};const deps=[...new Set(Object.values(d).flatMap(x=>Object.keys(x)))];
    deps.sort((x,y)=>(x===st.d?-1:y===st.d?1:x.localeCompare(y)));
    return deps.map((dp,i)=>{const v={};for(const l in d)for(const a in (d[l][dp]||{}))v[a]=(v[a]||0)+d[l][dp][a];
      return {k:dp,nombre:"Para "+titulo(dp),color:COL_D[i%COL_D.length],v};});}
  if(st.cv==="n"){
    if(nf()==="mef"){const d=(A.cng||{})[st.d]||{};return ["R","M"].filter(k=>d[k]).map(k=>({k,nombre:NIV[k][0],color:NIV[k][1],v:d[k]}));}
    const cb=(A.cb||{})[st.d]||{},g={};for(const t in cb){const k=GRUPO_PP[t]||"O";g[k]=g[k]||{};for(const a in cb[t])g[k][a]=(g[k][a]||0)+cb[t][a];}
    return ["R","M","O","D"].filter(k=>g[k]).map(k=>({k,nombre:NIV[k][0],color:NIV[k][1],v:g[k]}));}
  if(st.cv==="m"){const e=(A.cm||{})[st.c]||{d:[],v:{}};
    return munis().map((x,i)=>({k:x[0],nombre:x[1]+(x[3]==="PROVINCIAL"?" (provincial)":""),color:PAL[i%PAL.length],v:e.v[x[0]]||{}}));}
  const cb=(A.cb||{})[st.d];
  if(cb)return Object.keys(COL_B).filter(t=>cb[t]).map(t=>({k:t,nombre:NOM_B[t],color:COL_B[t],v:cb[t]}));
  return [{k:"c",nombre:"Canon y sobrecanon",color:COL_C,v:A.c[st.d]||{}}];
}
function munis(){const e=(A.cm||{})[st.c]||{d:[]};   // distritales por % del lote; las provinciales al final
  return e.d.slice().sort((x,y)=>(x[3]==="PROVINCIAL")-(y[3]==="PROVINCIAL")||(y[4]||0)-(x[4]||0));}
const UNI={p:["pozos",0],r:["millones de US$",2],c:["millones de S/",1]};
function uni(t){return t!=="c"?UNI[t]:st.cv==="l"?["millones de US$",2]:["millones de S/",2];}
function mesesNota(a){        // años con meses incompletos en la vista actual
  if(st.cv==="l"){const n=(A.cln||{})[a];return n&&n<12?`${a}: ${n} meses con dato.`:"";}
  if(st.cv==="m"){const ds=[...new Set(munis().map(x=>x[0].split("|")[0]))];
    return ds.map(d=>{const n=((A.cmn||{})[d]||{})[a];return n&&n<12?`${a}, ${titulo(d)}: ${n} meses con dato.`:"";}).filter(x=>x).join(" ");}
  return "";}
function grafico(t){
  const ss=series(t),anios=rango(t),[u,dec]=uni(t);
  const conFr=t==="c"&&st.cv==="m";       // municipalidades: franja con la fuente de cada periodo
  const W=720,H=conFr?248:230,ml=44,mr=8,mt=conFr?30:10,mb=24,w=(W-ml-mr)/anios.length;
  const tot=a=>ss.reduce((x,s)=>x+(s.v[a]||0),0);let vmax=Math.max(...anios.map(tot))||1;
  const paso=Math.pow(10,Math.floor(Math.log10(vmax)));const nice=[1,2,2.5,5,10].map(f=>f*paso).find(x=>vmax/x<=5)||paso*10;
  vmax=Math.ceil(vmax/nice)*nice;
  const y=v=>mt+(H-mt-mb)*(1-v/vmax);let g="";
  for(let v=0;v<=vmax+1e-9;v+=nice)g+=`<line x1="${ml}" x2="${W-mr}" y1="${y(v)}" y2="${y(v)}" stroke="#e8e6e0"/><text x="${ml-6}" y="${y(v)+3}" font-size="10" fill="#666" text-anchor="end">${num(v,v<10&&dec?1:0)}</text>`;
  if(t==="c"&&st.cv==="m"){const con=anios.map(a=>ss.some(s=>s.v[a]!=null));const i0=con.indexOf(true),i1=con.lastIndexOf(true);
    for(let i=i0+1;i<i1;i++){if(con[i])continue;let j=i;while(j+1<i1&&!con[j+1])j++;const x0=ml+i*w,ww=(j-i+1)*w;
      g+=`<rect x="${x0}" y="${mt}" width="${ww}" height="${H-mt-mb}" fill="#efede7"/>`+(ww>150?`<text x="${x0+ww/2}" y="${mt+16}" font-size="10.5" fill="#777" text-anchor="middle">Sin detalle por municipalidad (${anios[i]}–${anios[j]})</text><text x="${x0+ww/2}" y="${mt+30}" font-size="10" fill="#999" text-anchor="middle">la fuente no lo publica</text>`:ww>40?`<text x="${x0+ww/2}" y="${mt+16}" font-size="10" fill="#777" text-anchor="middle">Sin detalle</text>`:"");i=j;}}
  anios.forEach((a,i)=>{let base=0;const x=ml+i*w;
    ss.forEach(s=>{const v=s.v[a]||0;if(v<=0)return;const y1=y(base+v),y0=y(base);
      g+=`<rect class="b${a===st.anio?" sel":""}" data-a="${a}" x="${(x+1).toFixed(1)}" y="${y1.toFixed(1)}" width="${Math.max(w-2,1).toFixed(1)}" height="${Math.max(y0-y1,0.5).toFixed(1)}" fill="${s.color}" rx="1"/>`;base+=v;});
    g+=`<rect class="b" data-a="${a}" x="${x}" y="${mt}" width="${w}" height="${H-mt-mb}" fill="transparent"/>`;
    if(a%5===0||(i===0&&a%5<3)||(i===anios.length-1&&a%5>2))g+=`<text x="${x+w/2}" y="${H-8}" font-size="10" fill="#666" text-anchor="middle">${a}</text>`;});
  if(conFr){const tr=[];anios.forEach((a,i)=>{const con=ss.some(s=>s.v[a]!=null),f=con?fuenteCorta(fuenteMun(a)):"";
      const u=tr[tr.length-1];if(u&&u.f===f)u.i1=i;else tr.push({f,i0:i,i1:i});});
    tr.forEach((x,j)=>{if(!x.f)return;const x0=ml+x.i0*w,ww=(x.i1-x.i0+1)*w,c=COL_FUENTE[x.f]||"#8c96ad";
      g+=`<rect x="${(x0+1).toFixed(1)}" y="${mt-9}" width="${Math.max(ww-2,1).toFixed(1)}" height="4" fill="${c}" rx="1"><title>${esc(x.f)}</title></rect>`;
      const etq=ww>=x.f.length*5.2?x.f:ww>=34?x.f.split(" ")[0]:"";
      if(etq)g+=`<text x="${x0+ww/2}" y="${mt-14}" font-size="9.5" fill="#555" text-anchor="middle">${esc(etq)}</text>`;
      const prev=tr.slice(0,j).reverse().find(y=>y.f);
      if(prev&&prev.f!==x.f)g+=`<line x1="${x0}" x2="${x0}" y1="${mt-10}" y2="${H-mb}" stroke="#6b6b6b" stroke-width="1" stroke-dasharray="3,3"/>`;});}
  const ley=ss.length>1?`<div class="ley">${ss.map(s=>`<span><i style="background:${s.color}"></i>${s.nombre}</span>`).join("")}</div>`:"";
  const usadas=conFr?[...new Set(anios.filter(a=>ss.some(s=>s.v[a]!=null)).map(a=>fuenteCorta(fuenteMun(a))).filter(x=>x))]:[];
  const leyF=usadas.length?`<div class="ley" style="margin-top:2px"><span style="color:#666">Fuente (franja superior):</span>${usadas.map(f=>`<span><i style="background:${COL_FUENTE[f]||"#8c96ad"};height:4px;margin-top:5px"></i>${esc(f)}</span>`).join("")}</div>`:"";
  return {svg:`${ley}${leyF}<svg viewBox="0 0 ${W} ${H}" role="img" aria-label="Gráfico anual">${g}</svg>`,ss,tot,u,dec};
}
function detalle(t,a){
  if(t==="p"){const d=A.p[st.c]||{};const ls=Object.keys(d).filter(l=>(d[l].d||{})[a]||(d[l].e||{})[a]);
    if(!ls.length)return `<div class="nota">Sin pozos perforados en ${a}.</div>`;let td=0,te=0;
    const f=ls.map(l=>{const x=(d[l].d||{})[a]||0,e=(d[l].e||{})[a]||0;td+=x;te+=e;return `<tr><td>Lote ${l}</td><td>${num(x)}</td><td>${num(e)}</td><td>${num(x+e)}</td></tr>`;}).join("");
    return `<table><thead><tr><th>${a}</th><th>Desarrollo</th><th>Exploratorios y confirmatorios</th><th>Total</th></tr></thead><tbody>${f}</tbody>${ls.length>1?`<tfoot><tr><td>Total</td><td>${num(td)}</td><td>${num(te)}</td><td>${num(td+te)}</td></tr></tfoot>`:""}</table>`;}
  if(t==="r"){const d=A.r[st.c]||{};const hs=Object.keys(COL_R);const ls=Object.keys(d).filter(l=>hs.some(h=>(d[l][h]||{})[a]));
    if(!ls.length)return `<div class="nota">Sin regalías registradas en ${a}.</div>`;const usadas=hs.filter(h=>ls.some(l=>(d[l][h]||{})[a]));const tt={};
    const f=ls.map(l=>{let s=0;const c=usadas.map(h=>{const v=(d[l][h]||{})[a]||0;s+=v;tt[h]=(tt[h]||0)+v;return `<td>${v?num(v,2):"–"}</td>`;}).join("");return `<tr><td>Lote ${l}</td>${c}<td>${num(s,2)}</td></tr>`;}).join("");
    const T=usadas.reduce((x,h)=>x+tt[h],0);
    return `<table><thead><tr><th>${a} (millones de US$)</th>${usadas.map(h=>`<th>${titulo(h)}</th>`).join("")}<th>Total</th></tr></thead><tbody>${f}</tbody>${ls.length>1||usadas.length>1?`<tfoot><tr><td>Total</td>${usadas.map(h=>`<td>${num(tt[h],2)}</td>`).join("")}<td>${num(T,2)}</td></tr></tfoot>`:""}</table>${notaCanonRegalia(a)}`;}
  const nm=mesesNota(a),nota_m=nm?`<div class="nota">⚠️ Año incompleto. ${nm}</div>`:"";
  if(st.cv==="l"){const d=(A.cl||{})[st.c]||{};const deps=series("c").map(s=>s.k);const ls=Object.keys(d).filter(l=>deps.some(dp=>(d[l][dp]||{})[a]));
    if(!ls.length)return `<div class="nota">Sin canon registrado para este lote en ${a}.</div>`;const tt={};
    const f=ls.map(l=>{let s=0;const c=deps.map(dp=>{const v=(d[l][dp]||{})[a]||0;s+=v;tt[dp]=(tt[dp]||0)+v;return `<td>${v?num(v,2):"–"}</td>`;}).join("");return `<tr><td>Lote ${esc(l)}</td>${c}<td>${num(s,2)}</td></tr>`;}).join("");
    const T=deps.reduce((x,dp)=>x+(tt[dp]||0),0);
    return `<table><thead><tr><th>${a} (millones de US$)</th>${deps.map(dp=>`<th>${(A.cb||{})[dp]||(A.cbi||{})[dp]?`<button type="button" class="dep-b" data-dep="${esc(dp)}" aria-expanded="${st.depb===dp}" title="Ver cómo repartió ${titulo(dp)} su canon por beneficiario">Para ${titulo(dp)} ${st.depb===dp?"▴":"▾"}</button>`:"Para "+titulo(dp)}</th>`).join("")}<th>Total</th></tr></thead><tbody>${f}</tbody>${ls.length>1?`<tfoot><tr><td>Total</td>${deps.map(dp=>`<td>${num(tt[dp]||0,2)}</td>`).join("")}<td>${num(T,2)}</td></tr></tfoot>`:""}</table>${nota_m}
      <div class="nota">Canon y sobrecanon que generó la producción del lote en el mes de producción, en dólares, y la parte que le corresponde a cada departamento (PERUPETRO, «Canon Petrolero ${a}», determinación). No incluye los ajustes de meses anteriores. En soles, el departamento lo recibe al mes siguiente.${deps.some(dp=>(A.cb||{})[dp]||(A.cbi||{})[dp])?" Haz clic en un departamento para ver cómo repartió su canon por beneficiario.":""}</div>${notaCanonRegalia(a)}${st.depb&&deps.includes(st.depb)?benef(st.depb,a):""}`;}
  if(st.cv==="n"){const S=series("c").filter(s=>s.v[a]!=null);
    if(!S.length)return `<div class="nota">Sin datos por nivel de gobierno en ${a}${nf()==="mef"?" en el MEF (2003–2004 la consulta no tiene «Nivel de Gobierno»)":""}.</div>`;
    const T=S.reduce((x,s)=>x+s.v[a],0);
    return `<table><thead><tr><th>${a}</th><th>Millones de S/</th><th>Participación</th></tr></thead><tbody>${S.map(s=>`<tr><td>${s.nombre}</td><td>${num(s.v[a],2)}</td><td>${num(s.v[a]/T*100,1)} %</td></tr>`).join("")}</tbody><tfoot><tr><td>Total ${titulo(st.d)}</td><td>${num(T,2)}</td><td>100 %</td></tr></tfoot></table>
      <div class="nota">${nf()==="mef"?`Canon y sobrecanon petrolero <b>recaudado</b> en ${titulo(st.d)} por nivel de gobierno (MEF, Consulta Amigable de Ingresos: tipos «5» y «K» desde 2007, «4» en 2003–2006; no incluye el tipo NORMAL de 2005–2006, que reúne todos los canones). El gobierno nacional no se muestra: el MEF solo lo desagrega por departamento en 2007–2011, y mostrarlo esos años cortaría la serie. Desde 2012 el gobierno regional se toma del pliego «Gobierno Regional del Departamento de …».${st.d==="CUSCO"?" En Cusco se incluye el canon gasífero (Camisea), que es el grueso de lo que recibe.":""} El año en curso llega hasta el último mes que publica el MEF.`:
      `Canon y sobrecanon <b>transferido</b> a ${titulo(st.d)} por PERUPETRO («Canon por Año ${titulo(st.d)}», distribución), con los beneficiarios agrupados: gobierno regional, municipalidades (gobiernos locales) y universidades, institutos e IIAP, que se muestran juntos sin asignarles nivel de gobierno. Solo años completos.`}</div>`;}
  if(st.cv==="m"){const e=(A.cm||{})[st.c]||{v:{}};const ms=munis().filter(x=>(e.v[x[0]]||{})[a]!=null);
    if(!ms.length)return `<div class="nota">Sin datos por municipalidad en ${a}. PERUPETRO publica el reparto por municipalidad solo del año en curso (el notebook lo acumula desde 2025) y, en el Excel de Distribución, el de 1993–2002.</div>${nota_m}`;
    let T=0;const f=ms.map(x=>{const v=e.v[x[0]][a];T+=v;return `<tr><td style="text-align:left">${esc(x[1])}</td><td>${x[3]==="PROVINCIAL"?"Provincial":"Distrital"}</td><td style="text-align:left">${esc(x[2])}</td><td>${x[4]!=null?num(x[4],x[4]<10?1:0)+" %":"–"}</td><td>${num(v,2)}</td></tr>`;}).join("");
    return `<table><thead><tr><th>Municipalidad (${a})</th><th>Nivel</th><th style="text-align:left">Provincia</th><th>% del área del lote</th><th>Millones de S/</th></tr></thead><tbody>${f}</tbody>${ms.length>1?`<tfoot><tr><td>Total</td><td></td><td></td><td></td><td>${num(T,2)}</td></tr></tfoot>`:""}</table>${nota_m}
      <div class="nota">Canon y sobrecanon transferido a las municipalidades de los distritos donde está el lote y a la municipalidad provincial de su provincia. Fuente ${a}: ${esc(fuenteMun(a)||"–")}. Es lo que recibe cada municipalidad por todos los lotes del departamento, no solo por este.</div>`;}
  const cb=(A.cb||{})[st.d];
  if(cb){const ts=Object.keys(COL_B).filter(t=>(cb[t]||{})[a]!=null);if(!ts.length)return `<div class="nota">Sin datos en ${a}.</div>`;
    const T=ts.reduce((x,t)=>x+cb[t][a],0);
    return `<table><thead><tr><th>${a}</th><th>Millones de S/</th><th>Participación</th></tr></thead><tbody>${ts.map(t=>`<tr><td>${NOM_B[t]}</td><td>${num(cb[t][a],2)}</td><td>${num(cb[t][a]/T*100,1)} %</td></tr>`).join("")}</tbody><tfoot><tr><td>Total ${titulo(st.d)}</td><td>${num(T,2)}</td><td>100 %</td></tr></tfoot></table>
      <div class="nota">Canon y sobrecanon que recibió el departamento entero (por todos sus lotes), por beneficiario (PERUPETRO, «Canon por Año ${titulo(st.d)}», distribución). Solo años completos.${ts.includes("LEY 29289")?" «Descuento: pago de deuda (Ley 29289)», en gris, no es un beneficiario: es la parte del canon que se descontó antes del reparto para «Contrapartida nacional» y «Servicio de deuda» (Ley de Presupuesto 2009); figura en Loreto de 2009 a 2023.":""}</div>`;}
  const c=A.c[st.d]||{};const v=c[a];const prev=c[a-1];
  return `<table><thead><tr><th>${a}</th><th>Millones de S/</th><th>Variación vs ${a-1}</th></tr></thead><tbody><tr><td>${titulo(st.d)}</td><td>${v!=null?num(v,2):"–"}</td><td>${v!=null&&prev?num((v/prev-1)*100,1)+" %":"–"}</td></tr></tbody></table>
    <div class="nota">El canon y sobrecanon es la parte del valor de la producción que el Estado transfiere al departamento (gobierno regional, municipios y universidades). PERUPETRO lo publica solo por departamento: es del departamento entero, no solo de este lote.</div>`;
}
function notaCanonRegalia(a){  // por qué el canon de un lote puede superar a sus regalías
  const cl=(A.cl||{})[st.c];
  const deps=cl?[...new Set(Object.values(cl).flatMap(x=>Object.keys(x)))]:[];
  const lu=deps.includes("LORETO")||deps.includes("UCAYALI");
  return `<div class="nota">ℹ️ <b>Canon y regalía son cosas distintas.</b> La regalía es lo que la empresa paga al Estado según su contrato. El canon y sobrecanon petrolero no es una parte de la regalía: es un porcentaje del valor de la producción (barriles fiscalizados × canasta, menos flete, oleoducto, comercialización y fiscalización)${lu?"; en la zona Loreto / Ucayali es el 18,75 % de ese valor: 15 % para Loreto y 3,75 % para Ucayali (PERUPETRO, determinación)":""}. Por eso el canon de un lote puede superar a sus regalías.</div>`;}
// nombre corto y color de cada fuente del canon municipal (franja del gráfico)
const COL_FUENTE={"PERUPETRO · Distribución":"#8c96ad","PERUPETRO · Canon Municipalidades":"#5a5a9e","MEF":"#2a9da1",
  "MEF + NORMAL (todos los canones)":"#ffc41d","MEF · canon gasífero":"#181818"};
function fuenteCorta(f){if(!f)return "";const p=[];
  if(/Distribuci/i.test(f))p.push("PERUPETRO · Distribución");if(/Canon Municipalidades/i.test(f))p.push("PERUPETRO · Canon Municipalidades");
  if(/NORMAL/.test(f))p.push("MEF + NORMAL (todos los canones)");else if(/gasífero/i.test(f))p.push("MEF · canon gasífero");else if(/MEF/.test(f))p.push("MEF");
  return p.join(" + ");}
function fuenteMun(a){const d=(munis()[0]||[""])[0].split("|")[0];return ((A.cmf||{})[d]||{})[a]||"";}
function fuentesMun(){        // «1993–2002: fuente A · 2025: fuente B · 2003–2024: sin detalle por municipalidad»
  const e=(A.cm||{})[st.c]||{v:{}},an=rango("c"),con=a=>munis().some(x=>(e.v[x[0]]||{})[a]!=null);
  const tr=[];an.forEach(a=>{const f=con(a)?fuenteMun(a)||"sin fuente":"";const u=tr[tr.length-1];if(u&&u.f===f&&u.b===a-1)u.b=a;else tr.push({f,a,b:a});});
  const ini=an.find(con),fin=[...an].reverse().find(con);
  return tr.filter(t=>t.f||(t.a>ini&&t.b<fin)).map(t=>`${t.a===t.b?t.a:t.a+"–"+t.b}: ${t.f||(t.a>=2003&&t.b<=2006?"sin detalle por municipalidad (PERUPETRO entregaba el total municipal al Consejo Nacional de Descentralización y el MEF lo registra a nombre del CND, sin reparto)":"sin detalle por municipalidad (la fuente no lo publica)")}`).join(" · ");}
function benef(dp,a){         // reparto por beneficiario del departamento en el año (Excel de Distribución)
  const n=((A.cbn||{})[dp]||{})[a],src=n?(A.cbi||{})[dp]:(A.cb||{})[dp];
  const ts=src?Object.keys(COL_B).filter(t=>(src[t]||{})[a]!=null):[];
  if(!ts.length)return `<div class="benef"><b>${titulo(dp)} · ${a}</b><div class="nota">Sin datos de distribución por beneficiario en ${a}.</div></div>`;
  const T=ts.reduce((x,t)=>x+src[t][a],0);
  return `<div class="benef"><b>Canon y sobrecanon recibido por ${titulo(dp)} en ${a}, por beneficiario</b>
    <table><thead><tr><th>Beneficiario</th><th>Millones de S/</th><th>Participación</th></tr></thead><tbody>${ts.map(t=>`<tr><td>${NOM_B[t]}</td><td>${num(src[t][a],2)}</td><td>${num(src[t][a]/T*100,1)} %</td></tr>`).join("")}</tbody>
    <tfoot><tr><td>Total ${titulo(dp)}</td><td>${num(T,2)}</td><td>100 %</td></tr></tfoot></table>
    ${n?`<div class="nota">⚠️ Año incompleto: ${n} meses con dato.</div>`:""}
    <div class="nota">Es lo que recibió el departamento entero por todos sus lotes, en soles y por mes de pago (PERUPETRO, «Canon por Año ${titulo(dp)}», distribución); no es solo la parte de este lote, que está arriba en dólares.</div></div>`;}
function tituloVista(t){     // qué muestra la vista: lote, departamento y unidad
  const lote=st.l?"Lote "+st.l:"campo "+st.c,dep=st.d?titulo(st.d):"";
  if(t==="p")return `Pozos perforados en el ${lote} por año`;
  if(t==="r")return `Regalías pagadas por el ${lote} por año (millones de US$)`;
  if(st.cv==="n")return `${dep}: canon y sobrecanon transferido por nivel de gobierno (PERUPETRO, beneficiarios agrupados) (millones de S/)`;
  if(st.cv==="l")return `${lote}: canon y sobrecanon generado, por departamento que lo recibe (millones de US$)`;
  if(st.cv==="m"){const ds=[...new Set(munis().map(x=>x[0].split("|")[0]))].map(titulo),ps=[...new Set(munis().map(x=>x[2]))];
    return `Municipalidades donde está el ${lote}${ps.length?" (provincia"+(ps.length>1?"s":"")+" de "+ps.join(", ")+", "+(ds.join(" y ")||dep)+")":""}: canon y sobrecanon recibido (millones de S/)`;}
  return `${dep}: canon y sobrecanon recibido${(A.cb||{})[st.d]?" por beneficiario":""} (millones de S/)`;}
function resumen(t,G){const an=rango(t);
  const tot=an.map(G.tot);const s=tot.reduce((x,v)=>x+v,0);
  const conDato=an.filter((a,i)=>tot[i]>0);if(!conDato.length)return "Sin datos.";
  const imax=tot.indexOf(Math.max(...tot));
  return `Total ${an[0]}–${an[an.length-1]}: <b>${num(s,t==="p"?0:1)} ${G.u}</b> · máximo: <b>${num(tot[imax],t==="p"?0:1)}</b> en ${an[imax]} · último año con dato: ${conDato[conDato.length-1]}`;}
function filasTabla(t){       // [cabecera, filas] de la tabla completa por año (más reciente arriba)
  if(t==="e")return [["Lote","Operador","Desde","Hasta","Años con producción","Líquidos (bbl)","Gas natural (MPC)","Regalías (millones de US$)"],
    (E().o||[]).slice().sort((a,b)=>b[3]-a[3]||b[2]-a[2]).map(x=>[esc(x[0]),esc(x[1]),x[2],x[3],x[3]-x[2]+1,num(x[4]),num(x[5]),x[6]?num(x[6],2):"–"])];
  const an=rango(t).slice().reverse();
  if(t==="p"){const S=series("p");return [["Año","Desarrollo","Exploratorios y confirmatorios","Total"],
    an.map(a=>{const d=S[0].v[a]||0,e=S[1].v[a]||0;return [a,num(d),num(e),num(d+e)];})];}
  if(t==="r"){const S=series("r");return [["Año"].concat(S.map(s=>s.nombre+" (millones de US$)"),["Total (millones de US$)"]),
    an.map(a=>{let x=0;const c=S.map(s=>{const v=s.v[a]||0;x+=v;return v?num(v,2):"–";});return [a].concat(c,[x?num(x,2):"–"]);})];}
  if(st.cv!=="b"||(A.cb||{})[st.d]){const S=series("c"),[u]=uni("c");
    return [["Año"].concat(S.map(s=>s.nombre+" ("+u+")"),["Total ("+u+")"]),
      an.map(a=>{let x=0,h=false;const c=S.map(s=>{const v=s.v[a];if(v!=null){x+=v;h=true;}return v!=null?num(v,2):"–";});return [a].concat(c,[h?num(x,2):"–"]);})];}
  const c=A.c[st.d]||{};
  return [["Año","Canon y sobrecanon de "+titulo(st.d)+" (millones de S/)","Variación vs año anterior"],
    an.map(a=>[a,c[a]!=null?num(c[a],2):"–",c[a]!=null&&c[a-1]?num((c[a]/c[a-1]-1)*100,1)+" %":"–"])];
}
function tablaHTML(t){const [cab,fs]=filasTabla(t);
  return `<div class="tabla-wrap"><table><thead><tr>${cab.map(c=>`<th>${c}</th>`).join("")}</tr></thead><tbody>${fs.map(f=>`<tr>${f.map(x=>`<td>${x}</td>`).join("")}</tr>`).join("")}</tbody></table></div>`;}
function csv(t){              // datos detallados por lote (formato largo), separador «;» y coma decimal
  const dc=v=>String(Math.round(v*1e6)/1e6).replace(".",",");let cab,fs=[];
  if(t==="p"){cab=["AÑO","CAMPO","LOTE","DESARROLLO","EXPLORATORIOS_Y_CONFIRMATORIOS"];const d=A.p[st.c]||{};
    rango("p").forEach(a=>{for(const l in d){const x=(d[l].d||{})[a]||0,e=(d[l].e||{})[a]||0;if(x||e)fs.push([a,st.c,l,x,e]);}});}
  else if(t==="r"){cab=["AÑO","CAMPO","LOTE","HIDROCARBURO","REGALIA_MM_US$"];const d=A.r[st.c]||{};
    rango("r").forEach(a=>{for(const l in d)for(const h in d[l]){const v=d[l][h][a];if(v)fs.push([a,st.c,l,h,dc(v)]);}});}
  else if(t==="e"){cab=["TIPO","CAMPO","LOTE","EMPRESA","PARTICIPACION_PCT","DESDE","HASTA","LIQUIDOS_BBL","GAS_MPC","REGALIAS_MM_US$","FASE","FUENTE"];const e=E();
    (e.o||[]).forEach(x=>fs.push(["OPERADOR CON PRODUCCION",st.c,x[0],x[1],"",x[2],x[3],x[4],x[5],x[6]?dc(x[6]):"","","PERUPETRO, produccion fiscalizada y regalias"]));
    (e.v||[]).forEach(x=>fs.push(["REGALIAS SIN OPERADOR IDENTIFICADO",st.c,x[0],"","",x[1],x[1],"","",dc(x[2]),"","PERUPETRO, regalias (el PDF no distingue los contratos)"]));
    (e.s||[]).forEach(x=>fs.push(["SOCIO DEL CONTRATO VIGENTE",st.c,x[0],x[1],x[2]!=null?dc(x[2]):"",x[3],x[4],"","","",x[5],"PERUPETRO, contratos vigentes al 31/12/"+A.ed]));
    (e.t||[]).forEach(x=>fs.push(["CONTRATO TERMINADO",st.c,x[0],x[1],"",x[2],x[3],"","","","","PERUPETRO, contratos terminados "+x[4]]));
    fs=fs.map(f=>f.map(v=>String(v).includes(";")?'"'+String(v).replace(/"/g,'""')+'"':v));}
  else if(st.cv==="l"){cab=["AÑO","CAMPO","LOTE","DEPARTAMENTO","CANON_GENERADO_MM_US$","MESES_CON_DATO"];const d=(A.cl||{})[st.c]||{};
    rango("c").forEach(a=>{for(const l in d)for(const dp in d[l]){const v=d[l][dp][a];if(v)fs.push([a,st.c,l,dp,dc(v),(A.cln||{})[a]||""]);}});}
  else if(st.cv==="m"){cab=["AÑO","CAMPO","MUNICIPALIDAD","NIVEL","PROVINCIA","PCT_AREA_LOTE","CANON_MM_S/","MESES_CON_DATO","FUENTE"];const e=(A.cm||{})[st.c]||{v:{}};
    rango("c").forEach(a=>munis().forEach(x=>{const v=(e.v[x[0]]||{})[a];if(v!=null)fs.push([a,st.c,x[1],x[3],x[2],x[4]!=null?dc(x[4]):"",dc(v),((A.cmn||{})[x[0].split("|")[0]]||{})[a]||"",'"'+fuenteMun(a).replace(/"/g,'""')+'"']);}));}
  else if(st.cv==="n"){cab=["AÑO","DEPARTAMENTO","NIVEL_GOBIERNO","CANON_MM_S/","FUENTE"];
    const fu=nf()==="mef"?"MEF, Consulta Amigable de Ingresos (recaudado)":"PERUPETRO, distribución por beneficiario (agrupada)";
    rango("c").forEach(a=>series("c").forEach(s=>{if(s.v[a]!=null)fs.push([a,st.d,s.nombre,dc(s.v[a]),fu]);}));}
  else if((A.cb||{})[st.d]){cab=["AÑO","DEPARTAMENTO","BENEFICIARIO","CANON_MM_S/"];const cb=A.cb[st.d];
    rango("c").forEach(a=>{for(const t in cb)if(cb[t][a]!=null)fs.push([a,st.d,t,dc(cb[t][a])]);});}
  else{cab=["AÑO","DEPARTAMENTO","CANON_SOBRECANON_MM_S/"];const c=A.c[st.d]||{};
    rango("c").forEach(a=>{if(c[a]!=null)fs.push([a,st.d,dc(c[a])]);});}
  const s="\ufeff"+[cab].concat(fs).map(f=>f.join(";")).join("\n")+"\n";
  const nom=(t==="c"?(st.cv==="l"?"canon_generado_campo_"+st.c:st.cv==="n"?"canon_nivel_gobierno_"+st.d+"_"+nf():st.cv==="m"?"canon_municipalidades_campo_"+st.c:"canon_sobrecanon_"+st.d):{p:"pozos_perforados",r:"regalias",e:"empresas"}[t]+"_campo_"+st.c).replace(/[^\w-]+/g,"_")+".csv";
  const a=document.createElement("a");a.href=URL.createObjectURL(new Blob([s],{type:"text/csv"}));a.download=nom;
  document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(a.href),1000);
}
function tieneCtx(o,t){return t==="pp"?!!(o.pp&&window.ppAbrir):t==="p"?!!A.p[o.c]:t==="r"?!!A.r[o.c]:t==="e"?!!(A.e||{})[o.c]:VISTAS.some(v=>hayVista(v[0],o));}
function tieneDatos(t){return tieneCtx(st,t);}
// pestañas para el panel de producción (mismo lote): [clave, nombre, con datos]
window.ppActTabs=function(o){return TABS.filter(x=>x[0]!=="pp").map(([k,n])=>[k,n,tieneCtx(o,k)]);};
function irProduccion(){const o={c:st.c,l:st.l,d:st.d,pp:st.pp,n:st.n};cerrar();window.ppAbrir(o.pp,null,o);}
function pintar(){
  const t=st.t,tabs=caja.querySelector(".tabs");
  tabs.innerHTML=TABS.map(([k,n])=>`<button type="button" role="tab" data-t="${k}" aria-selected="${k===t}" ${tieneDatos(k)?"":"disabled"}>${n}</button>`).join("");
  tabs.querySelectorAll("button").forEach(b=>b.onclick=()=>{if(b.dataset.t==="pp"){irProduccion();return;}st.t=b.dataset.t;st.anio=null;pintar();});

  if(t==="e"){pintarE();return;}
  if(t==="c"&&!hayVista(st.cv))st.cv=VISTAS.map(v=>v[0]).find(v=>hayVista(v));
  const G=grafico(t);const an=rango(t);
  if(st.anio==null){const conDato=an.filter(a=>G.tot(a)>0);st.anio=conDato.length?conDato[conDato.length-1]:an[an.length-1];}
  const G2=grafico(t);
  const AT=rangoTotal(t),an0=rango(t);
  const opc=(sel)=>AT.map(a=>`<option value="${a}"${a===sel?" selected":""}>${a}</option>`).join("");
  const herr=`<div class="herr"><label class="rg">Desde <select class="s-desde">${opc(an0[0])}</select></label><label class="rg">Hasta <select class="s-hasta">${opc(an0[an0.length-1])}</select></label>${st.desde||st.hasta?`<button type="button" class="b-todo" title="Ver toda la serie">↺ Toda la serie</button>`:""}<button type="button" class="b-tabla">${st.tabla?"📈 Gráfico":"📋 Tabla"}</button><button type="button" class="b-csv" title="Descargar los datos por lote en CSV (Excel)">⬇ Descargar CSV</button>${st.tabla?"":`<button type="button" class="b-img" title="Guardar el gráfico como imagen (PNG)">🖼 Imagen</button>`}</div>`;
  const NV=Object.fromEntries(VISTAS);
  const vistas=(t==="c"?`<div class="vistas">${gruposVistas().map(([g,ks])=>{const b=ks.filter(k=>hayVista(k));
      return b.length?`<div class="grupo"><span>${esc(g)}</span>${b.map(k=>`<button type="button" data-v="${k}" aria-pressed="${k===st.cv}">${NV[k]}</button>`).join("")}</div>`:"";}).join("")}</div>`:"")
;
  caja.querySelector(".cuerpo").innerHTML=`${vistas}${herr}<div class="tit-v">${esc(tituloVista(t))}</div>${t==="c"?`<div class="alcance">${alcance()}</div>`:""}<div class="res">${resumen(t,G2)}</div>`+(st.tabla?tablaHTML(t):
    `${G2.svg}${t==="c"&&st.cv==="m"?`<div class="nota fuentes"><b>Fuente por periodo</b><ul>${fuentesMun().split(" · ").map(x=>{const i=x.indexOf(": ");return `<li><b>${esc(x.slice(0,i))}</b>: ${esc(x.slice(i+2))}</li>`;}).join("")}</ul></div>`:""}<div class="nota">Pasa el cursor por una barra para ver el valor; haz clic en ella o elige el año abajo para ver el detalle.</div>
    <label class="sel-anio">Año de la tabla: <select class="s-anio">${an.slice().reverse().map(a=>`<option value="${a}"${a===st.anio?" selected":""}>${a}${G2.tot(a)>0?"":" (sin dato)"}</option>`).join("")}</select></label>
    <div class="det">${detalle(t,st.anio)}</div>`);
  caja.querySelectorAll(".vistas button[data-v]").forEach(b=>b.onclick=()=>{st.cv=b.dataset.v;st.anio=null;st.depb=null;pintar();});
  caja.querySelectorAll(".vistas button[data-nf]").forEach(b=>b.onclick=()=>{st.nf=b.dataset.nf;st.anio=null;pintar();});
  caja.querySelector(".b-tabla").onclick=()=>{st.tabla=!st.tabla;pintar();};
  caja.querySelector(".b-csv").onclick=()=>csv(t);
  const bi=caja.querySelector(".b-img");
  if(bi)bi.onclick=()=>{const cu=caja.querySelector(".cuerpo"),al=cu.querySelector(".alcance"),fu=cu.querySelector(".nota.fuentes");
    const nt=[...cu.querySelectorAll(".det .nota")].map(n=>n.textContent).filter(x=>/Fuente|PERUPETRO|MEF/.test(x)).slice(0,1);
    window.ppImagen(cu.querySelector("svg"),{titulo:tituloVista(t),sub:(al?al.textContent+" ":"")+cu.querySelector(".res").textContent,
      ley:[...cu.querySelectorAll(".ley")].flatMap(l=>window.ppLeyendaDe(l)),
      nota:[fu?"Fuente por periodo: "+[...fu.querySelectorAll("li")].map(li=>li.textContent).join(" · ")+".":"",...nt].filter(x=>x).join(" "),
      archivo:(st.n||st.c||st.d||"")+"_"+tituloVista(t).split(":")[0]+"_"+(TABS.find(x=>x[0]===t)||["",""])[1]});};
  const sd=caja.querySelector(".s-desde"),sh=caja.querySelector(".s-hasta"),bt=caja.querySelector(".b-todo");
  if(sd)sd.onchange=()=>{st.desde=+sd.value;if(st.hasta&&st.hasta<st.desde)st.hasta=st.desde;st.anio=null;pintar();};
  if(sh)sh.onchange=()=>{st.hasta=+sh.value;if(st.desde&&st.desde>st.hasta)st.desde=st.hasta;st.anio=null;pintar();};
  if(bt)bt.onclick=()=>{st.desde=st.hasta=null;st.anio=null;pintar();};
  const sa=caja.querySelector(".s-anio");if(sa)sa.onchange=()=>{st.anio=+sa.value;pintar();caja.querySelector(".s-anio").focus();};
  caja.querySelectorAll(".dep-b").forEach(b=>b.onclick=()=>{st.depb=st.depb===b.dataset.dep?null:b.dataset.dep;pintar();});
  const svg=caja.querySelector("svg");if(!svg)return;
  svg.addEventListener("mousemove",ev=>{const r=ev.target.closest&&ev.target.closest(".b");if(!r){tip.style.display="none";return;}
    const a=+r.dataset.a;const partes=G2.ss.filter(s=>s.v[a]).map(s=>`${G2.ss.length>1?s.nombre+": ":""}${num(s.v[a],G2.dec)}`);
    tip.innerHTML=`<b>${a}</b>${partes.length?"<br>"+partes.join("<br>"):"<br>sin dato"}${G2.ss.length>1&&partes.length>1?`<br>Total: ${num(G2.tot(a),G2.dec)}`:""} ${G2.u}${t==="c"&&st.cv==="m"&&partes.length&&fuenteMun(a)?`<br><span style="color:#ccc">Fuente: ${esc(fuenteMun(a))}</span>`:""}`;
    tip.style.display="block";tip.style.left=Math.min(ev.clientX+12,innerWidth-180)+"px";tip.style.top=(ev.clientY+12)+"px";});
  svg.addEventListener("mouseleave",()=>tip.style.display="none");
  svg.addEventListener("click",ev=>{const r=ev.target.closest&&ev.target.closest(".b");if(!r)return;st.anio=+r.dataset.a;pintar();});
}
function pintarE(){const fs=filasE();
  const herr=`<div class="herr"><button type="button" class="b-tabla">${st.tabla?"📈 Línea de tiempo":"📋 Tabla"}</button><button type="button" class="b-csv" title="Descargar operadores, socios y contratos en CSV (Excel)">⬇ Descargar CSV</button>${st.tabla||!fs.length?"":`<button type="button" class="b-img" title="Guardar la línea de tiempo como imagen (PNG)">🖼 Imagen</button>`}</div>`;
  caja.querySelector(".cuerpo").innerHTML=`${herr}<div class="res">${resumenE()}</div>`+(st.tabla?tablaHTML("e"):
    (fs.length?`${gantt()}<div class="nota">Cada barra: años con producción fiscalizada a nombre de ese operador (un año sin producción corta la barra). Pasa el cursor para ver la producción y las regalías de cada empresa (en la pestaña «Regalías» van sumadas por lote); haz clic en una barra para ver su producción.</div>`:"")
    +((E().v||[]).length?`<div class="nota">Regalías que el PDF no permite atribuir a un operador (dos contratos del mismo lote sin distinción): ${E().v.map(x=>`lote ${esc(x[0])}, ${x[1]}: ${num(x[2],2)} MM US$`).join("; ")}.</div>`:""))+tablaSocios();
  caja.querySelector(".b-tabla").onclick=()=>{st.tabla=!st.tabla;pintar();};
  caja.querySelector(".b-csv").onclick=()=>csv("e");
  const bi=caja.querySelector(".b-img");
  if(bi)bi.onclick=()=>window.ppImagen(caja.querySelector(".cuerpo svg"),{titulo:(st.n||"Campo "+st.c)+": operadores con producción por año",
    sub:caja.querySelector(".cuerpo .res").textContent,nota:"Fuente: PERUPETRO, producción fiscalizada y regalías; contratos.",archivo:(st.n||st.c)+"_empresas"});
  const svg=caja.querySelector("svg");if(!svg)return;
  svg.addEventListener("mousemove",ev=>{const r=ev.target.closest&&ev.target.closest(".b");if(!r){tip.style.display="none";return;}
    const [j,n]=r.dataset.e.split("|").map(Number),f=fs[j],x=f.tr[n];
    tip.innerHTML=`<b>Lote ${esc(f.l)}</b> · ${esc(f.op)}<br>${x[2]}${x[3]!==x[2]?"–"+x[3]:""} (${x[3]-x[2]+1} año${x[3]>x[2]?"s":""})`+
      (x[4]?`<br>Líquidos: ${num(x[4])} bbl`:"")+(x[5]?`<br>Gas natural: ${num(x[5])} MPC`:"")+(x[6]?`<br>Regalías: ${num(x[6],2)} millones de US$`:"");
    tip.style.display="block";tip.style.left=Math.min(ev.clientX+12,innerWidth-240)+"px";tip.style.top=(ev.clientY+12)+"px";});
  svg.addEventListener("mouseleave",()=>tip.style.display="none");
  // clic en una barra: producción de ese lote en los años de esa empresa
  svg.addEventListener("click",ev=>{const r=ev.target.closest&&ev.target.closest(".b");if(!r||!window.ppAbrir)return;
    const [j,n]=r.dataset.e.split("|").map(Number),f=fs[j],x=f.tr[n];
    const id=(window.ppTieneSerie&&window.ppTieneSerie("L:"+f.l))?"L:"+f.l:st.pp;if(!id)return;
    const o={c:st.c,l:st.l,d:st.d,pp:st.pp,n:st.n};cerrar();window.ppAbrir(id,null,o,[x[2],x[3]]);});}
function abrir(o){st={c:o.c,l:o.l,d:o.d,pp:o.pp,n:o.n,t:o.t,anio:null,tabla:false,cv:null};
  st.cv=VISTAS.map(v=>v[0]).find(v=>hayVista(v))||"b";
  if(!st.t||st.t==="pp"||!tieneDatos(st.t))st.t=TABS.map(x=>x[0]).find(k=>k!=="pp"&&tieneDatos(k))||"p";
  document.getElementById("pp-act-tit").textContent=o.n||(o.l?"Lote "+o.l:titulo(o.d||""));
  const lotes=[...new Set(Object.keys(Object.assign({},A.p[o.c]||{},A.r[o.c]||{})).concat(((A.e||{})[o.c]||{}).o?((A.e||{})[o.c].o.map(x=>String(x[0]))):[]))];
  document.getElementById("pp-act-sub").textContent=(o.c?"Campo "+o.c+(lotes.length>1?" (lotes "+lotes.join(", ")+")":"")+(o.d?" · ":""):"Departamento: ")+(o.d?titulo(o.d):"");
  caja.style.display="block";fondo.style.display="block";pintar();caja.querySelector(".x").focus();}
function cerrar(){caja.style.display="none";fondo.style.display="none";tip.style.display="none";}
caja.querySelector(".x").onclick=cerrar;fondo.onclick=cerrar;
document.addEventListener("keydown",e=>{if(e.key==="Escape"&&caja.style.display==="block")cerrar();});
document.addEventListener("click",ev=>{const b=ev.target.closest&&ev.target.closest("[data-act]");if(!b||caja.contains(b))return;
  ev.preventDefault();try{abrir(JSON.parse(b.dataset.act));}catch(e){}});
window.ppActAbrir=abrir;
})();
</script>
'''
COMPARAR_HTML = r'''<style>
/* «Comparar lotes»: barras apiladas por año con la tabla de datos debajo (formato de cuadro estadístico) */
#cmp-fondo{position:fixed;inset:0;background:rgba(0,0,0,.35);z-index:10000;display:none}
#cmp{position:fixed;left:50%;top:50%;transform:translate(-50%,-50%);z-index:10001;display:none;background:#fff;border-radius:10px;
  box-shadow:0 4px 24px rgba(0,0,0,.35);width:min(1100px,calc(100vw - 24px));max-height:calc(100vh - 24px);overflow:auto;
  font:12px/1.4 "Barlow",system-ui,-apple-system,"Segoe UI",Roboto,sans-serif;color:#1a1a1a}
#cmp *:not(i){font-family:"Barlow",system-ui,-apple-system,"Segoe UI",Roboto,sans-serif}
#cmp .cab{display:flex;align-items:center;justify-content:space-between;gap:8px;padding:10px 14px 8px;border-bottom:1px solid #eee}
#cmp .cab b{font-size:15px}
#cmp .x{border:0;background:#f3f2ee;border-radius:6px;width:28px;height:28px;cursor:pointer;font:600 16px sans-serif}
#cmp .cuerpo{padding:8px 14px 14px}
#cmp .ctl{display:flex;flex-wrap:wrap;gap:6px 10px;align-items:flex-end}
#cmp .ctl label{display:flex;flex-direction:column;font-size:11px;color:#52514e;gap:2px}
#cmp select,#cmp input[type=search]{font:12px sans-serif;padding:3px 4px;border:1px solid #c9c8c2;border-radius:5px;background:#fff;color:#181818}
#cmp .bt{border:1px solid #c9c8c2;background:#f3f2ee;border-radius:5px;padding:4px 9px;cursor:pointer;font:600 12px sans-serif;color:#181818}
#cmp .bt:hover{background:#e7e6e0}
#cmp .lotes{display:flex;flex-wrap:wrap;gap:5px;align-items:center;margin:8px 0 4px}
#cmp .chip{display:inline-flex;align-items:center;gap:4px;border:1px solid #c9c8c2;border-radius:14px;padding:2px 4px 2px 8px;background:#f7f6f2;font:600 12px sans-serif}
#cmp .chip i{display:inline-block;width:10px;height:10px;border-radius:2px;border:1px solid #111149}
#cmp .chip button{border:0;background:none;cursor:pointer;font:700 13px sans-serif;color:#777;padding:0 3px}
#cmp .lista{border:1px solid #e3e1db;border-radius:8px;padding:8px;margin:4px 0 8px;background:#fbfaf8}
#cmp .lista .grs{display:flex;flex-wrap:wrap;gap:8px 18px;max-height:230px;overflow:auto;margin-top:6px}
#cmp .lista .gr{min-width:150px}
#cmp .lista .gr>b{display:block;font-size:11px;color:#6b6b6b;text-transform:uppercase;letter-spacing:.03em;margin-bottom:2px}
#cmp .lista label{display:block;white-space:nowrap;cursor:pointer}
#cmp .lista label.no{color:#aaa;cursor:default}
#cmp .ref{position:relative;border:1px dashed #c9c8c2;border-radius:6px;background:#fbfaf8;padding:6px 120px 6px 10px;margin:6px 0}
#cmp .ref .cp{position:absolute;right:6px;top:6px}
#cmp .r-tit{font:700 15px sans-serif;color:#000;text-align:center}
#cmp .r-uni{font:14px sans-serif;color:#000;text-align:center}
#cmp .r-notas{font:11px sans-serif;color:#111149}
#cmp .ref::before{content:"Referencia (no va en la imagen)";display:block;font:600 10px sans-serif;color:#8a8984;text-transform:uppercase;letter-spacing:.03em}
#cmp .graf{overflow-x:auto}
#cmp .graf svg{display:block;width:100%;height:auto}
#cmp .nota{color:#666;margin-top:6px}
#cmp .vacio{color:#777;padding:30px 0;text-align:center}
#cmp .tip{position:fixed;pointer-events:none;background:#222;color:#fff;border-radius:5px;padding:5px 8px;font-size:11px;display:none;z-index:10002;white-space:nowrap}
#cmp-abrir{display:block;width:100%;margin:4px 0 0;text-align:left;white-space:normal;border:1px solid #c9c8c2;background:#f3f2ee;border-radius:5px;
  padding:4px 7px;cursor:pointer;font:600 12px "Barlow",system-ui,sans-serif;color:#181818}
#cmp-abrir.flota{position:fixed;left:10px;bottom:56px;z-index:9999;width:auto}
</style>
<button id="cmp-abrir" type="button" class="flota" title="Comparar varios lotes: producción, canon y su distribución">📊 Comparar lotes</button>
<div id="cmp-fondo"></div>
<div id="cmp" role="dialog" aria-modal="true" aria-labelledby="cmp-tit">
  <div class="cab"><b id="cmp-tit">Comparar lotes</b><button type="button" class="x" title="Cerrar (Esc)" aria-label="Cerrar">×</button></div>
  <div class="cuerpo">
    <div class="ctl">
      <label>Variable<select id="cmp-var"><option value="p">Producción fiscalizada</option><option value="c">Canon y sobrecanon generado</option>
        <option value="d">Distribución del canon (departamento que lo recibe)</option></select></label>
      <label class="solo-p">Producto<select id="cmp-prod"><option value="T">Líquidos totales</option><option value="P">Petróleo</option>
        <option value="L">LGN y condensado</option><option value="G">Gas natural</option></select></label>
      <label class="solo-p">Unidad<select id="cmp-u"><option value="m">Millones de barriles</option><option value="k">Miles de barriles</option><option value="d">Barriles por día</option></select></label>
      <label>Apilar por<select id="cmp-modo"></select></label>
      <label>Desde<select id="cmp-desde"></select></label>
      <label>Hasta<select id="cmp-hasta"></select></label>
      <button type="button" class="bt" id="cmp-csv" title="Descargar el cuadro en CSV (Excel)">⬇ CSV</button>
      <button type="button" class="bt" id="cmp-img" title="Guardar el gráfico con su cuadro como imagen PNG (alta resolución, 4×)">🖼 Imagen</button>
      <button type="button" class="bt" id="cmp-svg" title="Guardar el gráfico con su cuadro en SVG (vectorial: nítido a cualquier tamaño en PowerPoint)">⬇ SVG</button>
    </div>
    <div class="lotes"><span class="chips"></span><button type="button" class="bt" id="cmp-mas" aria-expanded="false">＋ Agregar lotes</button></div>
    <div class="lista" hidden><input type="search" id="cmp-buscar" placeholder="Buscar lote (p. ej. 95, Z-2B)…" aria-label="Buscar lote">
      <button type="button" class="bt" id="cmp-limpiar" style="margin-left:6px">Quitar todos</button><div class="grs"></div></div>
    <div class="ref ref-t"><div class="r-tit"></div><div class="r-uni"></div>
      <button type="button" class="bt cp" data-cp="t" title="Copiar el título y la unidad para pegarlos en PowerPoint">📋 Copiar título</button></div>
    <div class="graf"></div>
    <div class="ref ref-n"><div class="r-notas"></div>
      <button type="button" class="bt cp" data-cp="n" title="Copiar las notas y la fuente">📋 Copiar notas</button></div>
    <div class="nota"></div>
  </div>
  <div class="tip"></div>
</div>
<script>
(function(){
const caja=document.getElementById("cmp"),fondo=document.getElementById("cmp-fondo"),tip=caja.querySelector(".tip");
const $=s=>caja.querySelector(s);
// paleta de estadísticas del proyecto
const PAL=["#111149","#72d2d5","#ccd3df","#ffc41d","#2a9da1","#5a5a9e","#8c96ad","#181818","#b9e9eb","#dd6140"];
const COL_DEP=["#111149","#72d2d5","#ccd3df","#ffc41d","#2a9da1","#5a5a9e"];
const MES=["ene","feb","mar","abr","may","jun","jul","ago","set","oct","nov","dic"];
const PROD={T:"líquidos totales",P:"petróleo",L:"LGN y condensado",G:"gas natural"};
const SIN_OP="Sin operador identificado";
const st={v:"p",prod:"T",u:"m",modo:{p:"l",c:"l",d:"d"},desde:null,hasta:null,sel:[]};
let U=null;                                   // universo de lotes (se arma al abrir, cuando los datos ya cargaron)
function num(v,d){if(v==null||!isFinite(v))return "–";const s=Math.abs(v).toFixed(d==null?1:d).split(".");
  return (v<0&&Math.abs(v)>=Math.pow(10,-(d==null?1:d))/2?"-":"")+s[0].replace(/\B(?=(\d{3})+(?!\d))/g,".")+(s[1]?","+s[1]:"");}
function esc(s){return String(s).replace(/[&<>"]/g,c=>({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;"}[c]));}
function titulo(s){s=String(s||"");return s.charAt(0)+s.slice(1).toLowerCase();}
const clave=l=>String(l).toUpperCase().replace(/\(T+\)$/,"").replace(/[\s\-_]/g,"");   // «Z-2B» = «Z2B», «I(T)» = «I»
function natural(a,b){return String(a).localeCompare(String(b),"es",{numeric:true});}

// ── universo de lotes: producción (serie del lote o de su campo de un solo lote) y canon generado por lote ──
function universo(){
  const P=window.ppProd||null,A=window.ppActDatos||{},u={};
  const toma=(l)=>{const k=clave(l);return u[k]=u[k]||{k,nombre:String(l),serie:null,canon:[],dep:""};};
  if(P){const lc=P.D.lc||{},n={};Object.values(lc).forEach(c=>n[c]=(n[c]||0)+1);
    Object.keys(lc).forEach(l=>{const c=lc[l],x=P.S["L:"+l]||(n[c]===1?P.S["C:"+c]:null);if(!x)return;
      const o=toma(l);o.nombre=String(l);o.serie=x;o.dep=x.dp||"";});}
  const cl=A.cl||{};
  for(const c in cl)for(const l in cl[c]){const o=toma(l);o.canon.push(cl[c][l]);
    if(!o.dep){const t={};for(const d in cl[c][l])for(const a in cl[c][l][d])t[d]=(t[d]||0)+cl[c][l][d][a];
      o.dep=Object.keys(t).sort((x,y)=>t[y]-t[x])[0]||"";}}
  return u;}
function tiene(o,v){return v==="p"?!!o.serie:o.canon.length>0;}

// ── años de cada variable ──
function aniosVar(){const P=window.ppProd,A=window.ppActDatos||{};
  if(st.v==="p"){if(!P)return [];const a0=P.D.ini,a1=P.D.ini+Math.floor((P.D.n-1)/12),o=[];for(let a=a0;a<=a1;a++)o.push(a);
    for(let a=a1+1;mesesProd(a).nd;a++)o.push(a);        // años que solo tienen meses cerrados del reporte diario
    return o;}
  const r=(A.rango||{}).cl;if(!r)return [];const o=[];for(let a=r[0];a<=r[1];a++)o.push(a);return o;}
// producción: meses de la estadística mensual (nm) y, a continuación, meses CERRADOS del reporte diario (nd);
// el mes en curso del reporte diario no entra (se vería más bajo)
function mesCerrado(a,m){const DIA=window.ppProd.D.dia;if(!DIA)return -1;
  const i0=Math.round((Date.UTC(a,m,1)-Date.UTC(DIA.ini[0],DIA.ini[1]-1,DIA.ini[2]))/864e5),nd=new Date(Date.UTC(a,m+1,0)).getUTCDate();
  return i0>=0&&i0+nd<=DIA.n?i0:-1;}
function mesesProd(a){const D=window.ppProd.D,t0=(a-D.ini)*12,nm=Math.max(0,Math.min(12,D.n-t0));let nd=0;
  if(t0+nm>=D.n)while(nm+nd<12&&mesCerrado(a,nm+nd)>=0)nd++;
  return {nm,nd};}
function mesesDe(a){                          // meses con dato del año (para marcar «2026*»)
  if(st.v==="p"){const r=mesesProd(a);return r.nm+r.nd;}
  const n=((window.ppActDatos||{}).cln||{})[a];return n==null?12:n;}
function diasDe(a){const D=window.ppProd.D,nm=mesesDe(a);let d=0;for(let m=0;m<nm;m++)d+=new Date(Date.UTC(a,m+1,0)).getUTCDate();return d||1;}

// ── valores ──
const elegir=(p,l,g)=>st.prod==="G"?g:st.prod==="P"?p:st.prod==="L"?l:p+l;
function prodDiaria(x,a){const r=mesesProd(a),dd=x.dd||{};let s=0;        // suma de los meses cerrados del reporte diario
  for(let m=r.nm;m<r.nm+r.nd;m++){const i0=mesCerrado(a,m),n=new Date(Date.UTC(a,m+1,0)).getUTCDate();
    for(let i=i0;i<i0+n;i++)s+=elegir((dd.p||[])[i]||0,(dd.l||[])[i]||0,(dd.g||[])[i]||0);}
  return s;}
function prodAnual(x,a){const D=window.ppProd.D,t0=(a-D.ini)*12,nm=mesesProd(a).nm;let s=0;
  for(let t=t0;t<t0+nm;t++){if(t<0)continue;s+=elegir((x.p||[])[t]||0,(x.l||[])[t]||0,(x.gs||[])[t]||0);}
  return s+prodDiaria(x,a);}
function ultimoOperador(x,a){                 // operador único del último año con producción mensual (o null)
  const D=window.ppProd.D;for(let i=Math.min(a-D.ini,Math.floor((D.n-1)/12));i>=0;i--){
    const ops=Object.keys(x.ob||{}).filter(op=>["p","l","g"].some(k=>((x.ob[op][k]||[])[i]||0)>0));if(ops.length)return ops.length===1?ops[0]:null;}
  return null;}
function prodOps(x,a){                        // {operador: volumen del año} (lo no atribuido va a «Sin operador identificado»)
  const D=window.ppProd.D,i=a-D.ini,ks=st.prod==="G"?["g"]:st.prod==="P"?["p"]:st.prod==="L"?["l"]:["p","l"],o={};let s=0;
  for(const op in (x.ob||{})){const v=ks.reduce((q,k)=>q+(((x.ob[op][k]||[])[i])||0),0);if(v>0){o[op]=v;s+=v;}}
  const dia=prodDiaria(x,a);                  // meses del reporte diario: al operador del lote si era uno solo
  if(dia>0){const op=ultimoOperador(x,a);if(op){o[op]=(o[op]||0)+dia;s+=dia;}}
  const r=prodAnual(x,a)-s;if(r>0.5)o[SIN_OP]=r;return o;}
function escala(v,a){                         // barriles → millones o miles de bbl, o bpd; gas (MPC) → miles de millones o millones de pies³, o MMPC/día
  if(st.u==="d")return v/(st.prod==="G"?1000:1)/diasDe(a);   // por día: bpd o millones de pies³ por día
  return st.u==="k"?v/1e3:v/1e6;}
function canonDep(o,a){const r={};o.canon.forEach(c=>{for(const d in c){const v=(c[d]||{})[a];if(v!=null)r[d]=(r[d]||0)+v;}});return r;}
function canonTot(o,a){const r=canonDep(o,a),ks=Object.keys(r);return ks.length?ks.reduce((s,d)=>s+r[d],0):null;}
function partesOps(o,a){                      // participación de cada operador en la producción del lote ese año
  if(!o.serie||!o.serie.ob)return null;const D=window.ppProd.D,i=a-D.ini;
  const vol=ks=>{const r={};let s=0;for(const op in o.serie.ob){const v=ks.reduce((q,k)=>q+(((o.serie.ob[op][k]||[])[i])||0),0);if(v>0){r[op]=v;s+=v;}}return [r,s];};
  let [r,s]=vol(["p","l"]);if(!s)[r,s]=vol(["g"]);if(!s)return null;
  for(const op in r)r[op]/=s;return r;}
function unidad(){if(st.v!=="p")return "millones de US$";
  if(st.prod==="G")return st.u==="d"?"millones de pies cúbicos por día":st.u==="k"?"millones de pies cúbicos":"miles de millones de pies cúbicos";
  return st.u==="d"?"barriles por día":st.u==="k"?"miles de barriles":"millones de barriles";}
function decimales(){return st.v==="p"&&st.u==="d"&&st.prod!=="G"?0:1;}

// ── modos de «Apilar por» ──
const MODOS={p:[["l","Lote"],["e","Empresa operadora"],["t","Lote (barras lado a lado)"]],
             c:[["l","Lote"],["e","Empresa operadora"],["t","Lote (barras lado a lado)"]],
             d:[["d","Departamento que lo recibe"],["ld","Lote y departamento (barras lado a lado)"]]};
// cuadro: {anios, grupos:[{nombre, series:[{nombre, color, v:{año:valor}}]}], filas:[{nombre, color, v}]}
function cuadro(){
  const an=anios(),lotes=st.sel.map(k=>U[k]).filter(o=>o&&tiene(o,st.v)),m=st.modo[st.v];
  const R={anios:an,grupos:[],filas:[],lotes};if(!lotes.length)return R;
  const valLote=(o,a)=>st.v==="p"?(()=>{const v=prodAnual(o.serie,a);return v||mesesDe(a)?escala(v,a):null;})():canonTot(o,a);
  if(st.v!=="d"&&(m==="l"||m==="t")){
    const ss=lotes.map((o,i)=>{const v={};an.forEach(a=>{const x=valLote(o,a);if(x!=null)v[a]=x;});return {nombre:"Lote "+o.nombre,color:PAL[i%PAL.length],v};});
    R.grupos=m==="l"?[{nombre:"",series:ss}]:ss.map(s=>({nombre:s.nombre,series:[s]}));R.filas=ss;return R;}
  if(st.v!=="d"&&m==="e"){const ops={};
    lotes.forEach(o=>an.forEach(a=>{
      if(st.v==="p"){const po=prodOps(o.serie,a);for(const op in po){ops[op]=ops[op]||{};ops[op][a]=(ops[op][a]||0)+escala(po[op],a);}}
      else{const t=canonTot(o,a);if(t==null)return;const p=partesOps(o,a);
        if(!p){ops[SIN_OP]=ops[SIN_OP]||{};ops[SIN_OP][a]=(ops[SIN_OP][a]||0)+t;return;}
        for(const op in p){ops[op]=ops[op]||{};ops[op][a]=(ops[op][a]||0)+t*p[op];}}}));
    const ini=op=>{const a=an.find(a=>(ops[op][a]||0)>0);return a==null?9999:a;};
    const visible=0.5*Math.pow(10,-decimales());            // sin filas que en el cuadro serían todo «0,0»
    const ks=Object.keys(ops).filter(op=>an.some(a=>(ops[op][a]||0)>=visible)).sort((x,y)=>(x===SIN_OP)-(y===SIN_OP)||ini(x)-ini(y)||natural(x,y));
    const ss=ks.map((op,i)=>({nombre:op,color:op===SIN_OP?"#ffffff":PAL[i%PAL.length],v:ops[op]}));
    R.grupos=[{nombre:"",series:ss}];R.filas=ss;return R;}
  // distribución del canon: parte de cada departamento
  const deps=[];lotes.forEach(o=>an.forEach(a=>Object.keys(canonDep(o,a)).forEach(d=>{if(!deps.includes(d))deps.push(d);})));
  const tdep=d=>lotes.reduce((s,o)=>s+an.reduce((q,a)=>q+(canonDep(o,a)[d]||0),0),0);
  deps.sort((x,y)=>tdep(y)-tdep(x));const col=d=>COL_DEP[deps.indexOf(d)%COL_DEP.length];
  if(m==="d"){const ss=deps.map(d=>{const v={};lotes.forEach(o=>an.forEach(a=>{const x=canonDep(o,a)[d];if(x!=null)v[a]=(v[a]||0)+x;}));
      return {nombre:"Para "+titulo(d),color:col(d),v};});
    R.grupos=[{nombre:"",series:ss}];R.filas=ss;return R;}
  R.grupos=lotes.map(o=>({nombre:"Lote "+o.nombre,series:deps.map(d=>{const v={};an.forEach(a=>{const x=canonDep(o,a)[d];if(x!=null)v[a]=x;});
    return {nombre:"Lote "+o.nombre+" → "+titulo(d),color:col(d),v};}).filter(s=>Object.keys(s.v).length)}));
  R.filas=R.grupos.flatMap(g=>g.series);return R;}
function anios(){const t=aniosVar();if(!t.length)return [];
  let d=st.desde,h=st.hasta;if(d==null||d<t[0]||d>t[t.length-1])d=Math.max(t[0],t[t.length-1]-7);
  if(h==null||h>t[t.length-1]||h<t[0])h=t[t.length-1];if(d>h)d=h;st.desde=d;st.hasta=h;return t.filter(a=>a>=d&&a<=h);}

// ── gráfico y cuadro en un solo SVG (es lo que se guarda como imagen); título, unidad y notas van aparte, como referencia ──
const AZ="#111149",FAM="Barlow, Arial, sans-serif";
function partir(t,max){const out=[];let l="";String(t).split(/\s+/).forEach(p=>{const q=l?l+" "+p:p;if(q.length>max&&l){out.push(l);l=p;}else l=q;});if(l)out.push(l);return out;}
function svgCuadro(R){
  const an=R.anios,n=an.length,dec=decimales(),apil=R.grupos.length===1,k=R.grupos.length;
  const tot=a=>{let s=0,h=false;R.filas.forEach(f=>{const v=f.v[a];if(v!=null){s+=v;h=true;}});return h?s:null;};
  const filas=apil?R.filas.slice().reverse():R.filas;          // apiladas: la de arriba de la barra va primero (como el cuadro)
  const largo=Math.max(...filas.map(f=>f.nombre.length),5);
  const LW=Math.min(330,Math.max(140,largo*6.6+30));
  const anchoNum=Math.max(...an.map(a=>Math.max(num(tot(a)||0,dec).length,...filas.map(f=>num(f.v[a]||0,dec).length))),4);
  const CW=Math.max(52,anchoNum*6.8+14),PX=10,Wt=LW+n*CW,W=Wt+2*PX;
  const T=14,CH=230,HR=22,RH=21,yb=T+CH,t0=yb+HR,nf=filas.length+1,H=t0+nf*RH+4;
  let vmax=0;an.forEach(a=>{if(apil)vmax=Math.max(vmax,tot(a)||0);else R.grupos.forEach(g=>vmax=Math.max(vmax,g.series.reduce((s,x)=>s+(x.v[a]||0),0)));});
  vmax=vmax||1;const paso=Math.pow(10,Math.floor(Math.log10(vmax)));
  const nice=[.1,.2,.25,.5,1,2,2.5,5,10].map(f=>f*paso).find(x=>vmax/x<=6)||paso*10;vmax=Math.ceil(vmax/nice)*nice;
  const y=v=>T+CH*(1-v/vmax),x0=PX+LW;
  let g=`<rect width="${W}" height="${H}" fill="#ffffff"/>`;
  for(let v=0;v<=vmax+1e-9;v+=nice){g+=`<line x1="${x0}" x2="${x0+n*CW}" y1="${y(v).toFixed(1)}" y2="${y(v).toFixed(1)}" stroke="#ececec"/>`+
    `<text x="${x0-10}" y="${(y(v)+3.5).toFixed(1)}" font-size="10" fill="#595959" text-anchor="end">${num(v,nice<1?2:0).replace(/,?0+$/,m=>m.startsWith(",")||nice<1?"":m)}</text>`;}
  const barra=(x,y1,y0,w,col)=>`<rect x="${x.toFixed(1)}" y="${y1.toFixed(1)}" width="${w.toFixed(1)}" height="${Math.max(y0-y1,0.6).toFixed(1)}" fill="${col}" stroke="${AZ}" stroke-width="0.8"/>`;
  an.forEach((a,i)=>{const cx=x0+i*CW+CW/2;
    if(apil){const bw=Math.min(CW*0.46,46);let base=0;
      R.grupos[0].series.forEach(s=>{const v=s.v[a]||0;if(v<=0)return;g+=barra(cx-bw/2,y(base+v),y(base),bw,s.color);base+=v;});}
    else{const gw=Math.min(CW*0.82,k*30),bw=Math.max(gw/k-2,2);
      R.grupos.forEach((gr,j)=>{let base=0;const bx=cx-gw/2+j*(gw/k)+1;
        gr.series.forEach(s=>{const v=s.v[a]||0;if(v<=0)return;g+=barra(bx,y(base+v),y(base),bw,s.color);base+=v;});});}});
  g+=`<line x1="${x0}" x2="${x0+n*CW}" y1="${yb}" y2="${yb}" stroke="#595959" stroke-width="1"/>`;
  // tabla: años en la cabecera, una fila por serie y TOTAL
  const lin=(x1,y1,x2,y2,w)=>`<line x1="${x1}" y1="${y1}" x2="${x2}" y2="${y2}" stroke="#7f7f7f" stroke-width="${w||0.7}"/>`;
  g+=lin(x0,yb,x0+n*CW,yb)+lin(x0+n*CW,yb,x0+n*CW,t0+nf*RH)+lin(x0,yb,x0,t0+nf*RH)+lin(PX,t0,PX,t0+nf*RH)+lin(PX,t0+nf*RH,x0+n*CW,t0+nf*RH);
  an.forEach((a,i)=>{if(i)g+=lin(x0+i*CW,yb,x0+i*CW,t0,0.5);
    g+=`<text x="${x0+i*CW+CW/2}" y="${yb+15}" font-size="11" font-weight="700" fill="${AZ}" text-anchor="middle">${a}${mesesDe(a)<12?"*":""}</text>`;});
  filas.concat([{nombre:"TOTAL",total:true}]).forEach((f,r)=>{const yy=t0+r*RH;g+=lin(PX,yy,x0+n*CW,yy);
    const max=Math.floor((LW-26)/6.4),et=f.nombre.length>max?f.nombre.slice(0,max-1)+"…":f.nombre;
    if(!f.total)g+=`<rect x="${PX+5}" y="${yy+6}" width="9" height="9" fill="${f.color}" stroke="${AZ}" stroke-width="0.8"/>`;
    g+=`<text x="${PX+(f.total?20:19)}" y="${yy+14.5}" font-size="11" font-weight="700" fill="${AZ}">${esc(et)}<title>${esc(f.nombre)}</title></text>`;
    an.forEach((a,i)=>{const v=f.total?tot(a):f.v[a];
      g+=`<text x="${x0+i*CW+CW/2}" y="${yy+14.5}" font-size="11" font-weight="700" fill="${AZ}" text-anchor="middle">${v==null?"–":num(v,dec)}</text>`;});});
  an.forEach((a,i)=>{g+=`<rect class="hit" data-i="${i}" x="${x0+i*CW}" y="${T}" width="${CW}" height="${CH+HR+nf*RH}" fill="transparent"/>`;});
  return {svg:`<svg viewBox="0 0 ${W} ${H}" width="${W}" height="${H}" font-family="${FAM}" style="width:100%;height:auto;min-width:${Math.min(W,LW+n*44)}px;max-width:${Math.round(W*1.25)}px" role="img" aria-label="${esc(tituloCuadro(R))}">${g}</svg>`,tot,filas,W,H};}

function lotesTexto(R){const ls=R.lotes.map(o=>o.nombre);return ls.length===1?"Lote "+ls[0]:"Lotes "+ls.slice(0,-1).join(", ")+" y "+ls[ls.length-1];}
function tituloAuto(R){const m=st.modo[st.v],an=R.anios,per=an[0]===an[an.length-1]?an[0]:an[0]+"-"+an[an.length-1];
  const emp=m==="e"?" por empresa operadora":"";
  const qué=st.v==="p"?`Producción fiscalizada de ${PROD[st.prod]}${emp}`:st.v==="c"?`Canon y sobrecanon generado${emp}`:
    `Distribución del canon y sobrecanon generado por departamento`;
  return `${lotesTexto(R)}: ${qué}, ${per}`;}
function tituloCuadro(R){return R&&R.lotes&&R.lotes.length?tituloAuto(R):"Comparar lotes";}
function subCuadro(R){return "("+unidad().charAt(0).toUpperCase()+unidad().slice(1)+")";}
const MESES=["enero","febrero","marzo","abril","mayo","junio","julio","agosto","setiembre","octubre","noviembre","diciembre"];
function notasCuadro(R){const an=R.anios,ult=aniosVar().slice(-1)[0],out=[];
  an.filter(a=>mesesDe(a)<12).forEach(a=>{const m=mesesDe(a);
    const r=st.v==="p"?mesesProd(a):{nm:m,nd:0},ini=0;
    out.push(a===ult&&m?`* ${a}: corresponde al periodo acumulado ${MESES[ini]}-${MESES[m-1]}`+
      (r.nd?(r.nm?` (${MESES[r.nm]}-${MESES[r.nm+r.nd-1]}: suma de los reportes diarios de PERUPETRO, meses cerrados)`:" (suma de los reportes diarios de PERUPETRO, meses cerrados)"):"")+".":`* ${a}: ${m} meses con dato.`);});
  if(st.v==="p"&&st.modo.p==="e"&&R.anios.some(a=>mesesProd(a).nd))out.push("En los meses del reporte diario, la producción se asigna al operador del lote (si tenía uno solo); si no, va a «Sin operador identificado».");
  if(st.v==="c"&&st.modo.c==="e")out.push("Si un lote tuvo más de un operador en el año, su canon se reparte según la producción fiscalizada de cada uno (PERUPETRO publica el canon por lote).");
  if(st.v==="d")out.push("Parte del canon y sobrecanon generado por cada lote que corresponde a cada departamento.");
  if(st.v!=="p")out.push("Canon y sobrecanon generado en el mes de producción, sin ajustes de meses anteriores.");
  out.push(st.v==="p"?(R.anios.some(a=>mesesProd(a).nd)?"Fuente: PERUPETRO, producción fiscalizada (estadística mensual y reportes diarios).":"Fuente: PERUPETRO, producción fiscalizada."):"Fuente: PERUPETRO, Canon Petrolero (determinación).");
  return out;}
function notaCuadro(R){return notasCuadro(R).join(" ");}

// ── lista de lotes ──
function pintarLista(){const q=clave($("#cmp-buscar").value||""),grs={};
  Object.values(U).sort((a,b)=>natural(a.nombre,b.nombre)).forEach(o=>{if(q&&!o.k.includes(q))return;
    const d=o.dep?titulo(o.dep):"Sin departamento";(grs[d]=grs[d]||[]).push(o);});
  $(".lista .grs").innerHTML=Object.keys(grs).sort(natural).map(d=>`<div class="gr"><b>${esc(d)}</b>${grs[d].map(o=>{const ok=tiene(o,st.v);
    return `<label class="${ok?"":"no"}" title="${ok?"":"Sin datos de esta variable"}"><input type="checkbox" data-k="${esc(o.k)}" ${st.sel.includes(o.k)?"checked":""} ${ok?"":"disabled"}> Lote ${esc(o.nombre)}</label>`;}).join("")}</div>`).join("")||`<div class="nota">Ningún lote coincide.</div>`;
  $(".lista .grs").querySelectorAll("input").forEach(c=>c.onchange=()=>{const k=c.dataset.k;
    st.sel=c.checked?st.sel.concat([k]):st.sel.filter(x=>x!==k);pintar(false);});}
function pintar(lista){
  $("#cmp-var").value=st.v;$("#cmp-prod").value=st.prod;$("#cmp-u").value=st.u;
  [["m","Millones de barriles","Miles de millones de pies³"],["k","Miles de barriles","Millones de pies³"],["d","Barriles por día","Millones de pies³ por día"]]
    .forEach(([v,l,g])=>{$("#cmp-u").querySelector(`option[value=${v}]`).textContent=st.prod==="G"?g:l;});
  caja.querySelectorAll(".solo-p").forEach(e=>e.style.display=st.v==="p"?"":"none");
  const ms=MODOS[st.v];if(!ms.some(x=>x[0]===st.modo[st.v]))st.modo[st.v]=ms[0][0];
  $("#cmp-modo").innerHTML=ms.map(([k,n])=>`<option value="${k}"${k===st.modo[st.v]?" selected":""}>${n}</option>`).join("");
  const R=cuadro(),t=aniosVar();
  ["desde","hasta"].forEach(k=>{$("#cmp-"+k).innerHTML=t.map(a=>`<option value="${a}"${a===st[k]?" selected":""}>${a}</option>`).join("");});
  const lotesV=st.sel.map(k=>U[k]).filter(Boolean);
  $(".chips").innerHTML=lotesV.map((o,i)=>`<span class="chip"${tiene(o,st.v)?"":` style="opacity:.5" title="Sin datos de esta variable"`}>${st.v!=="d"&&st.modo[st.v]!=="e"?`<i style="background:${PAL[R.lotes.indexOf(o)%PAL.length]};${tiene(o,st.v)?"":"display:none"}"></i>`:""}Lote ${esc(o.nombre)}<button type="button" data-q="${esc(o.k)}" title="Quitar" aria-label="Quitar lote ${esc(o.nombre)}">×</button></span>`).join("")||`<span class="nota" style="margin:0">Ningún lote elegido.</span>`;
  $(".chips").querySelectorAll("button[data-q]").forEach(b=>b.onclick=()=>{st.sel=st.sel.filter(x=>x!==b.dataset.q);pintar(true);});
  if(lista!==false&&!$(".lista").hidden)pintarLista();
  const g=$(".graf");
  const hay=R.lotes.length&&R.anios.length;caja.querySelectorAll(".ref").forEach(e=>e.style.display=hay?"":"none");
  if(!hay){$(".nota").textContent="";
    g.innerHTML=`<div class="vacio">${st.sel.length?"Los lotes elegidos no tienen datos de esta variable.":"Elige uno o más lotes con «＋ Agregar lotes»."}</div>`;R.vacio=true;st.R=R;return;}
  const S=svgCuadro(R);st.R=R;st.S=S;
  $(".r-tit").textContent=tituloCuadro(R);$(".r-uni").textContent=subCuadro(R);
  $(".r-notas").innerHTML=notasCuadro(R).map(esc).join("<br>");
  $(".nota").textContent="Pasa el cursor por una columna para ver el detalle. «🖼 Imagen» guarda solo el gráfico con su cuadro (PNG 4×); «⬇ SVG» lo guarda vectorial (nítido a cualquier tamaño en PowerPoint). El título y las notas se copian aparte.";
  g.innerHTML=S.svg;
  const svg=g.querySelector("svg"),dec=decimales();
  svg.addEventListener("mousemove",ev=>{const r=ev.target.closest&&ev.target.closest(".hit");if(!r){tip.style.display="none";return;}
    const a=R.anios[+r.dataset.i],ps=S.filas.filter(f=>f.v[a]!=null&&f.v[a]!==0).map(f=>`${esc(f.nombre)}: ${num(f.v[a],dec)}`);
    tip.innerHTML=`<b>${a}${mesesDe(a)<12?"*":""}</b><br>${ps.join("<br>")||"sin dato"}${ps.length>1?`<br><b>Total: ${num(S.tot(a),dec)}</b>`:""}<br><span style="color:#ccc">${esc(unidad())}</span>`;
    tip.style.display="block";tip.style.left=Math.min(ev.clientX+12,innerWidth-260)+"px";tip.style.top=(ev.clientY+12)+"px";});
  svg.addEventListener("mouseleave",()=>tip.style.display="none");}
function csv(){const R=st.R;if(!R||R.vacio)return;const S=st.S,dec=decimales();
  const dc=v=>v==null?"":String(Math.round(v*Math.pow(10,dec+3))/Math.pow(10,dec+3)).replace(".",",");
  const q=s=>/[;"\n]/.test(s)?'"'+String(s).replace(/"/g,'""')+'"':s;
  const filas=[[q(tituloCuadro(R)+" "+subCuadro(R))].concat(R.anios.map(a=>a+(mesesDe(a)<12?"*":"")))]
    .concat(S.filas.map(f=>[q(f.nombre)].concat(R.anios.map(a=>dc(f.v[a])))),[["TOTAL"].concat(R.anios.map(a=>dc(S.tot(a))))]);
  const s="﻿"+filas.map(f=>f.join(";")).join("\n")+"\n\n"+notasCuadro(R).map(q).join("\n")+"\n";
  const a=document.createElement("a");a.href=URL.createObjectURL(new Blob([s],{type:"text/csv"}));a.download=archivo()+".csv";
  document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(a.href),1000);}
function archivo(){return ("comparar_"+{p:"produccion",c:"canon",d:"distribucion_canon"}[st.v]+"_lotes_"+st.R.lotes.map(o=>o.nombre).join("_")).replace(/[^\w-]+/g,"_").slice(0,90);}
function abrir(lotes){if(!U)U=universo();
  (lotes||[]).forEach(l=>{const k=clave(l);if(U[k]&&!st.sel.includes(k))st.sel.push(k);});
  if(!st.sel.length&&!(lotes||[]).length){             // sin elección previa: los 3 lotes con más producción en el último año completo
    const P=window.ppProd;if(P){const D=P.D,a=D.ini+Math.floor(D.n/12)-1;
      st.sel=Object.values(U).filter(o=>o.serie).map(o=>[o.k,prodAnual(o.serie,a)]).filter(x=>x[1]>0).sort((x,y)=>y[1]-x[1]).slice(0,3).map(x=>x[0]);}}
  for(const k in window){try{const o=window[k];if(o&&window.L&&o instanceof L.Map)o.closePopup();}catch(e){}}
  caja.style.display="block";fondo.style.display="block";pintar();caja.querySelector(".x").focus();}
function cerrar(){caja.style.display="none";fondo.style.display="none";tip.style.display="none";}
window.ppCompararAbrir=abrir;
caja.querySelector(".x").onclick=cerrar;fondo.onclick=cerrar;
document.addEventListener("keydown",e=>{if(e.key==="Escape"&&caja.style.display==="block")cerrar();});
document.getElementById("cmp-abrir").onclick=()=>abrir();
document.addEventListener("click",ev=>{const b=ev.target.closest&&ev.target.closest("[data-cmp]");if(!b||caja.contains(b))return;ev.preventDefault();abrir([b.dataset.cmp]);});
$("#cmp-var").onchange=e=>{st.v=e.target.value;pintar();};
$("#cmp-prod").onchange=e=>{st.prod=e.target.value;pintar();};
$("#cmp-u").onchange=e=>{st.u=e.target.value;pintar();};
$("#cmp-modo").onchange=e=>{st.modo[st.v]=e.target.value;pintar();};
$("#cmp-desde").onchange=e=>{st.desde=+e.target.value;if(st.hasta<st.desde)st.hasta=st.desde;pintar();};
$("#cmp-hasta").onchange=e=>{st.hasta=+e.target.value;if(st.desde>st.hasta)st.desde=st.hasta;pintar();};
$("#cmp-mas").onclick=()=>{const l=$(".lista");l.hidden=!l.hidden;$("#cmp-mas").setAttribute("aria-expanded",!l.hidden);
  $("#cmp-mas").textContent=l.hidden?"＋ Agregar lotes":"▴ Cerrar la lista";if(!l.hidden){pintarLista();$("#cmp-buscar").focus();}};
$("#cmp-buscar").oninput=()=>pintarLista();
$("#cmp-limpiar").onclick=()=>{st.sel=[];pintar();};
$("#cmp-csv").onclick=csv;
// imagen: el mismo SVG a 4× (PNG) o vectorial (SVG), con la letra Barlow incrustada (si hay conexión; si no, Arial)
let FUENTE=null;
async function fuenteBarlow(){if(FUENTE!=null)return FUENTE;
  try{let css=await (await fetch("https://fonts.googleapis.com/css2?family=Barlow:wght@400;700&display=swap")).text();
    css=css.split("}").filter(b=>/U\+0000-00FF/.test(b)).join("}")+"}";      // solo el subconjunto latino
    for(const u of [...new Set(css.match(/https:[^)]+/g)||[])]){const b=await (await fetch(u)).blob();
      const d=await new Promise(r=>{const f=new FileReader();f.onload=()=>r(f.result);f.readAsDataURL(b);});css=css.split(u).join(d);}
    FUENTE=css;}catch(e){FUENTE="";}
  return FUENTE;}
async function svgLimpio(){const c=$(".graf svg").cloneNode(true);c.querySelectorAll(".hit,title").forEach(e=>e.remove());c.removeAttribute("style");
  c.setAttribute("xmlns","http://www.w3.org/2000/svg");const f=await fuenteBarlow();
  if(f){const s=document.createElementNS("http://www.w3.org/2000/svg","style");s.textContent=f;c.insertBefore(s,c.firstChild);}
  return new XMLSerializer().serializeToString(c);}
function bajar(blob,nombre){const a=document.createElement("a");a.href=URL.createObjectURL(blob);a.download=nombre;
  document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(a.href),1500);}
async function imagenSVG(){if(!st.R||st.R.vacio)return;bajar(new Blob([await svgLimpio()],{type:"image/svg+xml"}),archivo()+".svg");}
async function imagen(){const R=st.R;if(!R||R.vacio)return;const S=st.S,k=4;
  const txt=await svgLimpio();
  const img=new Image();
  img.onload=()=>{const cv=document.createElement("canvas");cv.width=S.W*k;cv.height=S.H*k;const x=cv.getContext("2d");
    x.fillStyle="#fff";x.fillRect(0,0,cv.width,cv.height);x.drawImage(img,0,0,cv.width,cv.height);
    cv.toBlob(b=>bajar(b,archivo()+".png"),"image/png");};
  img.src="data:image/svg+xml;charset=utf-8,"+encodeURIComponent(txt);}
$("#cmp-img").onclick=imagen;
$("#cmp-svg").onclick=imagenSVG;
caja.querySelectorAll(".cp").forEach(b=>b.onclick=()=>{const R=st.R;if(!R||R.vacio)return;
  const t=b.dataset.cp==="t"?tituloCuadro(R)+"\n"+subCuadro(R):notasCuadro(R).join("\n"),txt=b.textContent;
  const listo=()=>{b.textContent="✓ Copiado";setTimeout(()=>b.textContent=txt,1200);};
  if(navigator.clipboard&&navigator.clipboard.writeText)navigator.clipboard.writeText(t).then(listo,()=>{prompt("Copia el texto:",t);});
  else prompt("Copia el texto:",t);});
})();
</script>
<script>
// el botón va dentro del selector de capas, debajo de «Producción fiscalizada…» (o de «Productos» si no hay producción)
(function mover(n){const l=document.querySelector(".leaflet-control-layers-list"),b=document.getElementById("cmp-abrir");if(!b)return;
  const pp=document.getElementById("pp-abrir"),lista=pp&&pp.classList.contains("en-capas");
  if(l&&(lista||!pp||n>45)){const d=pp&&lista?pp.parentNode:(()=>{const x=document.createElement("div");x.className="pp-abrir-caja";
      x.style.cssText="border-top:1px solid #ddd;margin-top:6px;padding-top:6px";l.appendChild(x);return x;})();
    d.appendChild(b);b.classList.remove("flota");if(window.L&&L.DomEvent)L.DomEvent.disableClickPropagation(d);return;}
  if(n<120)setTimeout(()=>mover(n+1),100);})(0);
</script>
'''
def series_panel(prod, geo, gas=None):
    if prod is None or not len(prod):
        return None
    p_ = prod.copy()
    p_["BARRILES"] = pd.to_numeric(p_["BARRILES"], errors="coerce").fillna(0)
    if gas is not None and len(gas):                    # el gas entra como un producto más (MPC en la columna BARRILES)
        g_ = gas.rename(columns={"MPC": "BARRILES"}).copy()
        g_["BARRILES"] = pd.to_numeric(g_["BARRILES"], errors="coerce").fillna(0)
        g_["PRODUCTO"] = "GAS NATURAL"
        p_ = pd.concat([p_, g_[[c for c in g_.columns if c in p_.columns]]], ignore_index=True)
    p_["_T"] = (p_["AÑO"].astype(int) - ANIO_INI) * 12 + p_["MES"].astype(int) - 1
    n = int(p_.loc[p_["BARRILES"] > 0, "_T"].max()) + 1                     # hasta el último mes con datos
    p_ = p_[(p_["_T"] >= 0) & (p_["_T"] < n)]
    _pr = p_["PRODUCTO"].astype(str).str.upper()
    p_["_PET"] = _pr.map(lambda x: 0 if x.startswith("PETR") else (2 if x.startswith("GAS") else 1))   # 0 petróleo, 1 LGN/cond., 2 gas
    p_["LOTE"] = p_["LOTE"].astype(str)
    base = p_["LOTE_HISTORICO"] if "LOTE_HISTORICO" in p_ else p_["LOTE"]
    p_["_CAMPO"] = (p_["CAMPO"] if "CAMPO" in p_ else base).fillna(base).astype(str)
    dep_geo = dict(zip(geo["LOTE"].astype(str), geo.get("DEPARTAMENTO", pd.Series("", index=geo.index)).fillna("")))
    p_["_DEP"] = p_["DEPARTAMENTO"].fillna("").astype(str) if "DEPARTAMENTO" in p_ else ""
    p_.loc[p_["_DEP"].str.strip() == "", "_DEP"] = p_["LOTE"].map(dep_geo).fillna("")
    p_["_PERU"] = "PERÚ"
    lotes_por_campo = p_.groupby("_CAMPO")["LOTE"].nunique()
    # campo y departamento de cada serie: las pestañas de estadísticas siguen a la serie elegida en el selector
    _nd = lambda t: "".join(c for c in unicodedata.normalize("NFKD", str(t)) if not unicodedata.combining(c)).upper().strip()
    _moda = lambda s: s[s.astype(str).str.strip() != ""].mode().iloc[0] if (s.astype(str).str.strip() != "").any() else ""
    dep_campo = p_.groupby("_CAMPO")["_DEP"].agg(_moda).map(_nd).to_dict()
    campo_lote = p_.drop_duplicates("LOTE").set_index("LOTE")["_CAMPO"].to_dict()
    salida = []
    for grupo, col, pref, con_ops in [("Total", "_PERU", "T", False), ("Departamentos", "_DEP", "D", False),
                                       ("Campos (lotes sucesivos sumados)", "_CAMPO", "C", True), ("Lotes", "LOTE", "L", True)]:
        q = p_[p_[col].astype(str).str.strip() != ""]
        if pref == "L":                                   # un lote solo aparece aparte si su campo tiene varios lotes
            q = q[q["_CAMPO"].map(lotes_por_campo).fillna(1) > 1]
        if q.empty:
            continue
        t = q.groupby([col, "_PET", "_T"])["BARRILES"].sum().unstack(fill_value=0).reindex(columns=range(n), fill_value=0)
        ops, obs = {}, {}
        if con_ops and "OPERADOR" in q:
            o_ = q[(q["BARRILES"] > 0) & (q["OPERADOR"].fillna("").astype(str).str.strip() != "")]
            for (k, a), s_ in o_.groupby([col, "AÑO"])["OPERADOR"]:
                ops.setdefault(k, {})[int(a)] = " / ".join(sorted(set(s_.astype(str))))
            # producción anual por operador (p petróleo, l LGN/condensado, g gas): colorea las barras del panel por empresa
            ny = (n - 1) // 12 + 1
            for (k, op, pet), g_ in o_.assign(_A=o_["_T"] // 12).groupby([col, "OPERADOR", "_PET"]):
                arr = [0] * ny
                for a, v in g_.groupby("_A")["BARRILES"].sum().items():
                    arr[int(a)] = int(round(float(v)))
                obs.setdefault(k, {}).setdefault(str(op), {})["plg"[int(pet)]] = arr
        orden = q[q["_PET"] < 2].groupby(col)["BARRILES"].sum()          # de mayor a menor producción de líquidos
        for k in sorted(q[col].unique(), key=lambda x: (-float(orden.get(x, 0)), str(x))):
            serie = lambda pet: [int(round(float(v))) for v in t.loc[(k, pet)]] if (k, pet) in t.index else [0] * n
            nombre = {"T": "Perú (total)", "D": str(k).title(), "C": f"Campo {k}", "L": f"Lote {k}"}[pref]
            gs = serie(2)
            ctx = ({"cp": str(k), "dp": dep_campo.get(k, "")} if pref == "C" else
                   {"cp": str(campo_lote.get(k, k)), "dp": dep_campo.get(campo_lote.get(k, k), "")} if pref == "L" else
                   {"dp": _nd(k)} if pref == "D" else {})
            salida.append({"id": f"{pref}:{k}", "g": grupo, "n": nombre, "p": serie(0), "l": serie(1), **({"gs": gs} if any(gs) else {}), **ctx,
                           **({"ob": obs[k]} if obs.get(k) else {}),
                           **({"o": ops.get(k, {})} if con_ops else {})})
    if not salida:
        return None
    defecto = next((x["id"] for x in salida if x["id"] == "D:LORETO"), salida[0]["id"])
    dia = series_diarias(diaria, salida, dict(zip(p_["LOTE"], p_["_CAMPO"])), dep_geo) if diaria is not None else None
    return {"ini": ANIO_INI, "n": n, "s": salida, "def": defecto, "dia": dia,
            "lc": {str(l): str(c) for l, c in campo_lote.items()},      # lote → campo («Comparar lotes»)
            "grupos": ["Total", "Departamentos", "Campos (lotes sucesivos sumados)", "Lotes"]}

def series_diarias(diaria, salida, campo_de, dep_de):
    """Agrega a cada serie del panel su producción DIARIA (dd.p petróleo y condensado, dd.l LGN en bbl; dd.g gas en Mcf)."""
    d = diaria[diaria["UNIDAD"].isin(["BBL", "MCF"])].copy()
    if d.empty:
        return None
    d["VALOR"] = pd.to_numeric(d["VALOR"], errors="coerce").fillna(0)
    d["_F"] = pd.to_datetime(d["FECHA"], format="%d/%m/%Y")
    ini = d["_F"].min(); nd = int((d["_F"].max() - ini).days) + 1
    d["_i"] = (d["_F"] - ini).dt.days
    d["_K"] = ["g" if u == "MCF" else ("l" if str(p).upper().startswith("LGN") else "p") for u, p in zip(d["UNIDAD"], d["PRODUCTO"])]
    d["LOTE"] = d["LOTE"].astype(str)
    d["_CAMPO"] = d["LOTE"].map(lambda l: campo_de.get(l, CAMPO_DE.get(l, l)) if "CAMPO_DE" in globals() else campo_de.get(l, l))
    dep = d["DEPARTAMENTO"].fillna("").astype(str) if "DEPARTAMENTO" in d else pd.Series("", index=d.index)
    d["_DEP"] = [x if x.strip() else dep_de.get(l, "") for x, l in zip(dep, d["LOTE"])]
    d["_PERU"] = "PERÚ"
    por_id = {x["id"]: x for x in salida}
    for col, pref in [("_PERU", "T"), ("_DEP", "D"), ("_CAMPO", "C"), ("LOTE", "L")]:
        t = d[d[col].astype(str).str.strip() != ""].groupby([col, "_K", "_i"])["VALOR"].sum()
        for (k, kk), s_ in t.groupby(level=[0, 1]):
            x = por_id.get(f"{pref}:{k}")
            if x is None:
                continue
            arr = [0] * nd
            for i, v in zip(s_.index.get_level_values(2), s_.values):
                arr[int(i)] = int(round(float(v)))
            x.setdefault("dd", {})[kk] = arr
    return {"ini": [int(ini.year), int(ini.month), int(ini.day)], "n": nd}

diaria = globals().get("diaria")
DATOS_PANEL = series_panel(prod, geo, globals().get("gas")) if prod is not None else None
IDS_PANEL = {x["id"] for x in DATOS_PANEL["s"]} if DATOS_PANEL else set()

PRODUCTOS = [("p", "Petróleo", "bpd"), ("l", "LGN y condensado", "bpd"), ("g", "Gas natural", "MMPCD")]

def estados_producto(datos):
    """{campo: {"p"|"l"|"g": {"e": estado, "u": última producción mm/aaaa, "r": promedio diario de los últimos 12 meses}}}.
    En producción = produjo en los últimos 12 meses con datos; paralizado = produjo antes; sin producción = nunca.
    Los meses posteriores a la última estadística mensual se toman de los reportes diarios."""
    import calendar, datetime as _dt
    if not datos:
        return {}
    ini, n, dia = datos["ini"], datos["n"], datos.get("dia")
    n_tot, d0 = n, None
    if dia:
        d0 = _dt.date(*dia["ini"]); ult = d0 + _dt.timedelta(days=dia["n"] - 1)
        n_tot = max(n, (ult.year - ini) * 12 + ult.month)
    dias_mes = [calendar.monthrange(ini + t // 12, t % 12 + 1)[1] for t in range(n_tot)]
    out = {}
    for x in datos["s"]:
        if not x["id"].startswith("C:"):
            continue
        res = {}
        for k, clave in (("p", "p"), ("l", "l"), ("g", "gs")):
            mens = list(x.get(clave) or [0] * n) + [0] * (n_tot - n)
            dias = list(dias_mes)
            if dia and n_tot > n:
                cubiertos = [0] * (n_tot - n)
                for i, v in enumerate((x.get("dd") or {}).get(k) or [0] * dia["n"]):
                    f = d0 + _dt.timedelta(days=i); t = (f.year - ini) * 12 + f.month - 1
                    if t >= n:
                        mens[t] += v; cubiertos[t - n] += 1
                for j, c in enumerate(cubiertos):
                    dias[n + j] = c or dias[n + j]
            pos = [t for t, v in enumerate(mens) if v > 0]
            if not pos:
                res[k] = {"e": "SIN PRODUCCIÓN"}
                continue
            L, fin = pos[-1], n_tot - 1
            ult12 = range(max(0, fin - 11), fin + 1)
            r = sum(mens[t] for t in ult12) / sum(dias[t] for t in ult12)
            res[k] = {"e": "EN PRODUCCIÓN" if L > fin - 12 else "PARALIZADO", "u": f"{L % 12 + 1:02d}/{ini + L // 12}",
                      "r": round(r / 1000, 2) if k == "g" else round(r, 0)}
        out[x["id"][2:]] = res
    return out

EST_PRODUCTO = estados_producto(DATOS_PANEL)
if EST_PRODUCTO:                                       # tabla para usar fuera del mapa
    _filas = []
    for _c, _e in sorted(EST_PRODUCTO.items()):
        _f = {"FECHA_EXTRACCION": date.today().strftime("%d/%m/%Y"), "CAMPO": _c}
        for _k, _n in (("p", "PETROLEO"), ("l", "LGN_CONDENSADO"), ("g", "GAS")):
            _d = _e.get(_k, {})
            _f.update({f"ESTADO_{_n}": _d.get("e", "SIN PRODUCCIÓN"), f"ULTIMA_PRODUCCION_{_n}": _d.get("u", ""),
                       f"ACTUAL_{_n}" + ("_MMPCD" if _k == "g" else "_BPD"): _d.get("r")})
        _filas.append(_f)
    _p = os.path.join(ruta, "PP_LOTES_ESTADO_PRODUCTO.csv")
    pd.DataFrame(_filas).to_csv(_p, index=False, encoding="utf-8-sig")
    subir(_p, DRIVE_FOLDER_ID, datos_drive)

def estado_producto_html(campo):
    e = EST_PRODUCTO.get(str(campo)) or {}
    filas = ""
    for k, nombre, uni in PRODUCTOS:
        d = e.get(k, {"e": "SIN PRODUCCIÓN"})
        if d["e"] == "SIN PRODUCCIÓN":                 # productos que el lote nunca produjo: no se muestran
            continue
        if d["e"] == "EN PRODUCCIÓN":
            txt = f"En producción · {_num(d['r'], 2 if k == 'g' else 0)} {uni}"
        elif d["e"] == "PARALIZADO":
            txt = f"Paralizado · última producción {d['u']}"
        else:
            txt = "Sin producción"
        col = COLOR_ESTADO.get("SIN PRODUCCIÓN REGISTRADA" if d["e"] == "SIN PRODUCCIÓN" else d["e"], "#9e9e9e")
        filas += (f"<tr><td style='color:#666;padding-right:8px'>{nombre}</td><td><span style='display:inline-block;width:9px;"
                  f"height:9px;border-radius:2px;background:{col};margin-right:5px'></span>{_h.escape(txt)}</td></tr>")
    if not filas:
        return f"<div style='margin-top:4px;color:#666'>Sin producción fiscalizada registrada ({ANIO_INI}–hoy)</div>"
    return (f"<table style='margin-top:4px'><tr><td colspan=2 style='color:#333'>"
            f"<b>Estado por producto</b> <span style='color:#888'>(promedio de los últimos 12 meses)</span></td></tr>{filas}</table>")

# ── Pozos perforados, regalías y canon (del notebook de producción, sección 6.1) ──
def _leer_extra(nombre):
    if nombre not in datos_drive:
        return None
    p = os.path.join(ruta, nombre); bajar_de_drive(datos_drive[nombre], p)
    return pd.read_csv(p, encoding="utf-8-sig", dtype={"LOTE": str, "LOTE_HISTORICO": str})

def _campo_de_lote(df):
    base = df["LOTE_HISTORICO"].astype(str) if "LOTE_HISTORICO" in df else df["LOTE"].astype(str)
    return [CAMPO_DE.get(l, b) for l, b in zip(df["LOTE"].astype(str), base)]

def _dep(t):
    return "".join(c for c in unicodedata.normalize("NFKD", str(t)) if not unicodedata.combining(c)).upper().strip()

# ACT: datos del panel «Pozos, regalías y canon» (un solo bloque JSON en el HTML)
#   p[campo][lote] = {"d": {año: pozos}, "e": {...}}; r[campo][lote][hidrocarburo] = {año: MM US$}; c[departamento] = {año: MM S/}
#   e[campo] = {"o": [[lote, operador, año inicio, año fin, bbl, MPC, regalías MM US$]],
#               "s": [[lote, empresa, %, suscripción, término, fase]], "t": [[lote, operador, suscripción, último día, edición]],
#               "v": [[lote, año, MM US$]] regalías que el PDF no permite atribuir a un operador}
ACT = {"p": {}, "r": {}, "c": {}, "e": {}, "rango": {}}
POZOS_CAMPO, REGALIAS_CAMPO = {}, {}
_pz = _leer_extra("PP_pozos_perforados_lote_anio.csv")
if _pz is not None and len(_pz):
    _pz["_C"] = _campo_de_lote(_pz)
    _pz["_T"] = _pz["TIPO"].map(lambda t: "d" if str(t).startswith("DESARROLLO") else "e")   # confirmatorios con exploratorios
    for (c, l, t), g in _pz.groupby(["_C", "LOTE", "_T"]):
        v = g.groupby("AÑO")["POZOS"].sum()
        v = {int(a): int(n) for a, n in v.items() if n}
        if v:
            ACT["p"].setdefault(c, {}).setdefault(l, {})[t] = v
    for c, d in ACT["p"].items():
        POZOS_CAMPO[c] = {t: {} for t in "de"}
        for l in d.values():
            for t, v in l.items():
                for a, n in v.items():
                    POZOS_CAMPO[c][t][a] = POZOS_CAMPO[c][t].get(a, 0) + n
    ACT["rango"]["p"] = [int(_pz["AÑO"].min()), int(_pz["AÑO"].max())]
_rg = _leer_extra("PP_regalias_lote_anio.csv")
if _rg is not None and len(_rg):
    _rg["_C"] = _campo_de_lote(_rg)
    for (c, l, h), g in _rg.groupby(["_C", "LOTE", "HIDROCARBURO"]):
        v = (g.groupby("AÑO")["REGALIA_US$"].sum() / 1e6).round(3)
        ACT["r"].setdefault(c, {}).setdefault(l, {})[h] = {int(a): float(x) for a, x in v.items() if x}
    for c, g in _rg.groupby("_C"):
        REGALIAS_CAMPO[c] = (g.groupby("AÑO")["REGALIA_US$"].sum() / 1e6).round(2).to_dict()
    ACT["rango"]["r"] = [int(_rg["AÑO"].min()), int(_rg["AÑO"].max())]
_cn = _leer_extra("PP_canon_departamento_anio.csv")
if _cn is not None and "AÑO_COMPLETO" in _cn:      # serie del Excel de Distribución: el año en curso aún está incompleto
    _cn = _cn[_cn["AÑO_COMPLETO"].astype(str).str.upper().isin(["TRUE", "1", "1.0"])]
if _cn is not None and len(_cn):
    for d, g in _cn.groupby(_cn["DEPARTAMENTO"].map(_dep)):
        ACT["c"][d] = {int(a): round(float(x), 2) for a, x in zip(g["AÑO"], g["CANON_MM_SOLES"])}
    ACT["rango"]["c"] = [int(_cn["AÑO"].min()), int(_cn["AÑO"].max())]
# Canon (punto 4, Excel de canon de PERUPETRO, notebook de producción sección 9):
#   cl[campo][lote][dep] = {año: MM US$} canon generado por el lote y la parte de cada departamento (Determinación)
#   cb[dep][tipo] = {año: MM S/} canon del departamento por beneficiario, solo años completos (Distribución)
#   cm[campo] = {"d": [[clave, municipalidad, provincia, nivel, % del lote]], "v": {clave: {año: MM S/}}} municipalidades
#   del lote (sus distritos y la provincial de cada provincia); cmn[dep][año] = meses con dato municipal; cln[año] = meses
def _lote_canon(l):
    l = str(l).strip()
    for c_ in (l, l + "(T)", l + "(TT)", re.sub(r"\(T+\)$", "", l), l.replace("-", "")):
        if c_ in CAMPO_DE:
            return CAMPO_DE[c_]
    return re.sub(r"\(T+\)$", "", l)
_cl = _leer_extra("PP_canon_lote_mes.csv")
if _cl is not None and len(_cl):
    _cl = _cl[_cl["LOTE"].astype(str) != "AJUSTES MESES ANTERIORES"]
    for (l, a), g in _cl.groupby(["LOTE", "AÑO"]):
        c = str(_lote_canon(l))
        for dd, cc in (("DEPARTAMENTO_1", "CANON_DEP_1_US$"), ("DEPARTAMENTO_2", "CANON_DEP_2_US$")):
            for d, v in g.groupby(dd)[cc].sum().items():
                if isinstance(d, str) and d and v:
                    ACT.setdefault("cl", {}).setdefault(c, {}).setdefault(str(l), {}).setdefault(_dep(d), {})[int(a)] = round(v / 1e6, 4)
    ACT["cln"] = {int(a): int(n) for a, n in _cl.groupby("AÑO")["MES"].nunique().items()}
    ACT["rango"]["cl"] = [int(_cl["AÑO"].min()), int(_cl["AÑO"].max())]
_cd = _leer_extra("PP_canon_distribucion_mes.csv")
if _cd is not None and len(_cd):
    _n = _cd.groupby(["DEPARTAMENTO", "AÑO"])["MES"].nunique()
    for (d, t, a), v in _cd.groupby(["DEPARTAMENTO", "TIPO_BENEFICIARIO", "AÑO"])["MONTO_SOLES"].sum().items():
        if _n.get((d, a), 0) == 12:
            ACT.setdefault("cb", {}).setdefault(_dep(d), {}).setdefault(t, {})[int(a)] = round(v / 1e6, 4)
        else:                                          # año incompleto: solo para el detalle de la vista del lote
            ACT.setdefault("cbi", {}).setdefault(_dep(d), {}).setdefault(t, {})[int(a)] = round(v / 1e6, 4)
            ACT.setdefault("cbn", {}).setdefault(_dep(d), {})[int(a)] = int(_n.get((d, a), 0))
# cng[dep][nivel] = {año: MM S/}: canon petrolero recaudado en el departamento por nivel de gobierno (R regional, M locales)
# según el MEF (notebook PP_MEF_Canon_Municipal.ipynb); sin el tipo NORMAL de 2005–2006 (todos los canones) ni el nivel nacional,
# que el MEF no desagrega por departamento
_ng = None   # el mapa ya no muestra el recaudado del MEF por nivel (pedido del usuario, 21/10/2026); queda en el CSV
if _ng is not None and len(_ng):
    _ng = _ng[(_ng["DEPARTAMENTO"].astype(str) != "TOTAL PAÍS") & ~_ng["TIPO_RECURSO"].astype(str).str.contains("todos los canones")
              & _ng["NIVEL_GOBIERNO"].astype(str).str.match(r"^[MR]:")]     # el nacional (E) solo se desagrega en 2007–2011
    # canon gasífero (Camisea): solo para Cusco, cuyo canon es sobre todo gasífero
    _ng = _ng[~_ng["TIPO_RECURSO"].astype(str).str.contains("GASIFER") | (_ng["DEPARTAMENTO"].map(_dep) == "CUSCO")]
    for (d, n, a), v in _ng.groupby(["DEPARTAMENTO", "NIVEL_GOBIERNO", "AÑO"])["RECAUDADO_SOLES"].sum().items():
        if v:
            ACT.setdefault("cng", {}).setdefault(_dep(d), {}).setdefault(str(n)[0], {})[int(a)] = round(v / 1e6, 4)
    if "cng" in ACT:
        anios = [a for d in ACT["cng"].values() for n in d.values() for a in n]
        ACT["rango"]["cng"] = [min(anios), max(anios)]
        print(f"   🏛️ Canon por nivel de gobierno (MEF): {len(ACT['cng'])} departamentos")
_cm = _leer_extra("PP_canon_municipal_mes.csv")
# años sin detalle de PERUPETRO (2003–2024): canon petrolero recaudado por cada municipalidad según el MEF
# (notebook PP_MEF_Canon_Municipal.ipynb); se usa solo en los años y departamentos que PERUPETRO no cubre
_mef = _leer_extra("PP_canon_municipal_mef_anio.csv")
if _mef is not None and len(_mef):
    _mef["FUENTE_EXCEL"] = ["MEF_NORMAL" if "todos los canones" in str(t) else "MEF_GAS" if "GASIFER" in str(t) else "MEF"
                            for t in _mef["TIPO_RECURSO"]]
    _mef = (_mef.groupby(["DEPARTAMENTO", "AÑO", "MUNICIPALIDAD", "NIVEL", "PROVINCIA", "FUENTE_EXCEL"], dropna=False)["RECAUDADO_SOLES"].sum()
            .reset_index().rename(columns={"RECAUDADO_SOLES": "MONTO_SOLES"}).assign(MES=float("nan"), SECCION="MEF"))
    if _cm is not None and len(_cm):
        _ya = set(zip(_cm["DEPARTAMENTO"].map(_dep), _cm["AÑO"].astype(int)))
        _mef = _mef[[(_dep(d), int(a)) not in _ya for d, a in zip(_mef["DEPARTAMENTO"], _mef["AÑO"])]]
        _cm = pd.concat([_cm, _mef], ignore_index=True)
    else:
        _cm = _mef
    print(f"   🏛️ Canon municipal del MEF: {len(_mef)} filas en años sin detalle de PERUPETRO")
if _cm is not None and len(_cm) and "DISTRITO_DETALLE" in geo:
    import difflib
    _cm = _cm.assign(_D=_cm["DEPARTAMENTO"].map(_dep), _M=_cm["MUNICIPALIDAD"].map(_dep), _P=_cm["PROVINCIA"].fillna("").map(_dep))
    ACT["cmn"] = {}
    # fuente de cada año (el gráfico la muestra): PERUPETRO Distribución 1993–2002, PERUPETRO Canon Municipalidades (año en curso), MEF
    FUENTE_MUN = {"DISTRIBUCION": "PERUPETRO, Excel de Distribución (detalle por concejos)",
                  "MUNICIPAL": "PERUPETRO, Excel de Canon Municipalidades (validado con la PCM)",
                  "MEF": "MEF, Consulta Amigable de Ingresos (canon y sobrecanon petrolero recaudado por cada municipalidad)",
                  "MEF_GAS": "MEF, Consulta Amigable de Ingresos (canon gasífero recaudado por cada municipalidad; Cusco, Camisea)",
                  "MEF_NORMAL": "MEF, Consulta Amigable de Ingresos (tipo «NORMAL», 2005–2006: todos los canones recaudados, no solo el petrolero)"}
    ACT["cmf"] = {}
    for (d, a), g in _cm.groupby(["_D", "AÑO"]):
        if g["MES"].notna().any():                     # el MEF es anual: sin aviso de meses
            ACT["cmn"].setdefault(d, {})[int(a)] = int(g["MES"].nunique())
        fu = sorted({FUENTE_MUN.get(str(x).upper(), str(x)) for x in g.get("FUENTE_EXCEL", pd.Series(dtype=str)).dropna()})
        ACT["cmf"].setdefault(d, {})[int(a)] = " + ".join(fu)
    def _parecido(x, opciones):
        m_ = difflib.get_close_matches(x, opciones, n=1, cutoff=0.82)
        return m_[0] if m_ else None
    for _, r in geo.iterrows():
        campo_k = str(r["CAMPO"] if "CAMPO" in r and pd.notna(r["CAMPO"]) else r["LOTE"])
        for it in str(r.get("DISTRITO_DETALLE") or "").split(";"):
            m_ = re.match(r"\s*(.+?)\s+([\d.]+)%\s*$", it)
            if not m_ or m_.group(1).count("|") != 2:
                continue
            dep, prov, dist = m_.group(1).split("|")
            sub = _cm[_cm["_D"] == dep]
            if not len(sub):
                continue
            ent = ACT.setdefault("cm", {}).setdefault(campo_k, {"d": [], "v": {}})
            # municipalidad distrital (por nombre; tolera pequeñas diferencias de escritura) y la provincial de su provincia
            dist_m = dist if dist in set(sub["_M"]) else _parecido(dist, sorted(set(sub["_M"])))
            prov_m = prov if prov in set(sub["_P"]) else _parecido(prov, sorted(set(sub["_P"]) - {""}))
            pedidos = []
            if dist_m:
                pedidos.append((sub[sub["_M"] == dist_m], dist_m, float(m_.group(2))))
            if prov_m:
                pv = sub[(sub["_P"] == prov_m) & (sub["NIVEL"] == "PROVINCIAL")]
                for mun in sorted(set(pv["_M"])):
                    pedidos.append((pv[pv["_M"] == mun], mun, None))
            for filas_, mun, pct in pedidos:
                niv = "PROVINCIAL" if (filas_["NIVEL"] == "PROVINCIAL").all() else "DISTRITAL"
                clave = f"{dep}|{prov_m or prov}|{mun}"
                if clave in ent["v"]:
                    if pct is not None:            # el distrito ya entró como provincial: se le pone el %
                        for x in ent["d"]:
                            if x[0] == clave and x[4] is None:
                                x[4] = pct
                    continue
                v = (filas_.groupby("AÑO")["MONTO_SOLES"].sum() / 1e6).round(4)
                if not len(v):
                    continue
                ent["v"][clave] = {int(a): float(x) for a, x in v.items()}
                ent["d"].append([clave, mun.title(), (prov_m or prov).title(), niv, pct])
    if "cm" in ACT:
        anios = [a for e in ACT["cm"].values() for v in e["v"].values() for a in v]
        if anios:
            ACT["rango"]["cm"] = [min(anios), max(anios)]
if "cb" in ACT:
    anios = [a for d in ACT["cb"].values() for t in d.values() for a in t]
    ACT["rango"]["cb"] = [min(anios), max(anios)]
print(f"🛠️ Pozos: {len(POZOS_CAMPO)} campos · 💵 Regalías: {len(REGALIAS_CAMPO)} campos · 🏛️ Canon: {len(ACT['c'])} departamentos"
      f" · por lote {len(ACT.get('cl', {}))} campos · municipalidades de {len(ACT.get('cm', {}))} campos")

# ── Empresas: quién operó cada lote y desde cuándo (de la producción) y socios del contrato (sección 6.1) ──
def historial_operadores(prod, gas=None):
    """Una fila por lote, operador y periodo continuo de años con producción fiscalizada (líquidos o gas).
    Responde «quién lo opera y desde cuándo», también para contratos que ya terminaron."""
    cols = ["AÑO", "LOTE", "OPERADOR", "DEPARTAMENTO"]
    p_ = prod.assign(LIQ=pd.to_numeric(prod["BARRILES"], errors="coerce").fillna(0), GAS=0.0)
    p_ = p_[[c for c in cols if c in p_] + ["LIQ", "GAS"]]
    if gas is not None and len(gas):
        g_ = gas.assign(LIQ=0.0, GAS=pd.to_numeric(gas["MPC"], errors="coerce").fillna(0))
        p_ = pd.concat([p_, g_[[c for c in cols if c in g_] + ["LIQ", "GAS"]]], ignore_index=True)
    p_["LOTE"] = p_["LOTE"].astype(str)
    p_["OPERADOR"] = p_["OPERADOR"].fillna("").astype(str).str.strip()
    p_["DEPARTAMENTO"] = p_.get("DEPARTAMENTO", pd.Series("", index=p_.index)).fillna("").astype(str)
    p_ = p_[(p_["LIQ"] + p_["GAS"] > 0) & (p_["OPERADOR"] != "")]
    a = p_.groupby(["LOTE", "OPERADOR", "AÑO"], as_index=False).agg(
        LIQ=("LIQ", "sum"), GAS=("GAS", "sum"), DEP=("DEPARTAMENTO", lambda s: next((x for x in s if x), "")))
    filas = []
    for (lote, op), g in a.sort_values("AÑO").groupby(["LOTE", "OPERADOR"]):
        anios, ini = g["AÑO"].astype(int).tolist(), 0
        for i in range(1, len(anios) + 1):
            if i == len(anios) or anios[i] != anios[i - 1] + 1:          # un año sin producción corta el periodo
                t = g.iloc[ini:i]
                filas.append({"DEPARTAMENTO": next((x for x in t["DEP"] if x), ""), "CAMPO": CAMPO_DE.get(lote, lote),
                              "LOTE": lote, "OPERADOR": op, "AÑO_INICIO": anios[ini], "AÑO_FIN": anios[i - 1],
                              "AÑOS_CON_PRODUCCION": i - ini, "LIQUIDOS_BBL": int(round(t["LIQ"].sum())),
                              "GAS_MPC": int(round(t["GAS"].sum()))})
                ini = i
    out = pd.DataFrame(filas)
    return out.sort_values(["CAMPO", "AÑO_INICIO", "LOTE"]).reset_index(drop=True) if len(out) else out

def _lote_base(l):
    return re.sub(r"\(T+\)$", "", str(l).replace(" ", ""))
OPS = historial_operadores(prod, globals().get("gas")) if prod is not None else pd.DataFrame()
# Regalías por empresa: el notebook de producción asigna el operador a cada fila de regalías (PP_regalias_lote_anio.csv,
# columna OPERADOR). En la vista por lote se suman; aquí se separan (Lote 8 en 2025: Upland y PPN en liquidación).
REG_SIN_OP = {}
if len(OPS):
    OPS["REGALIAS_US$"] = 0.0
    if _rg is not None and "OPERADOR" in _rg:
        _ro = _rg.assign(_L=_rg["LOTE"].map(_lote_base), OPERADOR=_rg["OPERADOR"].fillna("").astype(str))
        _ro_op = _ro.groupby(["_L", "OPERADOR", "AÑO"])["REGALIA_US$"].sum().to_dict()
        OPS["REGALIAS_US$"] = [round(sum(_ro_op.get((_lote_base(l), o, a), 0) for a in range(a0, a1 + 1)), 2)
                               for l, o, a0, a1 in OPS[["LOTE", "OPERADOR", "AÑO_INICIO", "AÑO_FIN"]].itertuples(index=False)]
        # filas que el PDF no permite atribuir (dos contratos del mismo lote sin distinción, como el Lote 8 en 2024)
        _campo_l = dict(zip(_ro["_L"], _ro["_C"]))
        for (l, o, a), v in _ro_op.items():
            if o in ("", "VARIOS OPERADORES") and v:
                REG_SIN_OP.setdefault(_campo_l.get(l, l), []).append([l, int(a), round(v / 1e6, 3)])
if len(OPS):
    _p = os.path.join(ruta, "PP_operadores_lote.csv"); OPS.to_csv(_p, index=False, encoding="utf-8-sig")
    subir(_p, DRIVE_FOLDER_ID, datos_drive)
    ACT["rango"]["e"] = [int(OPS["AÑO_INICIO"].min()), int(OPS["AÑO_FIN"].max())]
    for c, g in OPS.groupby("CAMPO"):
        ACT.setdefault("e", {}).setdefault(c, {})["o"] = [[r.LOTE, r.OPERADOR, int(r.AÑO_INICIO), int(r.AÑO_FIN),
                                                           int(r.LIQUIDOS_BBL), int(r.GAS_MPC), round(float(r[-1]) / 1e6, 3)]
                                                          for r in g.itertuples(index=False)]
    for c, v in REG_SIN_OP.items():
        ACT.setdefault("e", {}).setdefault(c, {})["v"] = sorted(v, key=lambda x: (x[1], x[0]))
# Contratos (notebook de producción): socios y vigencia del contrato según la edición más reciente que lo trae
_ct = _leer_extra("PP_contratos.csv")
SOCIOS_LOTE, ED_CONTRATOS = {}, None
if _ct is not None and len(_ct):
    _ct["LOTE"] = _ct["LOTE"].astype(str)
    _ct["CAMPO_C"] = _campo_de_lote(_ct)
    ED_CONTRATOS = int(_ct["EDICION"].max())
    vig = _ct[(_ct["EDICION"] == ED_CONTRATOS) & (_ct["ESTADO"] == "VIGENTE")]
    _txt = lambda v: "" if pd.isna(v) else str(v)
    for l, g in vig.groupby("LOTE", sort=False):
        SOCIOS_LOTE[l] = g
        ACT.setdefault("e", {}).setdefault(g["CAMPO_C"].iloc[0], {}).setdefault("s", []).extend(
            [[l, r_.EMPRESA, None if pd.isna(r_.PARTICIPACION_PCT) else float(r_.PARTICIPACION_PCT),
              _txt(r_.FECHA_SUSCRIPCION), _txt(r_.FECHA_TERMINO),
              {"EXPLOTACION": "Explotación", "EXPLORACION": "Exploración"}.get(str(r_.FASE), str(r_.FASE))] for r_ in g.itertuples()])
    # contratos terminados de cada edición: una fila por contrato (operador = 1.ª empresa)
    ter = _ct[_ct["ESTADO"] == "TERMINADO"].drop_duplicates(["EDICION", "LOTE", "OPERADOR"])
    for r_ in ter.itertuples():
        ACT.setdefault("e", {}).setdefault(r_.CAMPO_C, {}).setdefault("t", []).append(
            [r_.LOTE, r_.OPERADOR, _txt(r_.FECHA_SUSCRIPCION), _txt(r_.FECHA_TERMINO), int(r_.EDICION)])
    ACT["ed"] = ED_CONTRATOS
print(f"🏢 Empresas: {OPS['OPERADOR'].nunique() if len(OPS) else 0} operadores en {OPS['LOTE'].nunique() if len(OPS) else 0} lotes"
      + (f" · contratos vigentes al 31/12/{ED_CONTRATOS}: {len(SOCIOS_LOTE)}" if ED_CONTRATOS else " · sin PP_contratos.csv (ejecuta la sección 6.1)"))

def tiene_act(ctx, t):
    if t == "c":
        return bool(ACT["c"].get(ctx["d"]) or str(ctx["c"]) in ACT.get("cl", {}) or str(ctx["c"]) in ACT.get("cm", {}))
    return str(ctx["c"]) in ACT.get(t, {})

def resumen_html(campo, dep):
    """Una línea con las cifras clave del último año: pozos perforados, regalías del campo y canon del departamento.
    Los gráficos y el detalle están en el panel «📊 Ver estadísticas»."""
    campo, dep, partes = str(campo), _dep(dep or ""), []
    pz = POZOS_CAMPO.get(campo)
    if pz and (sum(pz["d"].values()) + sum(pz["e"].values())) > 0:
        a1 = ACT["rango"]["p"][1]; n = pz["d"].get(a1, 0) + pz["e"].get(a1, 0)
        partes.append((a1, f"{_num(n, 0)} pozo{'' if n == 1 else 's'} perforado{'' if n == 1 else 's'}"))
    rg = REGALIAS_CAMPO.get(campo)
    if rg and sum(rg.values()) > 0:
        a1 = ACT["rango"]["r"][1]
        partes.append((a1, f"regalías US$ {_num(rg.get(a1, 0), 1)} millones"))
    cn = ACT["c"].get(dep)
    if cn:
        a1 = ACT["rango"]["c"][1]
        partes.append((a1, f"canon y sobrecanon de {_h.escape(dep.title())} S/ {_num(cn.get(a1, 0), 1)} millones (todo el departamento)"))
    if not partes:
        return ""
    anios = {a for a, _ in partes}
    txt = " · ".join(t if len(anios) == 1 else f"{t} ({a})" for a, t in partes)
    return (_TIT.format("Resumen " + (str(anios.pop()) if len(anios) == 1 else "del último año")) +
            f"<div style='margin-top:4px'>{txt}</div>")

SECCIONES = [
    ("Producción", ["PRIMERA_PRODUCCION", "ULTIMA_PRODUCCION", "PICO_AÑO", "PICO_BPD", "DECLINACION_ANUAL_5A_PCT", "DECLINACION_PERIODO",
                    "PARALIZACION_MAX_MESES", "PARALIZACION_MAX_DESDE", "PETROLEO_ACUM_CAMPO_MMBBL", "GAS_ACUM_CAMPO_BCF"]),
    ("Empresa y contrato", ["CAMPO", "OPERADOR_MAPA", "_SOCIOS", "_VIGENCIA", "OPERADORES", "ESTADO_MAPA", "TIPO_CONTRATO",
                            "F_SUSCRIPCION"]),
    ("Geografía y sociedad", ["DEPARTAMENTO_DETALLE", "PROVINCIA_DETALLE", "DISTRITO_DETALLE", "AREA_KM2", "DIST_OLEODUCTO_KM", "RIO_CERCANO", "DIST_IQUITOS_KM",
                              "ANP_DETALLE", "CCNN_DENTRO_N", "CCNN_5KM_N"]),
]
_ETQ = {c: (e, u) for c, e, u in ETIQUETAS}
SELVA = {"LORETO", "UCAYALI", "HUANUCO", "CUSCO", "SAN MARTIN", "MADRE DE DIOS", "JUNIN", "PASCO", "AMAZONAS", "PUNO", "AYACUCHO"}
_TIT = ("<div style='margin-top:8px;border-top:1px solid #ddd;padding-top:4px;color:#555;font-weight:700;"
        "text-transform:uppercase;letter-spacing:.04em;font-size:10.5px'>{}</div>")

def empresa_info(lote):
    """Socios con su participación y vigencia del contrato vigente (PP_contratos.csv, edición más reciente)."""
    g = SOCIOS_LOTE.get(str(lote))
    if g is None:
        g = SOCIOS_LOTE.get(_lote_base(lote))
    if g is None:
        return None, None
    soc = "; ".join(e + (f" ({_num(p, 2).rstrip('0').rstrip(',')} %)" if pd.notna(p) else "")
                    for e, p in zip(g["EMPRESA"], g["PARTICIPACION_PCT"]))
    f0, f1 = g["FECHA_SUSCRIPCION"].iloc[0], g["FECHA_TERMINO"].iloc[0]
    vig = (f"{f0 if pd.notna(f0) else '¿?'} → {f1}" if pd.notna(f1) else f"suscrito el {f0}" if pd.notna(f0) else None)
    return soc, (f"{vig} (PERUPETRO, al 31/12/{ED_CONTRATOS})" if vig else None)

def popup_html(r):
    r = r.copy()
    r["_SOCIOS"], r["_VIGENCIA"] = empresa_info(r["LOTE"])
    lote, campo = str(r["LOTE"]), r.get("CAMPO")
    campo_k = campo if pd.notna(campo) else lote
    desde_1994 = str(r.get("PRIMERA_PRODUCCION") or "").startswith("≤")   # la serie empieza antes del lote: pico y acumulado truncados
    e = EST_PRODUCTO.get(str(campo_k), {})
    produce = any(d.get("e") in ("EN PRODUCCIÓN", "PARALIZADO") for d in e.values())
    selva = _dep(r.get("DEPARTAMENTO") or "") in SELVA and str(r.get("ZOCALO", "NO")) != "SI"
    def fila(c):
        etq, u = _ETQ.get(c, (c, ""))
        x = fmt(c, r.get(c), u) if c in r else None
        if x is None:
            return ""
        if c in ("PARALIZACION_MAX_MESES", "PARALIZACION_MAX_DESDE") and not r.get("PARALIZACION_MAX_MESES"):
            return ""                                   # sin paralizaciones: no se muestra
        if c in ("GAS_ACUM_CAMPO_BCF", "PETROLEO_ACUM_CAMPO_MMBBL") and not (pd.notna(r.get(c)) and float(r.get(c) or 0) > 0):
            return ""                                   # acumulado 0: campo sin ese producto
        if c in ("RIO_CERCANO", "DIST_IQUITOS_KM") and not selva:
            return ""                                   # ríos amazónicos e Iquitos solo para lotes de la selva
        if c == "CAMPO" and str(campo) == lote:
            return ""                                   # el campo solo se muestra si suma varios contratos (1-AB/192)
        if c == "ESTADO_MAPA" and str(r.get(c, "")).upper().startswith("EXPLOTA") and produce:
            return ""                                   # la fase se deduce de la producción
        if c == "DEPARTAMENTO_DETALLE":                 # un solo departamento al 100 %: sin porcentaje
            x = re.sub(r"^([^;%]+?)\s+100[.,]0\s*%$", r"\1", x)
        if c in ("PROVINCIA_DETALLE", "DISTRITO_DETALLE"):   # «LORETO|REQUENA|PUINAHUA 85.0%» → «Puinahua (Requena) 85 %»
            partes = []
            for it in str(r.get(c)).split(";"):
                m_ = re.match(r"\s*(.+?)\s+([\d.]+)%\s*$", it)
                if not m_:
                    continue
                k_ = m_.group(1).split("|")
                nom = k_[-1].title() + (f" ({k_[-2].title()})" if c == "DISTRITO_DETALLE" and len(k_) > 2 and k_[-2] else "")
                pct = float(m_.group(2))
                partes.append(nom if pct >= 99.95 else f"{nom} {_num(pct, 1 if pct < 10 else 0)} %")
            if not partes:
                return ""
            x = _h.escape("; ".join(partes[:3]) + (f"; y {len(partes) - 3} más" if len(partes) > 3 else ""))   # ficha corta
        if c == "RIO_CERCANO":                          # río grande más cercano y su distancia (ya no por umbral de caudal)
            dist = fmt("DIST_RIO_GRANDE_KM", r.get("DIST_RIO_GRANDE_KM"), "km")
            etq, x = "Río grande más cercano", x + (f" · {dist}" if dist else "")
        if desde_1994 and c in ("PICO_AÑO", "PICO_BPD"):
            etq += " (desde 1994)"
        if desde_1994 and c in ("PETROLEO_ACUM_CAMPO_MMBBL", "GAS_ACUM_CAMPO_BCF"):
            etq += " (1994–hoy)"
        if c == "CAMPO":
            etq = "Campo (contratos sumados)"
        if c == "F_SUSCRIPCION" and r.get("_VIGENCIA"):
            return ""                                   # la vigencia del cuadro de contratos ya trae la suscripción
        if c == "OPERADORES":
            etq = "Operadores con producción"
        return f"<tr><td style='color:#666;padding-right:8px;vertical-align:top'>{etq}</td><td>{x}</td></tr>"
    nota_94 = ("<div style='color:#888;font-size:11px;margin-top:2px'>La serie empieza en 1994: el lote ya producía antes, "
               "así que el pico y el acumulado reales pueden ser mayores.</div>") if desde_1994 else ""
    html_ = f"<div style='font:12px sans-serif;min-width:260px'><b style='font-size:14px'>Lote {_h.escape(lote)}</b>"
    # un solo botón, arriba (debajo del nombre): abre el panel de estadísticas (Producción · Pozos · Regalías · Canon · Empresas)
    ctx = {"c": str(campo_k), "l": lote, "d": _dep(r.get("DEPARTAMENTO") or "")}
    id_ = f"C:{campo_k}"
    if id_ in IDS_PANEL:
        ctx["pp"] = id_
    if "pp" in ctx or any(tiene_act(ctx, t) for t in "prce"):
        html_ += (f"<button type='button' data-est='{_h.escape(json.dumps(ctx, ensure_ascii=False), quote=True)}' "
                  "title='Producción, pozos, regalías, canon y empresas' style='margin:6px 0 2px;width:100%;border:1px solid #b9b8b2;"
                  "background:#f3f2ee;border-radius:6px;padding:6px 8px;cursor:pointer;font:600 13px sans-serif'>"
                  "📊 Ver estadísticas</button>")
    if "pp" in ctx or str(campo_k) in ACT.get("cl", {}):
        html_ += (f"<button type='button' data-cmp='{_h.escape(lote, quote=True)}' title='Comparar este lote con otros: producción, canon y su distribución' "
                  "style='margin:0 0 2px;width:100%;border:1px solid #b9b8b2;background:#fff;border-radius:6px;padding:4px 8px;cursor:pointer;"
                  "font:600 12px sans-serif'>➕ Comparar con otros lotes</button>")
    for i, (tit, cols) in enumerate(SECCIONES):
        filas = "".join(fila(c) for c in cols)
        extra_ini = estado_producto_html(campo) if i == 0 else ""
        extra_fin = nota_94 if i == 0 else ""
        if filas or extra_ini or extra_fin:
            html_ += _TIT.format(tit) + extra_ini + (f"<table style='margin-top:4px'>{filas}</table>" if filas else "") + extra_fin
    res = resumen_html(campo_k, r.get("DEPARTAMENTO"))
    if res:
        html_ += res
    return html_ + "</div>"

# ── Capas de contexto ────────────────────────────────────────────────────────
if deps is not None:
    folium.GeoJson(simpl(deps, 0.005), name="Departamentos",
                   style_function=lambda f: {"color": "#666", "weight": 0.8, "fillOpacity": 0},
                   tooltip=folium.GeoJsonTooltip(["DEPARTAMENTO"], labels=False)).add_to(m)
if anp is not None:
    a_ = simpl(anp[[c for c in [anp_n] if c] + ["geometry"]])
    folium.GeoJson(a_, name="Áreas Naturales Protegidas", show=False,
                   style_function=lambda f: {"color": "#2e7d32", "weight": 0.5, "fillColor": "#66bb6a", "fillOpacity": 0.25},
                   tooltip=folium.GeoJsonTooltip([anp_n], labels=False) if anp_n else None).add_to(m)
CLASES_RIO = {"Afluente (1.000–5.000 m³/s)": 1, "Río troncal (5.000–20.000 m³/s)": 2.2,
              "Amazonas / tramo bajo (> 20.000 m³/s)": 3.5}
if rios_q is not None:
    r_ = rios_q[rios_q["DIS_AV_CMS"] >= 1000].copy()
    r_["CLASE"] = pd.cut(r_["DIS_AV_CMS"], [999, 5000, 20000, 1e9], labels=list(CLASES_RIO)).astype(str)
    r_ = simpl(r_[["CLASE", "geometry"]].dissolve("CLASE", as_index=False), 0.003)
    folium.GeoJson(r_, name="Ríos (HydroRIVERS)",
                   style_function=lambda f: {"color": "#1e88e5", "weight": CLASES_RIO.get(f["properties"]["CLASE"], 1), "opacity": 0.8},
                   tooltip=folium.GeoJsonTooltip(["CLASE"], labels=False)).add_to(m)
if ccnn is not None and ccnn_n:
    mc = MarkerCluster(name="Comunidades nativas", show=False)
    for _, r in ccnn.to_crs(4326).iterrows():
        if r.geometry is None or r.geometry.is_empty:
            continue
        pt = r.geometry.representative_point()
        folium.CircleMarker([pt.y, pt.x], radius=3, color="#8e24aa", fill=True, fill_opacity=0.8,
                            tooltip=_h.escape(str(r[ccnn_n]))).add_to(mc)
    mc.add_to(m)

if onp is not None:
    o_ = simpl(onp[["geometry"]].dissolve(), 0.002)
    o_["NOMBRE"] = "Oleoducto Norperuano"
    folium.GeoJson(o_, name="Oleoducto Norperuano",
                   style_function=lambda f: {"color": "#111", "weight": 3, "dashArray": "6 4"},
                   tooltip=folium.GeoJsonTooltip(["NOMBRE"], labels=False)).add_to(m)
# ── Lotes: una sola capa; su color y los círculos de producción actual dependen de los productos marcados ──
capa_l = geo[geo["GEOMETRIA_DE"] == geo["LOTE"]].copy() if "GEOMETRIA_DE" in geo else geo.copy()
capa_l = simpl(capa_l, 0.001)
fg_lotes = folium.FeatureGroup(name="Lotes", control=False)
CENTROS = {}
for _, r in capa_l.iterrows():
    campo = str(r.get("CAMPO") if pd.notna(r.get("CAMPO")) else r["LOTE"])
    gj = folium.GeoJson({"type": "Feature", "geometry": r.geometry.__geo_interface__,
                         "properties": {"pp_campo": campo, "pp_lote": str(r["LOTE"])}},
                        style_function=lambda f: {"color": "#9e9e9e", "weight": 1.2, "fillColor": "#9e9e9e", "fillOpacity": 0.45})
    # alto máximo con desplazamiento interno, para que la ficha no tape el mapa
    folium.Popup(f"<div style='max-height:min(430px,58vh);overflow-y:auto;padding-right:6px'>{popup_html(r)}</div>", max_width=380).add_to(gj)
    gj.add_to(fg_lotes)
    if campo not in CENTROS and pd.notna(r.get("LAT")):
        CENTROS[campo] = [round(float(r["LAT"]), 4), round(float(r["LON"]), 4)]
fg_lotes.add_to(m)
folium.Marker([IQUITOS.y, IQUITOS.x], tooltip="Iquitos", icon=folium.Icon(color="blue", icon="info-sign")).add_to(m)

caja = "display:inline-block;width:11px;height:11px;margin-right:5px;vertical-align:middle"
# Leyenda en la esquina inferior derecha: cerrada al abrir el mapa; el botón + / – la abre y la minimiza
leyenda = ("<div id='pp-ley' class='pp-caja'><div id='pp-ley-cab'><b>Leyenda</b>"
           "<button type='button' class='pp-btn' id='pp-ley-btn' title='Mostrar leyenda' aria-expanded='false' "
           "onclick='ppLey()'>+</button></div><div id='pp-ley-c' style='display:none;margin-top:4px'>"
           + "<span style='color:#555'>Estado según los <b>productos</b> marcados en el selector de capas:</span><br>"
           + "".join(f"<span style='{caja};background:{COLOR_ESTADO[e]};opacity:.7'></span>{t}<br>"
                     for e, t in [("EN PRODUCCIÓN", "En producción (alguno de los productos)"), ("PARALIZADO", "Paralizado"),
                                  ("SIN PRODUCCIÓN REGISTRADA", "Sin producción")])
           + f"<span style='{caja};border-radius:50%;background:#66bb6a'></span>Producción actual (círculo ∝ bep/d; "
             "gas: 6 mil pies³ = 1 bep)<br>"
           "<span style='display:inline-block;width:20px;border-top:3px dashed #111;margin-right:5px;vertical-align:middle'></span>"
           "Oleoducto Norperuano<br>"
           "<span style='display:inline-block;width:20px;border-top:3px solid #1e88e5;margin-right:5px;vertical-align:middle'></span>"
           "Ríos grandes<br><span style='color:#777'>En producción = produjo en los últimos 12 meses. "
           "Clic en un lote o círculo para ver su ficha (con el estado de cada producto); en la ficha, «📊 Ver estadísticas» abre la producción anual, mensual y diaria, los pozos, las regalías, el canon y las empresas."
           "</span></div></div>")
m.get_root().html.add_child(folium.Element(leyenda))
m.get_root().header.add_child(folium.Element("<title>Mapa de Hidrocarburos - Perú</title>"))   # nombre de la pestaña del navegador
if DATOS_PANEL:
    _js = json.dumps(DATOS_PANEL, ensure_ascii=False, separators=(",", ":")).replace("</", "<\\/")
    # folium.Element trata el texto como plantilla Jinja: {% raw %} evita que lea "{#" del CSS/JS como comentario
    m.get_root().html.add_child(folium.Element("{% raw %}" + PANEL_HTML.replace("__DATA__", _js) + "{% endraw %}"))
else:                                                  # sin producción: solo los estilos de la leyenda
    m.get_root().html.add_child(folium.Element("{% raw %}" + PANEL_HTML.split("</style>")[0] + "</style>"
                                               "<script>window.ppLey=function(){var c=document.getElementById('pp-ley-c'),"
                                               "b=document.getElementById('pp-ley-btn'),a=c.style.display==='none';"
                                               "c.style.display=a?'block':'none';b.textContent=a?'–':'+';};</script>{% endraw %}"))
# panel «Pozos, regalías y canon» (se abre desde la ficha del lote)
if ACT["p"] or ACT["r"] or ACT["c"] or ACT["e"]:
    _ja = json.dumps({**ACT, "p": ACT["p"], "r": ACT["r"]}, ensure_ascii=False, separators=(",", ":")).replace("</", "<\\/")
    m.get_root().html.add_child(folium.Element("{% raw %}" + ACT_HTML.replace("__ACT__", _ja) + "{% endraw %}"))
# «Comparar lotes»: barras por año con la tabla de datos debajo (producción, canon generado y su distribución)
if DATOS_PANEL or ACT.get("cl"):
    m.get_root().html.add_child(folium.Element("{% raw %}" + COMPARAR_HTML + "{% endraw %}"))
# botón «📊 Ver estadísticas» de la ficha: abre Producción (si el campo tiene serie) o la primera pestaña con datos
m.get_root().html.add_child(folium.Element("{% raw %}<script>document.addEventListener('click',function(ev){"
    "var b=ev.target.closest&&ev.target.closest('[data-est]');if(!b)return;ev.preventDefault();var o=JSON.parse(b.dataset.est);"
    "if(o.pp&&window.ppAbrir)window.ppAbrir(o.pp,null,o);else if(window.ppActAbrir)window.ppActAbrir(o);});</script>{% endraw %}"))
folium.LayerControl(collapsed=False).add_to(m)
# selector de productos (dentro del control de capas): colorea los lotes y dibuja los círculos de producción actual
PRODUCTOS_HTML = r'''<style>
.pp-prod-sel{margin-top:4px;padding-top:4px;border-top:1px solid #ddd}
.pp-prod-sel b{display:block;margin:2px 0 3px}
.pp-prod-sel label{display:block;cursor:pointer}
/* sin el rectángulo de foco del navegador al hacer clic en un polígono */
path.leaflet-interactive:focus{outline:none}
</style>
<script>
(function(){
const EST=__EST__, CENTROS=__CENTROS__;
const COL={"EN PRODUCCIÓN":"#2e7d32","PARALIZADO":"#ef6c00","SIN PRODUCCIÓN":"#9e9e9e"};
const NOMBRE={p:"petróleo y condensado",l:"LGN",g:"gas natural"}, UNI={p:"bpd",l:"bpd",g:"MMPCD"};
const sel={p:true,l:true,g:true};
let mapa=null,lotes=[],circulos=null;
function num(v,d){const s=Math.abs(v).toFixed(d||0).split(".");return s[0].replace(/\B(?=(\d{3})+(?!\d))/g,".")+(s[1]?","+s[1]:"");}
// estado combinado de los productos marcados: en producción si alguno produce; paralizado si alguno produjo antes
function estado(campo){
  const e=EST[campo]||{};let hay=false,par=false;
  for(const k in sel){if(!sel[k])continue;const d=e[k];if(!d)continue;
    if(d.e==="EN PRODUCCIÓN")hay=true;else if(d.e==="PARALIZADO")par=true;}
  return hay?"EN PRODUCCIÓN":par?"PARALIZADO":"SIN PRODUCCIÓN";
}
function detalle(campo){
  const e=EST[campo]||{},partes=[];
  for(const k in sel){if(!sel[k])continue;const d=e[k];if(!d||d.e==="SIN PRODUCCIÓN")continue;
    partes.push(NOMBRE[k]+": "+(d.e==="EN PRODUCCIÓN"?num(d.r,k==="g"?2:0)+" "+UNI[k]:"paralizado desde "+d.u));}
  return partes.join(" · ");
}
function bep(campo){                                   // barriles equivalentes por día de los productos marcados
  const e=EST[campo]||{};let s=0;
  for(const k in sel){const d=e[k];if(sel[k]&&d&&d.e==="EN PRODUCCIÓN")s+=k==="g"?d.r*1000/6:d.r;}
  return s;
}
function aplicar(){
  // con la línea de tiempo animada activa (mapa con plugins), ella pinta los lotes
  if(window.ppTL&&window.ppTL.activo){window.ppTL.pintar();return;}
  const ninguno=!sel.p&&!sel.l&&!sel.g;
  lotes.forEach(o=>{
    const el=o.capa.getElement&&o.capa.getElement();
    if(ninguno){                                       // sin productos marcados: los lotes no se muestran
      o.capa.setStyle({opacity:0,fillOpacity:0});if(el)el.style.pointerEvents="none";o.capa.closeTooltip();return;}
    const est=estado(o.campo),c=COL[est];
    o.capa.setStyle({color:c,fillColor:c,opacity:1,fillOpacity:0.45,weight:1.2});if(el)el.style.pointerEvents="";
    const det=detalle(o.campo);
    o.capa.setTooltipContent("Lote "+o.lote+" · "+est.toLowerCase()+(det?"<br>"+det:""));
  });
  circulos.clearLayers();
  if(ninguno)return;
  Object.keys(CENTROS).forEach(campo=>{
    const v=bep(campo);if(!(v>0))return;
    const ll=CENTROS[campo],grupo=lotes.find(o=>o.campo===campo);
    const c=L.circleMarker(ll,{radius:4+Math.sqrt(v)/10,color:"#1b5e20",weight:1,fillColor:"#66bb6a",fillOpacity:0.75});
    c.bindTooltip("Campo "+campo+": "+num(v,0)+" bep/d<br>"+detalle(campo)+"<br><i>clic para la ficha</i>");
    if(grupo)c.on("click",()=>grupo.grupo.openPopup(L.latLng(ll)));      // el círculo abre la ficha del lote
    circulos.addLayer(c);
  });
}
function iniciar(){
  for(const k in window){try{if(window[k] instanceof L.Map){mapa=window[k];break;}}catch(e){}}
  if(!mapa)return;
  mapa.eachLayer(g=>{
    if(!(g instanceof L.GeoJSON))return;
    g.eachLayer(c=>{const p=c.feature&&c.feature.properties;
      if(p&&p.pp_campo!==undefined){c.bindTooltip("");lotes.push({campo:String(p.pp_campo),lote:p.pp_lote,capa:c,grupo:g});}});
  });
  circulos=L.featureGroup().addTo(mapa);
  // casillas de productos dentro del selector de capas
  const lista=document.querySelector(".leaflet-control-layers-list");
  if(lista){
    const caja=document.createElement("div");caja.className="pp-prod-sel";
    const t=document.createElement("b");t.textContent="Productos";caja.appendChild(t);
    [["p","Petróleo"],["l","LGN y condensado"],["g","Gas natural"]].forEach(([k,txt])=>{
      const lab=document.createElement("label"),ch=document.createElement("input");ch.type="checkbox";ch.checked=sel[k];
      ch.className="leaflet-control-layers-selector";
      ch.addEventListener("change",()=>{sel[k]=ch.checked;aplicar();});
      const s=document.createElement("span");s.textContent=" "+txt;lab.appendChild(ch);lab.appendChild(s);caja.appendChild(lab);});
    lista.appendChild(caja);L.DomEvent.disableClickPropagation(caja);
  }
  aplicar();
  // para los complementos del mapa con plugins (buscador, línea de tiempo)
  window.ppProductos={mapa:mapa,lotes:lotes,sel:sel,aplicar:aplicar,circulos:()=>circulos,COL:COL};
  // al prender una capa del selector, Leaflet la dibuja encima de todo y tapa los lotes: se los vuelve a poner arriba
  const arriba=()=>{lotes.forEach(o=>o.capa.bringToFront());circulos.eachLayer(c=>c.bringToFront());};
  mapa.on("overlayadd baselayerchange",()=>setTimeout(arriba,0));
}
if(document.readyState==="complete")setTimeout(iniciar,0);else window.addEventListener("load",()=>setTimeout(iniciar,0));
})();
</script>
'''
_pj = (PRODUCTOS_HTML.replace("__EST__", json.dumps(EST_PRODUCTO, ensure_ascii=False))
       .replace("__CENTROS__", json.dumps(CENTROS, ensure_ascii=False)).replace("</script>\n</", "</script>\n</"))
m.get_root().html.add_child(folium.Element("{% raw %}" + _pj + "{% endraw %}"))
p = os.path.join(ruta, "PP_mapa_lotes.html"); m.save(p)
subir(p, MAPA_FOLDER_ID, mapa_drive, mime="text/html")
print(f"☁️  Mapa guardado en DATA/MAPA/PP_mapa_lotes.html ({os.path.getsize(p) / 1e6:.1f} MB). "
      "Drive no muestra mapas interactivos: descárgalo y ábrelo con doble clic en Chrome/Edge.")
m